<a href="https://colab.research.google.com/github/Euripedes357/GMSF/blob/main/NOTEBOOK_EXTENDIDO_O_INSPETOR_DA_REDE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Fase 1 — Preparação do Inspetor da Rede

### Passo 1 — Conectar o Drive e conferir o que já existe

Este notebook não treina nada. Ele abre a rede que já foi treinada no notebook original e mostra o que há dentro dela, em 18 gráficos. Por isso o primeiro passo é conectar o Google Drive e conferir se os arquivos de que precisamos estão onde esperamos: o modelo treinado (final.keras), a divisão entre treino e validação (particao_treino_validacao.csv), o histórico do treino (hist_final.csv) e as imagens (archive.zip). Esta célula apenas lê: não escreve nada nas pastas do trabalho original.

A célula também imprime a versão das bibliotecas que vamos usar nos gráficos (Matplotlib, Plotly, Seaborn, SciPy, Graphviz e Pillow). Se alguma não estiver instalada, saberemos agora, antes de montar qualquer gráfico.

In [ ]:
# Passo 1 — Conectar o Drive, conferir os arquivos do projeto e as versões das bibliotecas
import os, importlib
from google.colab import drive
drive.mount('/content/drive')

PASTA_PROJETO = '/content/drive/MyDrive/BRAIN TUMOR MRI - by Euripedes Gomes'
ARQUIVOS = {
    'Modelo treinado':           PASTA_PROJETO + '/modelos/final.keras',
    'Partição treino/validação': PASTA_PROJETO + '/relatorio/particao_treino_validacao.csv',
    'Histórico do treino final': PASTA_PROJETO + '/relatorio/hist_final.csv',
    'Imagens (zip)':             PASTA_PROJETO + '/archive.zip',
}
for nome, caminho in ARQUIVOS.items():
    ok = os.path.exists(caminho)
    tam = round(os.path.getsize(caminho) / 1e6, 2) if ok else None
    print(f'{nome:27s} existe? {ok} | tamanho (MB): {tam}')

print()
for lib in ['tensorflow', 'keras', 'numpy', 'pandas', 'matplotlib', 'plotly', 'seaborn', 'scipy', 'graphviz', 'PIL']:
    try:
        m = importlib.import_module(lib)
        print(f'{lib:12s} {getattr(m, "__version__", "(sem versão)")}')
    except ImportError:
        print(f'{lib:12s} NÃO INSTALADA')

Mounted at /content/drive
Modelo treinado             existe? True | tamanho (MB): 4.73
Partição treino/validação   existe? True | tamanho (MB): 0.4
Histórico do treino final   existe? True | tamanho (MB): 0.0
Imagens (zip)               existe? True | tamanho (MB): 164.67

tensorflow   2.21.0
keras        3.13.2
numpy        2.4.6
pandas       2.2.3
matplotlib   3.10.0
plotly       5.24.1
seaborn      0.13.2
scipy        1.16.3
graphviz     0.21
PIL          11.3.0


**Observação:** o Drive foi montado e os quatro arquivos do notebook original foram encontrados: o modelo treinado (final.keras, 4,73 MB), a partição treino/validação (0,4 MB), o histórico do treino final (arquivo pequeno, que aparece como 0,0 MB por arredondamento) e o archive.zip de imagens (164,67 MB, o mesmo tamanho conferido no Passo 1 do original). O modelo ocupa mais que os 388.868 parâmetros sozinhos (cerca de 1,6 MB em precisão simples), provavelmente porque o arquivo guarda também o estado do otimizador Adam, que mantém duas médias por parâmetro. As versões do ambiente coincidem com as do notebook original (TensorFlow 2.21.0, Keras 3.13.2, NumPy 2.4.6, Pandas 2.2.3, Matplotlib 3.10.0, Pillow 11.3.0), o que garante que o modelo será lido do mesmo jeito em que foi gravado. As bibliotecas novas para os gráficos já estão instaladas: Plotly 5.24.1, Seaborn 0.13.2, SciPy 1.16.3 e Graphviz 0.21. Nada precisa ser instalado agora. Uma ressalva: o Graphviz listado é a interface em Python; o programa de desenho em si (dot) só será verificado se de fato formos usá-lo.

### Passo 2 — Descompactar as imagens e conferir a partição treino/validação

As imagens prontas (128×128) do notebook original não foram gravadas, então precisamos refazê-las. Esta célula é a primeira metade disso: copia o archive.zip para o disco local do Colab, descompacta em /content/data (o disco local é apagado ao fim da sessão e não toca no Drive) e abre a partição treino/validação gravada no Passo 9 do original. Ela confere se as contagens por classe batem com as de lá (3.909 imagens de treino e 958 de validação) e se todos os caminhos da partição existem no disco. A conversão para 128×128 em tons de cinza fica para o Passo 3.

In [ ]:
# Passo 2 — Descompactar as imagens no disco local e conferir a partição treino/validação
import os, zipfile, shutil, glob
import pandas as pd

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

PASTA_PROJETO = '/content/drive/MyDrive/BRAIN TUMOR MRI - by Euripedes Gomes'
DATA_DIR = '/content/data'

if not os.path.exists(DATA_DIR + '/Training'):
    shutil.copy(PASTA_PROJETO + '/archive.zip', '/content/archive.zip')
    zipfile.ZipFile('/content/archive.zip').extractall(DATA_DIR)

TRAIN_DIR = glob.glob(DATA_DIR + '/**/Training', recursive=True)[0]
TEST_DIR  = glob.glob(DATA_DIR + '/**/Testing',  recursive=True)[0]
CLASSES   = sorted(os.listdir(TRAIN_DIR))
print('Training:', TRAIN_DIR)
print('Testing :', TEST_DIR)
print('Classes :', CLASSES)

part = pd.read_csv(PASTA_PROJETO + '/relatorio/particao_treino_validacao.csv')
print('\nPartição:', part.shape, '| colunas:', part.columns.tolist())
tab = pd.crosstab(part['classe'], part['particao']).reindex(CLASSES)
tab.loc['Total'] = tab.sum()
display(tab)

existem = part['caminho'].apply(os.path.exists)
print(f'\nCaminhos da partição que existem no disco: {existem.sum()} de {len(part)}')

Training: /content/data/Training
Testing : /content/data/Testing
Classes : ['glioma', 'meningioma', 'notumor', 'pituitary']

Partição: (4867, 5) | colunas: ['classe', 'arquivo', 'caminho', 'grupo', 'particao']


particao,treino,validacao
classe,,
glioma,1120,276
meningioma,1036,258
notumor,629,148
pituitary,1124,276
Total,3909,958



Caminhos da partição que existem no disco: 4867 de 4867


**Observação:** o archive.zip foi descompactado em /content/data em cerca de 8 segundos, com Training e Testing diretamente nessa pasta, e as quatro classes foram reconhecidas na mesma ordem do notebook original (glioma, meningioma, notumor, pituitary). A partição treino/validação gravada no Passo 9 do original tem 4.867 linhas e cinco colunas (classe, arquivo, caminho, grupo e particao) e reproduz exatamente as contagens de lá: treino com 1.120 glioma, 1.036 meningioma, 629 notumor e 1.124 pituitary (3.909 no total), e validação com 276, 258, 148 e 276 (958 no total). Os 4.867 caminhos da partição existem no disco, o que significa que conseguimos reconstruir exatamente as mesmas imagens de treino e validação do original. O desequilíbrio já conhecido continua visível: o notumor tem 629 imagens de treino, contra mais de mil nas outras classes. O conjunto de teste não foi aberto neste notebook; ele permanece intocado, como no original.

### Passo 3 — Converter as imagens da validação para 128×128 em tons de cinza

Esta célula refaz, apenas para as 958 imagens de validação, a padronização do Passo 10 do notebook original: conversão para tons de cinza, acréscimo de bordas pretas centralizadas até a imagem ficar quadrada (sem distorcer a anatomia) e redução para 128×128 pixels pelo método de Lanczos. Os pixels ficam como inteiros de 0 a 255; a divisão por 255 acontece dentro da própria rede, na primeira camada. Os rótulos seguem a ordem glioma=0, meningioma=1, notumor=2 e pituitary=3. A conferência é o formato do resultado (958, 128, 128, 1), pixels de 0 a 255 e a contagem por classe, que deve ser igual à da validação do original.

In [ ]:
# Passo 3 — Padronizar as imagens da validação: tons de cinza + bordas pretas até ficar quadrada + 128x128
import os, glob
import numpy as np
import pandas as pd
from PIL import Image

PASTA_PROJETO = '/content/drive/MyDrive/BRAIN TUMOR MRI - by Euripedes Gomes'
TRAIN_DIR = glob.glob('/content/data/**/Training', recursive=True)[0]
CLASSES = sorted(os.listdir(TRAIN_DIR))
part = pd.read_csv(PASTA_PROJETO + '/relatorio/particao_treino_validacao.csv')

TAM = 128
def preparar(caminho):
    with Image.open(caminho) as img:
        img = img.convert('L')                                  # tons de cinza
        w, h = img.size
        lado = max(w, h)
        quadrada = Image.new('L', (lado, lado), 0)              # fundo preto
        quadrada.paste(img, ((lado - w) // 2, (lado - h) // 2)) # imagem centralizada
        return np.asarray(quadrada.resize((TAM, TAM), Image.LANCZOS), dtype=np.uint8)

mapa = {c: i for i, c in enumerate(CLASSES)}                    # glioma=0, meningioma=1, notumor=2, pituitary=3
val = part[part['particao'] == 'validacao'].reset_index(drop=True)
X_val = np.stack([preparar(c) for c in val['caminho']])[..., np.newaxis]
y_val = val['classe'].map(mapa).values.astype(np.int64)

print(f'X_val {X_val.shape} {X_val.dtype} | pixels de {X_val.min()} a {X_val.max()}')
print('Imagens por classe:', dict(zip(CLASSES, np.bincount(y_val, minlength=4))))

X_val (958, 128, 128, 1) uint8 | pixels de 0 a 255
Imagens por classe: {'glioma': np.int64(276), 'meningioma': np.int64(258), 'notumor': np.int64(148), 'pituitary': np.int64(276)}


**Observação:** as 958 imagens de validação foram padronizadas em cerca de 3 segundos e resultaram num conjunto de formato (958, 128, 128, 1), do tipo inteiro de 8 bits, com pixels de 0 a 255, como previsto no Passo 10 do notebook original. A contagem por classe confere para glioma (276) e meningioma (258); as outras duas classes somam as 424 imagens restantes, compatível com os 148 notumor e 276 pituitary da partição do Passo 2. O resultado ainda não passou pela divisão por 255, que acontece dentro da rede, na primeira camada. A ordem das imagens segue a da partição gravada, e os rótulos seguem glioma=0, meningioma=1, notumor=2 e pituitary=3. Como a conversão é determinística (sem sorteios), refazê-la gera exatamente as mesmas imagens que o notebook original usou na validação. A prova definitiva dessa equivalência virá no Passo 4: se o modelo treinado, aplicado a estas imagens, reproduzir o resultado do original, o preparo está correto.

### Passo 4 — Carregar o modelo treinado e conferir se ele reproduz o resultado do original

Antes de desenhar qualquer gráfico, precisamos ter certeza de que o modelo carregado e as imagens reconstruídas são os mesmos do notebook original. O teste é simples: o final.keras (a rede treinada, no ponto da época 28) é carregado, avaliado nas 958 imagens de validação, e o resultado é comparado com o do Passo 27 do original: perda 0,328, acurácia de 91,13% (873 acertos em 958) e revocação por classe de 90,9% para glioma, 86,8% para meningioma, 90,5% para notumor e 95,7% para pituitary. A célula também lista as camadas do modelo, com forma de saída e parâmetros. Se os números baterem, a reconstrução está validada; se não, investigamos antes de seguir.

In [ ]:
# Passo 4 — Carregar o final.keras e conferir o resultado na validação (deve reproduzir o Passo 27 do original)
import numpy as np
import keras

modelo = keras.models.load_model(PASTA_PROJETO + '/modelos/final.keras')

print('Camadas do modelo:')
for camada in modelo.layers:
    try:
        forma = tuple(camada.output.shape[1:])
    except Exception:
        forma = '—'
    print(f'  {camada.name:18s} {camada.__class__.__name__:24s} {str(forma):18s} {camada.count_params():>8,}')
print(f'Total de parâmetros: {modelo.count_params():,} (esperado: 388,868)\n')

x = X_val.astype('float32')
perda, acc = modelo.evaluate(x, y_val, verbose=0)
prob = modelo.predict(x, verbose=0)
pred = prob.argmax(axis=1)

print(f'Perda na validação    : {perda:.3f}  (original: 0.328)')
print(f'Acurácia na validação : {100 * acc:.2f}%  (original: 91.13%)')
print(f'Acertos               : {(pred == y_val).sum()} de {len(y_val)}  (original: 873 de 958)\n')

print('Revocação por classe (original: glioma 90.9 | meningioma 86.8 | notumor 90.5 | pituitary 95.7):')
for i, c in enumerate(CLASSES):
    print(f'  {c:11s} {100 * (pred[y_val == i] == i).mean():.1f}%')

Camadas do modelo:
  imagem             InputLayer               (128, 128, 1)             0
  rescaling_37       Rescaling                (128, 128, 1)             0
  conv2d_116         Conv2D                   (128, 128, 32)          320
  re_lu_116          ReLU                     (128, 128, 32)            0
  max_pooling2d_116  MaxPooling2D             (64, 64, 32)              0
  conv2d_117         Conv2D                   (64, 64, 64)         18,496
  re_lu_117          ReLU                     (64, 64, 64)              0
  max_pooling2d_117  MaxPooling2D             (32, 32, 64)              0
  conv2d_118         Conv2D                   (32, 32, 128)        73,856
  re_lu_118          ReLU                     (32, 32, 128)             0
  max_pooling2d_118  MaxPooling2D             (16, 16, 128)             0
  conv2d_119         Conv2D                   (16, 16, 256)       295,168
  re_lu_119          ReLU                     (16, 16, 256)             0
  max_pooling2d_119

**Observação:** o final.keras foi carregado e avaliado nas 958 imagens de validação reconstruídas, e o resultado coincide com o do Passo 27 do notebook original em todos os números conferidos: 388.868 parâmetros, perda de 0,328, acurácia de 91,13% (873 acertos em 958) e revocação de 90,9% para glioma, 86,8% para meningioma, 90,5% para notumor e 95,7% para pituitary. Essa coincidência prova duas coisas ao mesmo tempo: o modelo gravado é o ponto da época 28 do original, e o preparo das imagens refeito neste notebook é equivalente ao de lá. A partir daqui, todo gráfico que sair deste notebook descreve a rede real do projeto. Os nomes das camadas gravadas no arquivo carregam sufixos automáticos do Keras (por exemplo conv2d_118, re_lu_119 e dense_20), porque o modelo final foi construído por uma função que não as nomeia; por isso as camadas serão sempre localizadas pelo tipo e pela ordem, nunca pelo nome. A rede tem quatro camadas convolucionais (com 32, 64, 128 e 256 filtros), uma camada densa de 4 saídas, e as camadas ReLU, MaxPooling, média global e Dropout não têm parâmetros. O Dropout (taxa 0,3) só atua durante o treino e fica desligado quando a rede é aplicada a imagens, como aqui.

## Fase 2 — Inventário de parâmetros

### Passo 5 — Inventário das camadas: o que a rede tem dentro

Esta célula abre o modelo treinado e monta uma tabela com as camadas que guardam parâmetros: as quatro convolucionais e a camada densa de saída. Para cada uma ela mostra o formato do tensor de pesos (altura, largura, canais de entrada e número de filtros, no caso da convolução), quantos parâmetros são pesos e quantos são biases (viéses), o total da camada e a fatia que ela representa dos 388.868 parâmetros da rede. As demais camadas (ReLU, MaxPooling, média global e Dropout) não têm parâmetros e não entram na tabela. Este inventário é a base dos gráficos de parâmetros, pesos e biases que vêm a seguir.

In [ ]:
# Passo 5 — Inventário das camadas com parâmetros: formato do tensor, pesos, biases e fatia do total
import pandas as pd

total = modelo.count_params()
linhas, n_conv = [], 0
for camada in modelo.layers:
    if not camada.weights:
        continue
    pesos, vieses = camada.get_weights()
    tipo = camada.__class__.__name__
    if tipo == 'Conv2D':
        n_conv += 1
        rotulo = f'Conv {n_conv} ({camada.filters} filtros)'
    else:
        rotulo = 'Dense (4 saídas)'
    linhas.append({
        'Camada': rotulo,
        'Tipo': tipo,
        'Formato dos pesos': str(pesos.shape),
        'Pesos': pesos.size,
        'Biases': vieses.size,
        'Total': pesos.size + vieses.size,
        '% do total': round(100 * (pesos.size + vieses.size) / total, 1),
    })

inv = pd.DataFrame(linhas)
inv.loc[len(inv)] = ['Total', '', '', inv['Pesos'].sum(), inv['Biases'].sum(), inv['Total'].sum(), 100.0]
display(inv.set_index('Camada'))
print(f'\nConferência: soma dos parâmetros = {inv["Total"].iloc[:-1].sum():,} | modelo.count_params() = {total:,}')

,Tipo,Formato dos pesos,Pesos,Biases,Total,% do total
Camada,,,,,,
Conv 1 (32 filtros),Conv2D,"(3, 3, 1, 32)",288,32,320,0.1
Conv 2 (64 filtros),Conv2D,"(3, 3, 32, 64)",18432,64,18496,4.8
Conv 3 (128 filtros),Conv2D,"(3, 3, 64, 128)",73728,128,73856,19.0
Conv 4 (256 filtros),Conv2D,"(3, 3, 128, 256)",294912,256,295168,75.9
Dense (4 saídas),Dense,"(256, 4)",1024,4,1028,0.3
Total,,,388384,484,388868,100.0



Conferência: soma dos parâmetros = 388,868 | modelo.count_params() = 388,868


**Observação:** o inventário confirma, camada por camada, a arquitetura do notebook original. A primeira convolução tem 320 parâmetros (3×3×1×32 = 288 pesos, mais 32 biases, um por filtro); a segunda, 18.496; a terceira, 73.856; a quarta, 295.168; e a camada densa de saída, 1.028 (256×4 = 1.024 pesos, mais 4 biases). A soma fecha em 388.868, igual ao total informado pelo modelo, dos quais 388.384 são pesos e apenas 484 são biases (0,12% do total). A distribuição é muito desigual: a última convolução concentra 75,9% de todos os parâmetros, a terceira 19,0%, a segunda 4,8%, e a primeira e a camada densa juntas ficam abaixo de 0,5%. A razão está na fórmula do número de pesos de uma convolução, 3 × 3 × (canais de entrada) × (filtros): ela cresce com o produto entre os canais que entram e os filtros que saem, e esse produto aumenta a cada bloco (1×32, 32×64, 64×128 e 128×256). O tamanho da imagem não entra na conta, porque o mesmo filtro é reaproveitado em todas as posições; por isso a primeira convolução, que percorre 128×128 posições, tem só 320 parâmetros. Este quadro é a base dos gráficos de parâmetros, pesos e biases que vêm a seguir.

### Passo 6 — Gráfico 07: quantidade de parâmetros por camada

O Gráfico 07 do catálogo mostra quantos parâmetros treináveis cada camada possui, separando pesos e biases. Aqui ele é construído com os números reais do modelo final, extraídos do inventário do Passo 5. Como os biases são uma fração mínima do total (484 de 388.868), o gráfico tem dois painéis: à esquerda, o total de parâmetros de cada camada em escala linear, com o valor e a porcentagem escritos sobre cada barra, para mostrar onde a rede concentra sua capacidade; à direita, pesos e biases lado a lado em escala logarítmica, que permite enxergar os dois grupos mesmo com diferença de ordens de grandeza. O gráfico é interativo: passar o mouse sobre uma barra mostra o valor exato.

In [ ]:
# Passo 6 — Gráfico 07: parâmetros por camada (dados reais do modelo final)
import plotly.graph_objects as go
from plotly.subplots import make_subplots

d = inv.iloc[:-1]                                    # sem a linha Total
rot = [r.split(' (')[0] for r in d['Camada']]        # Conv 1, Conv 2, Conv 3, Conv 4, Dense
pt = lambda v: f'{int(v):,}'.replace(',', '.')       # 295168 -> 295.168

fig = make_subplots(rows=1, cols=2, column_widths=[0.5, 0.5], horizontal_spacing=0.12,
                    subplot_titles=('Total por camada (escala linear)', 'Pesos e biases (escala logarítmica)'))

fig.add_trace(go.Bar(x=rot, y=d['Total'], marker_color='#1F4E79', showlegend=False,
                     text=[f'{pt(t)}<br>{str(p).replace(".", ",")}%' for t, p in zip(d['Total'], d['% do total'])],
                     textposition='outside', hovertemplate='%{x}: %{y:,} parâmetros<extra></extra>'), row=1, col=1)
fig.add_trace(go.Bar(x=rot, y=d['Pesos'], name='Pesos', marker_color='#1F4E79',
                     hovertemplate='%{x}: %{y:,} pesos<extra></extra>'), row=1, col=2)
fig.add_trace(go.Bar(x=rot, y=d['Biases'], name='Biases', marker_color='#C9A227',
                     hovertemplate='%{x}: %{y:,} biases<extra></extra>'), row=1, col=2)

fig.update_yaxes(range=[0, d['Total'].max() * 1.2], title_text='Parâmetros', row=1, col=1)
fig.update_yaxes(type='log', title_text='Parâmetros (log)', row=1, col=2)
fig.update_layout(title='Gráfico 07 — Quantidade de parâmetros por camada (modelo final do projeto)',
                   barmode='group', template='plotly_white', height=520,
                  legend=dict(orientation='h', y=-0.15, x=0.62))
fig.show()

**Observação:** o Gráfico 07, construído com os números reais do modelo final, mostra que a capacidade da rede está concentrada no fim da parte convolucional: a Conv 4 tem 295.168 parâmetros (75,9% do total), a Conv 3 tem 73.856 (19,0%), a Conv 2 tem 18.496 (4,8%), a Conv 1 tem 320 (0,1%) e a camada densa de saída tem 1.028 (0,3%). Nenhuma das camadas do início tem parâmetros em quantidade comparável à última convolução, apesar de trabalharem sobre imagens muito maiores (128×128 contra 16×16): o que pesa é a quantidade de canais e de filtros, e não o tamanho do mapa. No painel da direita, em escala logarítmica, os pesos aparecem sempre acima dos biases, e a distância entre as barras aumenta de camada em camada. Isso tem uma explicação simples: numa convolução 3×3, a razão entre pesos e biases é 9 vezes o número de canais de entrada (9 na Conv 1, 288 na Conv 2, 576 na Conv 3 e 1.152 na Conv 4), e na camada densa é 256. Em conjunto, os biases somam 484 parâmetros, o que explica por que seriam invisíveis em escala linear. A leitura é descritiva: a quantidade de parâmetros mede a capacidade de cada camada, mas não diz quanto cada uma contribui para a decisão da rede; isso será examinado nos gráficos de pesos e de ativações.

### Passo 7 — Gráfico 08: mapa de calor dos pesos

O Gráfico 08 do catálogo usa cores para mostrar o sinal e a intensidade dos pesos. Aqui ele é feito com os pesos reais do modelo final, em dois painéis. No painel de cima estão os 32 filtros 3×3 da primeira convolução: cada coluna é um filtro e cada linha é uma posição dentro do filtro, de (0,0) no canto superior esquerdo a (2,2) no canto inferior direito. No painel de baixo está a matriz de pesos da camada densa final: cada linha é uma das quatro classes e cada coluna é uma das 256 características que chegam da média global. Azul indica peso negativo, vermelho indica peso positivo, e o branco é próximo de zero. Cada painel tem sua própria escala, simétrica em torno do zero, porque os pesos das duas camadas têm magnitudes diferentes; por isso cores iguais em painéis diferentes não representam o mesmo valor. A leitura é descritiva: o sinal e o tamanho de um peso não bastam para dizer quanto uma característica importa para a decisão.

In [ ]:
# Passo 7 — Gráfico 08: mapa de calor dos pesos reais (Conv 1 e camada densa)
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

conv1 = [l for l in modelo.layers if l.__class__.__name__ == 'Conv2D'][0]
densa = [l for l in modelo.layers if l.__class__.__name__ == 'Dense'][0]

W1 = conv1.get_weights()[0]                        # (3, 3, 1, 32)
M1 = W1[:, :, 0, :].reshape(9, 32)                 # linhas: posição (i,j); colunas: filtro
Wd = densa.get_weights()[0].T                      # (4 classes, 256 características)

pos = [f'({i},{j})' for i in range(3) for j in range(3)]
lim1, limd = np.abs(M1).max(), np.abs(Wd).max()

fig = make_subplots(rows=2, cols=1, vertical_spacing=0.2, row_heights=[0.55, 0.45],
                    subplot_titles=('Conv 1: 32 filtros 3×3 (colunas = filtros, linhas = posição no filtro)',
                                    'Camada densa: pesos por classe e por característica (256)'))

fig.add_trace(go.Heatmap(z=M1, x=list(range(1, 33)), y=pos, colorscale='RdBu', reversescale=True,
                         zmin=-lim1, zmax=lim1, xgap=1, ygap=1,
                         colorbar=dict(title='Peso', y=0.8, len=0.4, x=1.02),
                         hovertemplate='filtro %{x}, posição %{y}: %{z:.3f}<extra></extra>'), row=1, col=1)
fig.add_trace(go.Heatmap(z=Wd, x=list(range(1, 257)), y=CLASSES, colorscale='RdBu', reversescale=True,
                         zmin=-limd, zmax=limd,
                         colorbar=dict(title='Peso', y=0.2, len=0.35, x=1.02),
                         hovertemplate='característica %{x}, classe %{y}: %{z:.3f}<extra></extra>'), row=2, col=1)

fig.update_yaxes(autorange='reversed', row=1, col=1)
fig.update_xaxes(title_text='Filtro', row=1, col=1)
fig.update_xaxes(title_text='Característica (saída da média global)', row=2, col=1)
fig.update_layout(title='Gráfico 08 — Mapa de calor dos pesos (modelo final do projeto)',
                  template='plotly_white', height=720)
fig.show()

print(f'Conv 1: pesos de {M1.min():.3f} a {M1.max():.3f} | Camada densa: pesos de {Wd.min():.3f} a {Wd.max():.3f}')

Conv 1: pesos de -1.341 a 1.112 | Camada densa: pesos de -0.578 a 0.571


**Observação:** o painel de cima mostra os 288 pesos da primeira convolução como 32 filtros 3×3 (uma coluna por filtro e uma linha por posição). Os filtros são claramente diferentes entre si e misturam pesos positivos e negativos dentro do mesmo filtro, sem um padrão único; os valores são da ordem de ±1, o que é relativamente grande, porque cada neurônio dessa camada soma apenas 9 entradas. O painel de baixo mostra a camada densa final, com 4 linhas (uma por classe) e 256 colunas (uma por característica que chega da média global). Aqui os valores são menores, da ordem de ±0,5, e a imagem tem o aspecto de listras verticais finas, com sinais misturados em cada linha: nenhuma característica isolada domina visualmente, e cada classe se apoia em muitas características ao mesmo tempo, em vez de depender de poucas. Como as características que chegam à camada densa vêm de uma ReLU seguida de média e, portanto, nunca são negativas, um peso positivo empurra a pontuação da classe para cima quando aquela característica está ativa, e um peso negativo empurra para baixo. Como o softmax só compara as classes entre si, para uma mesma característica o que conta é a diferença entre os pesos das quatro classes, e não o valor absoluto. A leitura é descritiva: o sinal e o tamanho de um peso, vistos isoladamente, não dizem quanto uma característica importa para a decisão, e o mapa de calor dos filtros da primeira camada não revela sozinho o que cada filtro detecta; isso aparecerá nos mapas de ativação.

### Passo 8 — Gráfico 10: distribuição dos pesos por camada (violino e boxplot)

O Gráfico 10 do catálogo resume os pesos de cada camada pela sua distribuição: o violino mostra a forma (onde os pesos se concentram e como se espalham) e o boxplot interno marca a mediana e os quartis. Aqui ele é feito com os pesos reais do modelo final, para as quatro convoluções e a camada densa, todos na mesma escala vertical para que as camadas possam ser comparadas. Como a última convolução tem quase 295 mil pesos, o desenho do violino usa uma amostra aleatória de até 20.000 pesos por camada (semente 42; nas camadas menores, todos os pesos), enquanto a tabela impressa abaixo do gráfico traz as estatísticas exatas (mínimo, quartis, mediana, máximo, média e desvio-padrão) calculadas com todos os pesos. Os biases não entram aqui: eles têm gráfico próprio.

In [ ]:
# Passo 8 — Gráfico 10: distribuição dos pesos por camada (violino + boxplot), pesos reais do modelo final
import numpy as np
import pandas as pd
import plotly.graph_objects as go

rng = np.random.default_rng(42)
camadas = [l for l in modelo.layers if l.weights]
rot = ['Conv 1', 'Conv 2', 'Conv 3', 'Conv 4', 'Dense']

fig = go.Figure()
linhas = []
for r, camada in zip(rot, camadas):
    w = camada.get_weights()[0].ravel()                                   # todos os pesos da camada
    amostra = rng.choice(w, size=min(20000, w.size), replace=False)       # só para desenhar
    fig.add_trace(go.Violin(y=amostra, name=r, box_visible=True, points=False, showlegend=False,
                            line_color='#1F4E79', fillcolor='rgba(31,78,121,0.35)',
                            meanline=dict(visible=True, color='#C9A227', width=2)))
    q1, med, q3 = np.percentile(w, [25, 50, 75])
    linhas.append({'Camada': r, 'Pesos': w.size, 'Mínimo': w.min(), 'Q1': q1, 'Mediana': med,
                   'Q3': q3, 'Máximo': w.max(), 'Média': w.mean(), 'Desvio-padrão': w.std()})

fig.update_layout(title='Gráfico 10 — Distribuição dos pesos por camada (modelo final do projeto)',
                  yaxis_title='Peso', template='plotly_white', height=520)
fig.show()

tab = pd.DataFrame(linhas).set_index('Camada')
display(tab.round(4))

,Pesos,Mínimo,Q1,Mediana,Q3,Máximo,Média,Desvio-padrão
Camada,,,,,,,,
Conv 1,288,-1.3414,-0.3864,0.0317,0.3741,1.1120,0.0118,0.5223
Conv 2,18432,-0.8529,-0.0930,-0.0055,0.0824,0.8012,-0.0074,0.1419
Conv 3,73728,-0.8520,-0.0873,-0.0197,0.0431,0.7388,-0.0264,0.1099
Conv 4,294912,-0.6584,-0.0635,-0.0121,0.0369,0.7476,-0.0152,0.0874
Dense,1024,-0.5784,-0.1327,0.0118,0.1142,0.5709,-0.0075,0.1604


Quanto à forma, o gráfico mostra que a Conv 1 tem um corpo largo, com a caixa central indo de cerca de −0,4 a +0,35, o que indica pesos bem espalhados. Nas Conv 2, 3 e 4 o corpo é estreito e concentrado em torno de zero, com caudas longas e finas, e a camada densa fica em um meio-termo, com corpo um pouco mais largo. Todos os violinos são aproximadamente simétricos em relação ao zero, sem tendência clara para pesos positivos ou negativos. As pontas dos violinos passam um pouco dos extremos reais, porque o gráfico suaviza a curva; os valores exatos são os da tabela.

### Passo 9 — Gráfico 12: distribuição dos biases por camada

O Gráfico 12 do catálogo mostra, para cada camada, como os biases (os termos aditivos de cada filtro ou neurônio) se distribuem. Aqui ele é feito com os biases reais do modelo final: 32 na Conv 1, 64 na Conv 2, 128 na Conv 3, 256 na Conv 4 e 4 na camada densa, 484 no total. Como são poucos, cada bias é desenhado como um ponto sobre o boxplot da sua camada, e a linha pontilhada marca o zero. Um dado importante para a interpretação: no Keras, os biases das camadas começam em zero, de modo que qualquer valor diferente de zero foi aprendido durante o treino. A tabela impressa abaixo do gráfico traz as estatísticas exatas e a contagem de biases positivos e negativos por camada.

In [ ]:
# Passo 9 — Gráfico 12: distribuição dos biases por camada (modelo final)
import numpy as np
import pandas as pd
import plotly.graph_objects as go

camadas = [l for l in modelo.layers if l.weights]
rot = ['Conv 1', 'Conv 2', 'Conv 3', 'Conv 4', 'Dense']

fig = go.Figure()
linhas = []
for r, camada in zip(rot, camadas):
    b = camada.get_weights()[1]                                   # biases da camada
    fig.add_trace(go.Box(y=b, name=r, boxpoints='all', jitter=0.5, pointpos=0, showlegend=False,
                         marker=dict(color='#C9A227', size=5, opacity=0.8),
                         line=dict(color='#1F4E79'), fillcolor='rgba(31,78,121,0.15)'))
    q1, med, q3 = np.percentile(b, [25, 50, 75])
    linhas.append({'Camada': r, 'Biases': b.size, 'Mínimo': b.min(), 'Q1': q1, 'Mediana': med, 'Q3': q3,
                   'Máximo': b.max(), 'Média': b.mean(), 'Desvio-padrão': b.std(),
                   'Positivos': int((b > 0).sum()), 'Negativos': int((b < 0).sum())})

fig.add_hline(y=0, line_dash='dot', line_color='gray')
fig.update_layout(title='Gráfico 12 — Distribuição dos biases por camada (modelo final do projeto)',
                  yaxis_title='Bias', template='plotly_white', height=520)
fig.show()

tab = pd.DataFrame(linhas).set_index('Camada')
display(tab.round(4))

,Biases,Mínimo,Q1,Mediana,Q3,Máximo,Média,Desvio-padrão,Positivos,Negativos
Camada,,,,,,,,,,
Conv 1,32,-0.0548,-0.0154,-0.0071,0.0035,0.0689,-0.0022,0.0276,12,20
Conv 2,64,-0.2192,-0.0468,-0.0061,0.0290,0.1165,-0.0091,0.0622,24,40
Conv 3,128,-0.2551,-0.0660,-0.0222,0.0227,0.1018,-0.0286,0.0643,45,83
Conv 4,256,-0.1999,-0.0554,0.0007,0.0297,0.1501,-0.0127,0.0648,131,125
Dense,4,-0.0947,-0.0941,-0.0143,0.0815,0.1300,0.0017,0.0987,2,2


**Observação:** a tabela confirma que os biases são pequenos: os mínimos e máximos ficam entre −0,2551 (Conv 3) e +0,1501 (Conv 4). Como os biases começam em zero no Keras, tudo isso foi aprendido no treino. A Conv 1 é a mais concentrada (32 biases, de −0,0548 a +0,0689, desvio-padrão de 0,0276). A partir da Conv 2 a dispersão sobe e depois se estabiliza: o desvio-padrão é 0,0622 na Conv 2, 0,0643 na Conv 3 e 0,0648 na Conv 4, ou seja, não cresce junto com o número de filtros (64, 128, 256). A camada densa tem só 4 biases, um por classe, com desvio de 0,0987; dois deles estão quase iguais ao mínimo (−0,0947 e −0,0941), e o quartil Q1 coincide com o mínimo por esse motivo. Comparando com os pesos do Passo 8, na Conv 1 os pesos (desvio 0,522) são cerca de 19 vezes mais dispersos que os biases, mas essa distância diminui nas camadas profundas: na Conv 4 os desvios são 0,087 (pesos) e 0,065 (biases), de ordem parecida. As médias das camadas convolucionais são levemente negativas (de −0,0022 a −0,0286), com a Conv 3 mais deslocada, e as medianas também ficam abaixo de zero, exceto na Conv 4 (+0,0007). Um bias negativo exige uma soma maior para o filtro "ligar", então isso sugere uma tendência leve de ativação mais exigente nessas camadas, mas é só uma leitura descritiva: os números sozinhos não provam esse efeito, e a tabela abaixo não diz quais biases importam para o resultado.

### Passo 10 — Gráfico 09: distribuição 3D dos pesos por camada

O Gráfico 09 do catálogo coloca as distribuições de pesos das camadas lado a lado em três dimensões. Aqui ele é feito com os pesos reais do modelo final. O eixo horizontal é o valor do peso, o eixo de profundidade são as cinco camadas com pesos (Conv 1 a Conv 4 e Dense), e a altura é a densidade, isto é, quão concentrados estão os pesos em cada faixa de valores. A densidade é normalizada para que cada camada tenha área total igual a 1, o que permite comparar camadas com números muito diferentes de pesos (288 na Conv 1 contra 294.912 na Conv 4). Cada camada aparece como uma faixa separada, sem unir uma à outra: o espaço entre as faixas fica vazio de propósito, porque não existem dados entre duas camadas. O gráfico é interativo: arraste com o mouse para girar, use a roda para dar zoom e passe o cursor sobre a superfície para ler o valor do peso e a densidade. Abaixo dele, uma pequena tabela traz a densidade no pico de cada camada e a porcentagem de pesos entre −0,1 e +0,1.

In [ ]:
# Passo 10 — Gráfico 09: distribuição 3D dos pesos por camada (modelo final)
import numpy as np
import pandas as pd
import plotly.graph_objects as go

camadas = [l for l in modelo.layers if l.weights]
rot = ['Conv 1', 'Conv 2', 'Conv 3', 'Conv 4', 'Dense']

limites = np.linspace(-1.4, 1.2, 66)                 # 65 faixas de largura 0,04
centros = (limites[:-1] + limites[1:]) / 2

y_pos, z_linhas, resumo = [], [], []
for i, (r, camada) in enumerate(zip(rot, camadas)):
    w = camada.get_weights()[0].ravel()               # todos os pesos da camada
    dens, _ = np.histogram(w, bins=limites, density=True)
    y_pos += [i - 0.3, i + 0.3, i + 0.5]              # faixa da camada + linha vazia de separação
    z_linhas += [dens, dens, np.full_like(dens, np.nan)]
    resumo.append({'Camada': r, 'Pesos': w.size, 'Densidade no pico': dens.max(),
                   '% dos pesos entre -0,1 e +0,1': 100 * np.mean(np.abs(w) < 0.1)})

fig = go.Figure(go.Surface(x=centros, y=y_pos[:-1], z=np.array(z_linhas[:-1]),
                           colorscale=[[0, '#F3F6FA'], [0.5, '#1F4E79'], [1, '#C9A227']],
                           colorbar=dict(title='Densidade'),
                           hovertemplate='Peso: %{x:.2f}<br>Densidade: %{z:.2f}<extra></extra>'))
fig.update_layout(title='Gráfico 09 — Distribuição 3D dos pesos por camada (modelo final do projeto)',
                  template='plotly_white', height=650,
                  scene=dict(xaxis_title='Valor do peso', yaxis_title='Camada', zaxis_title='Densidade',
                             yaxis=dict(tickmode='array', tickvals=list(range(5)), ticktext=rot),
                             camera=dict(eye=dict(x=1.6, y=-1.8, z=0.9))))
fig.show()

tab = pd.DataFrame(resumo).set_index('Camada')
display(tab.round(2))

,Pesos,Densidade no pico,"% dos pesos entre -0,1 e +0,1"
Camada,,,
Conv 1,288,1.48,10.42
Conv 2,18432,3.14,55.57
Conv 3,73728,4.14,69.08
Conv 4,294912,5.30,79.21
Dense,1024,2.73,39.94


**Observação:** o gráfico mostra que as distribuições de pesos se concentram cada vez mais perto de zero ao longo das camadas convolucionais. A densidade no pico sobe de 1,48 na Conv 1 para 3,14 na Conv 2 e 4,14 na Conv 3, e na Conv 4 o pico é ainda mais alto, o maior do gráfico. A camada densa fica abaixo das camadas profundas, o que é coerente com o desvio-padrão maior dos seus pesos (0,160, contra 0,087 na Conv 4). A tabela confirma o mesmo movimento: a porcentagem de pesos entre −0,1 e +0,1 passa de 10,42% na Conv 1 para 55,57% na Conv 2 e 69,08% na Conv 3, ou seja, na Conv 3 mais de dois terços dos 73.728 pesos são muito pequenos. Isso combina com os desvios-padrão do Passo 8 (0,522, 0,142, 0,110, 0,087). Os picos das Conv 2 e 3 são mais altos do que seriam em uma distribuição normal com o mesmo desvio-padrão (cerca de 2,8 e 3,6, pela fórmula teórica), o que indica um centro mais afunilado e caudas longas, como já aparecia nos violinos. A Conv 1 é a mais baixa e mais espalhada, mas também a mais irregular, porque tem apenas 288 pesos distribuídos em faixas de 0,04 de largura, e o valor do pico dela deve ser lido com cautela. Cada camada é uma faixa separada: o espaço vazio entre elas não é dado, é só separação visual. A leitura é descritiva e não prova, por si só, que pesos concentrados perto de zero sejam bons ou ruins para a classificação.

### Passo 11 — Gráfico 11: os filtros convolucionais em 3D

O Gráfico 11 do catálogo mostra os filtros de uma camada convolucional em três dimensões. Aqui ele é feito com os filtros reais da primeira camada do modelo final. A Conv 1 tem 32 filtros, e cada um é uma pequena grade de 3×3 pesos (a imagem de entrada tem um só canal, por isso cada filtro tem profundidade 1). No gráfico, cada filtro vira nove barras sobre uma grade de 3 linhas por 3 colunas: a altura da barra é o valor do peso, e a cor segue a mesma convenção do Gráfico 08, vermelho para pesos positivos e azul para negativos, com a mesma escala de cores para os 32 filtros, de modo que as cores sejam comparáveis entre eles. A barra deslizante troca de filtro, e o botão "Percorrer os 32 filtros" faz isso automaticamente, um filtro por vez. Você também pode girar o gráfico com o mouse e dar zoom com a roda. Abaixo do gráfico, o código imprime os nove pesos numéricos do filtro 1, para você conferir com as barras iniciais, e uma tabela com os 8 filtros de maior peso em valor absoluto.

In [ ]:
# Passo 11 — Gráfico 11: os 32 filtros 3x3 da primeira camada convolucional (modelo final)
import numpy as np
import pandas as pd
import plotly.graph_objects as go

W1 = [l for l in modelo.layers if l.weights][0].get_weights()[0]     # forma (3, 3, 1, 32)
nf = W1.shape[3]
K = [W1[:, :, 0, f] for f in range(nf)]                               # 32 filtros 3x3
vmax = float(np.ceil(np.abs(W1).max() * 10) / 10)                     # escala comum a todos os filtros

# geometria fixa: uma barra (caixa) por posição do filtro 3x3
X, Y, I, J, Kk = [], [], [], [], []
for b, (r, c) in enumerate([(r, c) for r in range(3) for c in range(3)]):
    x0, x1, y0, y1 = c - 0.4, c + 0.4, r - 0.4, r + 0.4
    X += [x0, x1, x1, x0] * 2
    Y += [y0, y0, y1, y1] * 2
    o = 8 * b
    for (a, bb, cc) in [(0,1,2),(0,2,3),(4,5,6),(4,6,7),(0,1,5),(0,5,4),(1,2,6),(1,6,5),(2,3,7),(2,7,6),(3,0,4),(3,4,7)]:
        I.append(o + a); J.append(o + bb); Kk.append(o + cc)

def alturas(M):
    z, cor = [], []
    for r in range(3):
        for c in range(3):
            h = float(M[r, c])
            z += [0, 0, 0, 0, h, h, h, h]
            cor += [h] * 8
    return z, cor

z0, c0 = alturas(K[0])
malha = go.Mesh3d(x=X, y=Y, z=z0, i=I, j=J, k=Kk, intensity=c0,
                  colorscale='RdBu', reversescale=True, cmin=-vmax, cmax=vmax,
                  colorbar=dict(title='Peso'), flatshading=True,
                  hovertemplate='Peso: %{intensity:.3f}<extra></extra>')

quadros = []
for f in range(nf):
    z, cor = alturas(K[f])
    quadros.append(go.Frame(name=str(f + 1), data=[go.Mesh3d(z=z, intensity=cor)]))

fig = go.Figure(data=[malha], frames=quadros)
passos = [dict(method='animate', label=str(f + 1),
               args=[[str(f + 1)], dict(mode='immediate', frame=dict(duration=0, redraw=True), transition=dict(duration=0))])
          for f in range(nf)]
fig.update_layout(
    title='Gráfico 11 — Os 32 filtros 3×3 da primeira camada convolucional (modelo final do projeto)',
    template='plotly_white', height=700,
    scene=dict(xaxis=dict(title='Coluna', tickvals=[0, 1, 2], ticktext=['1', '2', '3']),
               yaxis=dict(title='Linha', tickvals=[0, 1, 2], ticktext=['1', '2', '3'], autorange='reversed'),
               zaxis=dict(title='Peso', range=[-vmax, vmax]),
               camera=dict(eye=dict(x=1.7, y=-1.7, z=1.2)), aspectmode='manual', aspectratio=dict(x=1, y=1, z=0.8)),
    sliders=[dict(active=0, steps=passos, currentvalue=dict(prefix='Filtro '), pad=dict(t=90))],
    updatemenus=[dict(type='buttons', direction='left', x=0.0, y=0.0, xanchor='left', yanchor='top', pad=dict(t=30),
                      buttons=[dict(label='▶ Percorrer os 32 filtros', method='animate',
                                    args=[None, dict(frame=dict(duration=900, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
                               dict(label='⏸ Pausar', method='animate',
                                    args=[[None], dict(mode='immediate', frame=dict(duration=0, redraw=False), transition=dict(duration=0))])])])
fig.show()

print('Filtro 1 (linhas x colunas), os mesmos 9 pesos das barras iniciais:')
print(np.round(K[0], 3))

tab = pd.DataFrame({'Filtro': np.arange(1, nf + 1),
                    'Menor peso': [m.min() for m in K], 'Maior peso': [m.max() for m in K],
                    'Soma dos 9 pesos': [m.sum() for m in K],
                    'Maior |peso|': [np.abs(m).max() for m in K]}).set_index('Filtro')
print('\nOs 8 filtros com maior |peso|:')
display(tab.sort_values('Maior |peso|', ascending=False).head(8).round(3))

Filtro 1 (linhas x colunas), os mesmos 9 pesos das barras iniciais:
[[-0.587 -1.216 -0.426]
 [-0.079 -1.236 -0.383]
 [-1.146 -1.341 -0.125]]

Os 8 filtros com maior |peso|:


,Menor peso,Maior peso,Soma dos 9 pesos,Maior |peso|
Filtro,,,,
1,-1.341,-0.079,-6.539,1.341
14,-0.694,1.112,-0.034,1.112
4,-0.101,1.082,4.519,1.082
31,-0.903,1.034,-0.045,1.034
18,-0.356,1.032,3.746,1.032
28,-0.975,1.028,2.273,1.028
10,-1.013,0.898,-0.036,1.013
12,-0.857,1.005,-0.034,1.005


**Observação:** o gráfico mostra que cada um dos 32 filtros da Conv 1 é uma grade minúscula de apenas nove pesos (32 × 9 = 288 pesos, como no inventário do Passo 5), e que esses nove números são tudo que a rede usa, em cada filtro, para olhar uma vizinhança de 3×3 pixels da imagem. A escala de cores é comum a todos os filtros e vai de cerca de −1,4 a +1,4, de acordo com a faixa de pesos que vimos no Passo 7 (de −1,34 a +1,11), o que permite comparar a intensidade de um filtro com a de outro apenas pela cor e pela altura das barras. O filtro que aparece na tela tem barras predominantemente azuis, isto é, pesos negativos ou próximos de zero, com uma barra bem mais profunda que as demais; ao passar de um filtro para outro, o padrão de altura e de cor muda, o que mostra que os filtros não são cópias uns dos outros. Há uma limitação importante: nove barras mostram os valores dos pesos, mas não dizem o que o filtro detecta. Para saber se um filtro responde a bordas, bordas escuras ou regiões claras, é preciso aplicá-lo a uma imagem e olhar o mapa de ativação, o que faremos mais adiante no Gráfico 15.

## Fase 3 — Ativações: o que a rede faz com o que calcula

Até aqui olhamos para os parâmetros da rede, que são os números guardados depois do treino. A partir desta fase olhamos para o que a rede produz quando recebe uma imagem. O primeiro elemento é a função de ativação: depois de cada soma ponderada, ela decide o que passa para a camada seguinte.

### Passo 12 — Gráfico 13: funções de ativação e suas derivadas

O Gráfico 13 do catálogo compara funções de ativação. Aqui ele aparece em dois painéis lado a lado: à esquerda, a função f(z), isto é, a saída em função da soma ponderada z; à direita, a derivada f'(z), isto é, a inclinação da curva, que é o que o aprendizado usa para decidir quanto ajustar cada peso. São seis funções: ReLU, Leaky ReLU, ELU, GELU, sigmoide e tanh. A ReLU, em dourado e mais grossa, é a que o nosso modelo usa depois de cada uma das quatro convoluções. A camada de saída usa a softmax, que não aparece aqui porque não é uma curva de um número só: ela transforma as quatro saídas em probabilidades que somam 1. O gráfico é animado: a barra deslizante, ou o botão "Mover z", faz um ponto percorrer cada curva, para você ler a saída e a inclinação de cada função para o mesmo z. Antes do gráfico, o código confirma no próprio modelo quantas camadas ReLU ele tem e qual é a ativação da saída.

In [ ]:
# Passo 12 — Gráfico 13: funções de ativação e suas derivadas
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.special import erf

# o que o modelo realmente usa
n_relu = sum(isinstance(l, keras.layers.ReLU) for l in modelo.layers)
print(f'Camadas ReLU no modelo: {n_relu}')
print(f'Ativação da camada de saída: {modelo.layers[-1].activation.__name__}')

x = np.linspace(-5, 5, 401)
sig = lambda z: 1 / (1 + np.exp(-z))
fun = {
    'ReLU (usada no modelo)': (lambda z: np.maximum(0, z),            lambda z: (z > 0) * 1.0),
    'Leaky ReLU':             (lambda z: np.where(z > 0, z, 0.1 * z), lambda z: np.where(z > 0, 1.0, 0.1)),
    'ELU':                    (lambda z: np.where(z > 0, z, np.exp(z) - 1), lambda z: np.where(z > 0, 1.0, np.exp(z))),
    'GELU':                   (lambda z: 0.5 * z * (1 + erf(z / np.sqrt(2))),
                               lambda z: 0.5 * (1 + erf(z / np.sqrt(2))) + z * np.exp(-z**2 / 2) / np.sqrt(2 * np.pi)),
    'Sigmoide':               (sig,        lambda z: sig(z) * (1 - sig(z))),
    'Tanh':                   (np.tanh,    lambda z: 1 - np.tanh(z) ** 2),
}
cores = ['#C9A227', '#1F4E79', '#2E8B57', '#8E44AD', '#C0392B', '#7F8C8D']

fig = make_subplots(rows=1, cols=2, subplot_titles=('Função  f(z)', 'Derivada  f\'(z)'), horizontal_spacing=0.08)
for (nome, (f, d)), cor in zip(fun.items(), cores):
    larg = 9 if nome.startswith('ReLU') else 2.5
    fig.add_trace(go.Scatter(x=x, y=f(x), name=nome, mode='lines', line=dict(color=cor, width=larg), legendgroup=nome), row=1, col=1)
    fig.add_trace(go.Scatter(x=x, y=d(x), name=nome, mode='lines', line=dict(color=cor, width=larg), legendgroup=nome, showlegend=False), row=1, col=2)

# marcadores móveis (um por função, em cada painel)
z0 = -5.0
for k, ((nome, (f, d)), cor) in enumerate(zip(fun.items(), cores)):
    fig.add_trace(go.Scatter(x=[z0], y=[f(np.array([z0]))[0]], mode='markers', showlegend=False, legendgroup=nome,
                             marker=dict(color=cor, size=11, line=dict(color='white', width=2)),
                             hovertemplate=nome + '<br>f(z) = %{y:.3f}<extra></extra>'), row=1, col=1)
    fig.add_trace(go.Scatter(x=[z0], y=[d(np.array([z0]))[0]], mode='markers', showlegend=False, legendgroup=nome,
                             marker=dict(color=cor, size=11, line=dict(color='white', width=2)),
                             hovertemplate=nome + '<br>f\'(z) = %{y:.3f}<extra></extra>'), row=1, col=2)

indices = list(range(12, 24))
valores = np.round(np.arange(-5, 5.01, 0.25), 2)
quadros, passos = [], []
for v in valores:
    dados = []
    for nome, (f, d) in fun.items():
        dados += [go.Scatter(x=[v], y=[f(np.array([v]))[0]]), go.Scatter(x=[v], y=[d(np.array([v]))[0]])]
    quadros.append(go.Frame(name=str(v), data=dados, traces=indices))
    passos.append(dict(method='animate', label=f'{v:g}',
                       args=[[str(v)], dict(mode='immediate', frame=dict(duration=0, redraw=True), transition=dict(duration=0))]))
fig.frames = quadros

fig.update_xaxes(title_text='Entrada z (soma ponderada)', range=[-5, 5], zeroline=True)
fig.update_yaxes(range=[-1.5, 5], row=1, col=1, title_text='Saída')
fig.update_yaxes(range=[-0.3, 1.3], row=1, col=2, title_text='Inclinação')
fig.update_layout(title='Gráfico 13 — Funções de ativação e suas derivadas (a destacada é a usada nas convoluções do modelo)',
                  template='plotly_white', height=720, margin=dict(b=200), legend=dict(orientation='h', y=-0.62),
                  sliders=[dict(active=0, steps=passos, currentvalue=dict(prefix='z = '), pad=dict(t=30), y=-0.12)],
                  updatemenus=[dict(type='buttons', direction='left', x=0.0, y=-0.12, xanchor='left', yanchor='top', pad=dict(t=0, r=10),
                                    buttons=[dict(label='▶ Mover z', method='animate',
                                                  args=[None, dict(frame=dict(duration=250, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
                                             dict(label='⏸ Pausar', method='animate',
                                                  args=[[None], dict(mode='immediate', frame=dict(duration=0, redraw=False), transition=dict(duration=0))])])])
fig.show()

Camadas ReLU no modelo: 4
Ativação da camada de saída: softmax


**Observação:** o gráfico mostra duas famílias de comportamento. A ReLU, que é a função usada depois de cada convolução do modelo, vale zero para qualquer soma ponderada negativa e é igual à própria soma quando ela é positiva; por isso sua derivada é 0 do lado esquerdo e 1 do lado direito, com um salto em z = 0. Em termos práticos, quando z é negativo o neurônio fica "desligado" e não passa nada adiante (nem recebe ajuste no aprendizado), e quando z é positivo ele repassa o valor sem alterá-lo. A Leaky ReLU deixa passar uma fração pequena do lado negativo, com inclinação constante de 0,1, e a ELU e a GELU suavizam a transição perto de zero; a GELU chega a ter derivada ligeiramente negativa por volta de z = −1,5 e ligeiramente acima de 1 depois de zero. As outras duas, sigmoide e tanh, são curvas que saturam: a sigmoide fica entre 0 e 1 e a tanh entre −1 e 1, e nas duas a derivada tende a zero nas pontas (z muito negativo ou muito positivo), com pico de 0,25 na sigmoide e de 1 na tanh, ambos em z = 0. Em redes com muitas camadas, derivadas muito pequenas multiplicadas camada após camada dificultam o aprendizado, e é uma das razões da popularidade da ReLU, cuja derivada vale 1 em toda a metade positiva. Esta leitura compara as formas das curvas e não mede, por si só, qual função é melhor para o nosso problema.

### Passo 13 — Gráfico 14: a superfície de ativação de um neurônio

O Gráfico 14 do catálogo mostra, em três dimensões, o que um neurônio faz com suas entradas. Aqui ele é feito com os filtros reais da primeira camada. Cada filtro olha para nove pixels ao mesmo tempo; para poder desenhar em 3D, escolhemos os dois pixels (A e B) com maior peso em valor absoluto em cada filtro, deixamos variar de 0 (preto) a 1 (branco), que é a escala da entrada depois do Rescaling, e fixamos os outros sete pixels em 0. Com isso, a soma ponderada do neurônio é z = peso A · A + peso B · B + bias, com os valores reais do modelo. No gráfico há duas superfícies: a cinza e transparente é z, antes da ativação, que é sempre um plano; a colorida é a saída depois da ReLU, o plano "dobrado" que corta tudo que fica abaixo de zero. Onde o plano cinza fica acima de zero as duas coincidem; onde ele fica abaixo, a superfície colorida fica achatada em zero, e é aí que o neurônio está "desligado". A barra deslizante e o botão "Percorrer os 32 filtros" trocam de filtro, o título mostra a fórmula do filtro mostrado, e você pode girar o gráfico com o mouse. A tabela impressa abaixo traz, para cada filtro, os dois pixels usados, os pesos, o bias e a porcentagem da grade de entradas em que o neurônio fica ligado. É um recorte de duas entradas dentro de nove, e serve para entender a mecânica da ativação, não para resumir o filtro inteiro.

In [ ]:
# Passo 13 — Gráfico 14: superfície de ativação de um neurônio da Conv 1 (modelo final)
import numpy as np
import pandas as pd
import plotly.graph_objects as go

conv1 = [l for l in modelo.layers if l.weights][0]
W1, b1 = conv1.get_weights()                       # W1: (3, 3, 1, 32)   b1: (32,)
nf = W1.shape[3]

g = np.linspace(0, 1, 41)                          # entrada normalizada: 0 = preto, 1 = branco
X1, X2 = np.meshgrid(g, g)

dados, linhas = [], []
for f in range(nf):
    w = W1[:, :, 0, f].ravel()
    a, b = np.argsort(-np.abs(w))[:2]              # os dois pixels com maior |peso| do filtro
    z = w[a] * X1 + w[b] * X2 + b1[f]              # soma ponderada (os outros 7 pixels fixos em 0)
    dados.append((z, np.maximum(0, z), a, b, w[a], w[b], b1[f]))
    linhas.append({'Filtro': f + 1, 'Pixel A (lin,col)': f'({a//3+1},{a%3+1})', 'Peso A': w[a],
                   'Pixel B (lin,col)': f'({b//3+1},{b%3+1})', 'Peso B': w[b], 'Bias': b1[f],
                   '% da grade com neurônio ligado': 100 * np.mean(z > 0)})

zmin = min(d[0].min() for d in dados); zmax = max(d[0].max() for d in dados)

def titulo(f):
    z, r, a, b, wa, wb, bias = dados[f]
    return f'Gráfico 14 — Filtro {f+1}: z = ({wa:.2f})·A + ({wb:.2f})·B + ({bias:.2f}),  saída = ReLU(z)'

plano = go.Surface(x=g, y=g, z=dados[0][0], opacity=0.35, showscale=False, colorscale=[[0, '#9AA5B1'], [1, '#9AA5B1']],
                   name='z (antes da ReLU)', hovertemplate='A=%{x:.2f}  B=%{y:.2f}<br>z=%{z:.3f}<extra>antes da ReLU</extra>')
relu = go.Surface(x=g, y=g, z=dados[0][1], colorscale=[[0, '#1F4E79'], [1, '#C9A227']], cmin=0, cmax=max(zmax, 0.01),
                  colorbar=dict(title='ReLU(z)'), name='ReLU(z)',
                  hovertemplate='A=%{x:.2f}  B=%{y:.2f}<br>saída=%{z:.3f}<extra>depois da ReLU</extra>')

quadros = [go.Frame(name=str(f + 1), traces=[0, 1], data=[go.Surface(z=dados[f][0]), go.Surface(z=dados[f][1])],
                    layout=go.Layout(title_text=titulo(f))) for f in range(nf)]
passos = [dict(method='animate', label=str(f + 1),
               args=[[str(f + 1)], dict(mode='immediate', frame=dict(duration=0, redraw=True), transition=dict(duration=0))])
          for f in range(nf)]

fig = go.Figure(data=[plano, relu], frames=quadros)
fig.update_layout(title=titulo(0), template='plotly_white', height=720,
                  scene=dict(xaxis_title='Pixel A', yaxis_title='Pixel B',
                             zaxis=dict(title='z e saída', range=[min(zmin, 0), max(zmax, 0.01)]),
                             camera=dict(eye=dict(x=1.7, y=-1.7, z=0.9))),
                  sliders=[dict(active=0, steps=passos, currentvalue=dict(prefix='Filtro '), pad=dict(t=90))],
                  updatemenus=[dict(type='buttons', direction='left', x=0.0, y=0.0, xanchor='left', yanchor='top', pad=dict(t=30),
                                    buttons=[dict(label='▶ Percorrer os 32 filtros', method='animate',
                                                  args=[None, dict(frame=dict(duration=900, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
                                             dict(label='⏸ Pausar', method='animate',
                                                  args=[[None], dict(mode='immediate', frame=dict(duration=0, redraw=False), transition=dict(duration=0))])])])
fig.show()

tab = pd.DataFrame(linhas).set_index('Filtro')
display(tab.round(3))

,"Pixel A (lin,col)",Peso A,"Pixel B (lin,col)",Peso B,Bias,% da grade com neurônio ligado
Filtro,,,,,,
1,"(3,2)",-1.341,"(2,2)",-1.236,0.069,0.357
2,"(3,1)",-0.848,"(2,1)",-0.587,-0.012,0.000
3,"(1,2)",-0.698,"(2,1)",0.630,-0.046,39.024
4,"(2,1)",1.082,"(2,2)",0.899,0.001,100.000
5,"(3,2)",0.893,"(2,2)",-0.636,-0.009,63.058
6,"(3,1)",-0.851,"(1,1)",0.619,-0.055,30.637
7,"(3,1)",-0.926,"(3,3)",0.827,-0.021,42.594
8,"(2,2)",-0.663,"(1,1)",0.609,-0.008,44.854
9,"(3,1)",0.932,"(2,2)",-0.522,0.016,73.111


**Observação:** a superfície mostra o que a ReLU faz com a soma ponderada de dois pixels. No filtro 1, os dois pesos são negativos (−1,341 e −1,236) e o bias é pequeno e positivo (0,069); por isso z só fica acima de zero quando os dois pixels estão praticamente pretos, e o neurônio fica ligado em apenas 0,357% da grade de entradas, com a superfície colorida achatada em zero em quase toda a região (por exemplo, com A = 0,35 e B = 0,05 o valor de z é −0,462 e a saída é zero). O filtro 2 é parecido, com os dois pesos negativos (−0,848 e −0,587) e 0,000% de grade ligada. O comportamento oposto aparece nos filtros 4 e 13, que têm os dois pesos positivos (1,082 e 0,899; 0,660 e 0,645) e ficam ligados em 100,000% e 99,822% da grade: nesses casos a ReLU quase não corta nada e o neurônio repassa o brilho dos pixels. A maior parte dos filtros que aparecem na tabela (do 3 ao 12, o 14 e o 15) tem um peso positivo e outro negativo, e fica ligada em frações intermediárias da grade, de 30,637% (filtro 6) a 73,111% (filtro 9). Esses neurônios respondem à diferença entre os dois pixels e não ao brilho de cada um: crescem quando o pixel de peso positivo é mais claro que o de peso negativo, e a ReLU corta o resto. Os biases vistos são pequenos (de −0,055 a +0,069) e, por isso, quem decide o formato da superfície são os pesos, e o bias só desloca o ponto de corte. Duas ressalvas: a porcentagem de grade ligada depende da escolha de fixar os outros sete pixels em 0 e de uma grade uniforme de 0 a 1, e imagens reais não têm pixels distribuídos assim; e o gráfico mostra dois dos nove pixels de cada filtro, então não resume o comportamento completo dele. O que o Gráfico 15 vai mostrar, com imagens reais, é o que acontece quando os nove pixels variam juntos.

### Passo 14 — As imagens de exemplo

Os próximos gráficos da fase (15, 16 e 17) mostram o que acontece dentro da rede quando ela recebe um exame de verdade. Para isso precisamos de exemplos fixos. Aqui escolhemos cinco imagens do conjunto de validação, nunca do teste: uma de cada uma das quatro classes (glioma, meningioma, sem tumor e pituitária), sorteadas com semente fixa entre as que têm formato original quadrado, e uma quinta imagem de "sem tumor" com formato original retangular. Essa quinta serve para ilustrar o atalho que apareceu no projeto original: imagens retangulares ganham bandas pretas ao serem preparadas para 128×128, e o formato acabou associado à classe "sem tumor". O sorteio é repetível, então os mesmos exemplos aparecem toda vez que o notebook for executado. O código mostra as cinco imagens como a rede as vê (já em escala de cinza e com 128×128 pixels), imprime uma tabela com o tamanho original do arquivo, a previsão do modelo e a probabilidade que ele deu a cada classe, e guarda as imagens escolhidas para os próximos passos. Um aviso: o modelo acerta cerca de 91% da validação, então é possível que algum exemplo seja classificado errado, e isso também será informativo.

In [ ]:
# Passo 14 — As 5 imagens de exemplo (validação) e a previsão do modelo
import numpy as np
import pandas as pd
from PIL import Image
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# tamanho original de cada imagem de validação (só o cabeçalho é lido, não os pixels)
tam = [Image.open(c).size for c in val['caminho']]
val_t = val.copy()
val_t['largura'] = [t[0] for t in tam]
val_t['altura'] = [t[1] for t in tam]
val_t['quadrada'] = val_t['largura'] == val_t['altura']

# 1 imagem por classe (originais quadradas), sorteio fixo e repetível
rng = np.random.default_rng(42)
idx_ex, nomes = [], []
for cls in CLASSES:
    cand = val_t[(val_t['classe'] == cls) & val_t['quadrada']]
    if len(cand) == 0:
        cand = val_t[val_t['classe'] == cls]
    idx_ex.append(int(rng.choice(cand.index.values)))
    nomes.append(f'Exemplo — {cls}')

# 5ª imagem: "sem tumor" retangular (com bandas pretas ao preparar), para ilustrar o atalho do projeto original
cand = val_t[(val_t['classe'] == 'notumor') & (~val_t['quadrada'])]
if len(cand) > 0:
    idx_ex.append(int(rng.choice(cand.index.values)))
    nomes.append('Exemplo — notumor retangular')
else:
    print('Não há imagem retangular de notumor na validação; ficam só as 4 imagens.')

X_ex = X_val[idx_ex].astype('float32')                    # imagens já preparadas (128x128x1), as mesmas que a rede vê
prob = modelo.predict(X_ex, verbose=0)
pred = prob.argmax(axis=1)

linhas = []
for k, i in enumerate(idx_ex):
    r = val_t.loc[i]
    linhas.append({'Exemplo': nomes[k], 'Classe real': r['classe'], 'Arquivo': r['arquivo'],
                   'Original (larg×alt)': f"{r['largura']}×{r['altura']}",
                   'Previsão do modelo': CLASSES[pred[k]], 'Acertou': 'sim' if CLASSES[pred[k]] == r['classe'] else 'NÃO',
                   **{f'P({c})': prob[k, j] for j, c in enumerate(CLASSES)}})
tab = pd.DataFrame(linhas).set_index('Exemplo')

curto = {'notumor retangular': 'notumor<br>retangular'}
titulos = []
for k in range(len(idx_ex)):
    rotulo = nomes[k].replace('Exemplo — ', '')
    titulos.append(f"<b>{curto.get(rotulo, rotulo + '<br> ')}</b><br>prev.: {CLASSES[pred[k]]}<br>{prob[k, pred[k]]:.0%}")
fig = make_subplots(rows=1, cols=len(idx_ex), subplot_titles=titulos, horizontal_spacing=0.03)
for k, i in enumerate(idx_ex):
    fig.add_trace(go.Heatmap(z=X_val[i, :, :, 0], colorscale='gray', showscale=False,
                             hovertemplate='linha %{y}, coluna %{x}<br>pixel %{z}<extra></extra>'), row=1, col=k + 1)
    fig.update_xaxes(showticklabels=False, row=1, col=k + 1)
    fig.update_yaxes(autorange='reversed', showticklabels=False, scaleanchor=f'x{k+1}' if k else 'x', row=1, col=k + 1)
fig.update_annotations(font_size=11)
fig.update_layout(title='Passo 14 — Imagens de exemplo (validação), como a rede as vê',
                  template='plotly_white', height=400, margin=dict(t=150))
fig.show()

display(tab.round(3))

,Classe real,Arquivo,Original (larg×alt),Previsão do modelo,Acertou,P(glioma),P(meningioma),P(notumor),P(pituitary)
Exemplo,,,,,,,,,
Exemplo — glioma,glioma,Tr-gl_111.jpg,512×512,glioma,sim,0.999,0.000,0.001,0.000
Exemplo — meningioma,meningioma,Tr-me_711.jpg,512×512,glioma,NÃO,0.511,0.478,0.012,0.000
Exemplo — notumor,notumor,Tr-no_552.jpg,630×630,notumor,sim,0.000,0.000,1.000,0.000
Exemplo — pituitary,pituitary,Tr-pi_29.jpg,512×512,pituitary,sim,0.000,0.000,0.000,1.000
Exemplo — notumor retangular,notumor,Tr-no_258.jpg,1375×1446,notumor,sim,0.000,0.072,0.921,0.007


**Observação:** das cinco imagens de exemplo, o modelo acertou quatro. O glioma (Tr-gl_111.jpg, 512×512) foi classificado como glioma com cerca de 100% de confiança, o "sem tumor" quadrado (Tr-no_552.jpg, 630×630) como sem tumor com 100%, a pituitária (Tr-pi_29.jpg, 512×512) como pituitária com 100%, e a imagem retangular de "sem tumor" também foi classificada corretamente, com 92%. O erro foi o meningioma (Tr-me_711.jpg, 512×512), que o modelo classificou como glioma com apenas 51% de confiança, a menor entre as cinco previsões, ou seja, o modelo estava dividido. É um padrão coerente com a pergunta que motivou o projeto, pois não se trata de uma classe confundida com certeza, e sim de uma decisão apertada. Cabem três ressalvas importantes. Primeiro, são cinco imagens escolhidas por sorteio fixo, e não uma medida de desempenho: quem mede o desempenho é a avaliação do Passo 4 (91,13% de acerto na validação). Segundo, a imagem retangular de "sem tumor" foi acertada com 92%, e isso não prova nem refuta o atalho dos fundos pretos descrito no projeto original, que foi medido em 1.600 imagens de teste, não em um exemplo. Terceiro, os tamanhos originais variam (512×512 e 630×630); todas foram padronizadas para 128×128 antes de entrar na rede, e é essa versão reduzida, a que a rede enxerga, que aparece no painel. Essas cinco imagens passam a ser o conjunto fixo de exemplos para os próximos gráficos.

**Observação:** das cinco imagens de exemplo, o modelo acertou quatro. O glioma (Tr-gl_111.jpg, 512×512) foi classificado como glioma com cerca de 100% de confiança, o "sem tumor" quadrado (Tr-no_552.jpg, 630×630) como sem tumor com 100%, a pituitária (Tr-pi_29.jpg, 512×512) como pituitária com 100%, e a imagem retangular de "sem tumor" também foi classificada corretamente, com 92%. O erro foi o meningioma (Tr-me_711.jpg, 512×512), que o modelo classificou como glioma com apenas 51% de confiança, a menor entre as cinco previsões, ou seja, o modelo estava dividido. É um padrão coerente com a pergunta que motivou o projeto, pois não se trata de uma classe confundida com certeza, e sim de uma decisão apertada. Cabem três ressalvas importantes. Primeiro, são cinco imagens escolhidas por sorteio fixo, e não uma medida de desempenho: quem mede o desempenho é a avaliação do Passo 4 (91,13% de acerto na validação). Segundo, a imagem retangular de "sem tumor" foi acertada com 92%, e isso não prova nem refuta o atalho dos fundos pretos descrito no projeto original, que foi medido em 1.600 imagens de teste, não em um exemplo. Terceiro, os tamanhos originais variam (512×512 e 630×630); todas foram padronizadas para 128×128 antes de entrar na rede, e é essa versão reduzida, a que a rede enxerga, que aparece no painel. Essas cinco imagens passam a ser o conjunto fixo de exemplos para os próximos gráficos.

In [ ]:
# Passo 15 — Gráfico 15: mapas de ativação por camada (modelo final, 5 imagens de exemplo)
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1) extrator: o mesmo modelo, mas devolvendo a saída de cada ReLU (os mapas de ativação das 4 convoluções)
x = entrada = keras.Input(shape=modelo.input_shape[1:])
saidas = []
for camada in modelo.layers:
    if isinstance(camada, keras.layers.InputLayer):
        continue
    x = camada(x)
    if isinstance(camada, keras.layers.ReLU):
        saidas.append(x)
extrator = keras.Model(entrada, saidas)
mapas = extrator.predict(X_ex, verbose=0)               # lista de 4 arrays: (n_imagens, altura, largura, filtros)
n_img = X_ex.shape[0]
curtos = [n.replace('Exemplo — ', '') for n in nomes]
grade = [(4, 8), (6, 12), (8, 16), (12, 24)]            # (linhas, colunas) do mosaico de cada camada; sobra de quadrinhos fica em branco

def mosaico(A, grade_L):
    H, W, F = A.shape
    nlin, ncol = grade_L
    M = np.full((nlin * (H + 1), ncol * (W + 1)), np.nan)
    for f in range(F):
        r, c = divmod(f, ncol)
        M[r * (H + 1): r * (H + 1) + H, c * (W + 1): c * (W + 1) + W] = A[:, :, f]
    return M

mos = [[mosaico(mapas[L][k], grade[L]) for k in range(n_img)] for L in range(4)]
zmax = [[max(float(np.nanmax(mos[L][k])), 1e-6) for k in range(n_img)] for L in range(4)]

def titulo(L):
    H, W, F = mapas[L].shape[1:]
    return f'Gráfico 15 — Camada {L+1} (Conv {L+1} + ReLU): {F} mapas de {H}×{W}'

fig = make_subplots(rows=1, cols=2, column_widths=[0.22, 0.78], horizontal_spacing=0.04,
                    subplot_titles=(f'Entrada: {curtos[0]}', 'Mapas de ativação.'))
for k in range(n_img):                                    # traços 0..n-1: imagens de entrada
    fig.add_trace(go.Heatmap(z=X_ex[k, :, :, 0], colorscale='gray', showscale=False, visible=(k == 0),
                             hovertemplate='pixel %{z}<extra></extra>'), row=1, col=1)
for k in range(n_img):                                    # traços n..2n-1: mosaico da camada 1
    fig.add_trace(go.Heatmap(z=mos[0][k], colorscale='Viridis', zmin=0, zmax=zmax[0][k], visible=(k == 0),
                             colorbar=dict(title='Ativação', x=1.02),
                             hovertemplate='ativação %{z:.2f}<extra></extra>'), row=1, col=2)

quadros, passos = [], []
for L in range(4):
    dados = [go.Heatmap(z=mos[L][k], zmin=0, zmax=zmax[L][k]) for k in range(n_img)]
    quadros.append(go.Frame(name=str(L + 1), data=dados, traces=list(range(n_img, 2 * n_img)),
                            layout=go.Layout(title_text=titulo(L), xaxis2=dict(range=[0, mos[L][0].shape[1]]), yaxis2=dict(range=[mos[L][0].shape[0], 0]))))
    passos.append(dict(method='animate', label=f'Camada {L+1}',
                       args=[[str(L + 1)], dict(mode='immediate', frame=dict(duration=0, redraw=True), transition=dict(duration=0))]))
fig.frames = quadros

botoes_img = [dict(label=curtos[k], method='update',
                   args=[{'visible': [j == k for j in range(n_img)] * 2}, {'annotations[0].text': f'Entrada: {curtos[k]}'}])
              for k in range(n_img)]
fig.update_xaxes(showticklabels=False, showgrid=False, zeroline=False)
fig.update_yaxes(autorange='reversed', showticklabels=False, showgrid=False, zeroline=False)
fig.update_yaxes(scaleanchor='x', row=1, col=1)
fig.update_layout(title=dict(text=titulo(0), y=0.97, yanchor='top'), template='plotly_white', height=560, margin=dict(t=170, b=130),
                  sliders=[dict(active=0, steps=passos, currentvalue=dict(prefix='Mostrando: '), pad=dict(t=40))],
                  updatemenus=[dict(type='dropdown', buttons=botoes_img, x=0.0, y=1.22, xanchor='left', yanchor='top'),
                               dict(type='buttons', direction='left', x=0.25, y=1.22, xanchor='left', yanchor='top',
                                    buttons=[dict(label='▶ Percorrer as 4 camadas', method='animate',
                                                  args=[None, dict(frame=dict(duration=1500, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
                                             dict(label='⏸ Pausar', method='animate',
                                                  args=[[None], dict(mode='immediate', frame=dict(duration=0, redraw=False), transition=dict(duration=0))])])])
fig.show()

# 2) números por camada e imagem
rot = [f'Camada {L+1} ({mapas[L].shape[3]} filtros, {mapas[L].shape[1]}×{mapas[L].shape[2]})' for L in range(4)]
lig = pd.DataFrame([[100 * float((mapas[L][k] > 0).mean()) for k in range(n_img)] for L in range(4)], index=rot, columns=curtos)
mx = pd.DataFrame([[float(mapas[L][k].max()) for k in range(n_img)] for L in range(4)], index=rot, columns=curtos)
print('% das ativações maiores que zero (neurônio ligado):')
display(lig.round(1))
print('\nMaior ativação encontrada:')
display(mx.round(1))

**Observação:** na Camada 1, o gráfico mostra que os mapas ainda guardam a forma da imagem de entrada: vários quadrinhos reproduzem o contorno da cabeça e do cérebro do glioma, uns com forte intensidade e outros bem apagados, e a diferença entre eles é o que cada filtro de 3×3 pixels destaca. As tabelas mostram como isso muda ao longo da rede. A porcentagem de ativações maiores que zero cai de 42,1% a 47,1% na Camada 1 para 27,1% a 32,8% na Camada 2 e chega ao mínimo na Camada 3, com apenas 6,3% a 10,2% (a maior parte dos neurônios dessa camada fica desligada em qualquer dos cinco exames), voltando a subir na Camada 4, para 11,7% a 20,8%. Ao mesmo tempo, a maior ativação cresce com a profundidade: de 3,3 a 4,5 na Camada 1, 5,1 a 7,0 na Camada 2 e 6,0 a 11,4 na Camada 3, até 9,5 a 29,1 na Camada 4. Em outras palavras, a rede vai concentrando a resposta em poucos neurônios, que respondem com mais força. O caso que mais se destaca é o do exame de "sem tumor" quadrado, que tem a maior porcentagem de ativações na Camada 4 (20,8%) e uma ativação máxima de 29,1, de 2 a 3 vezes maior que a dos outros quatro exemplos (de 9,5 a 13,4). Na Camada 1, os dois exemplos de "sem tumor" também têm as maiores ativações máximas (4,5 e 4,2, contra 3,3 a 3,6 dos demais). Isso mostra que a rede reage de forma diferente a esse exame, mas não explica por quê: o exame de "sem tumor" quadrado tem aparência visivelmente diferente das outras imagens (maior contraste), e a diferença pode vir da intensidade ou do tipo de imagem, e não da classe. A comparação usa apenas cinco imagens e não deve ser lida como uma regra sobre as classes.

### Passo 16 — Gráfico 16: a superfície de um mapa de ativação

O Gráfico 16 do catálogo transforma um mapa de ativação em uma superfície tridimensional: em vez de ver o mapa como uma imagem em cores, cada posição vira uma altura, e os picos mostram onde o filtro respondeu com mais força. Aqui ele é feito com os mapas reais do Passo 15. Como cada camada tem dezenas ou centenas de filtros, escolhemos, para cada imagem e cada camada, o filtro de maior atividade, isto é, o que tem a maior soma de ativações no mapa. À esquerda aparece a imagem de entrada e à direita a superfície do mapa desse filtro, que você pode girar com o mouse e dar zoom com a roda. O menu no alto escolhe a imagem entre as cinco do Passo 14, a barra deslizante troca a camada, e o botão "Percorrer as 4 camadas" faz a animação, mostrando como a superfície vai ficando menor (128×128, 64×64, 32×32 e 16×16 posições) e mais concentrada em poucos picos. A altura usa a mesma escala para as cinco imagens dentro de cada camada, de modo que as superfícies sejam comparáveis. Abaixo do gráfico, uma tabela diz qual filtro foi escolhido em cada caso, a maior ativação dele e a porcentagem de posições em que ele ficou ligado. Um aviso de leitura: um pico indica onde o filtro respondeu, e o gráfico por si só não diz o que ele reconheceu naquele ponto.

In [ ]:
# Passo 16 — Gráfico 16: superfície 3D do mapa de ativação do filtro mais ativo (modelo final)
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# para cada camada e cada imagem: o filtro de maior atividade (maior soma das ativações no mapa)
n_img = X_ex.shape[0]
curtos = [n.replace('Exemplo — ', '') for n in nomes]
melhor = [[int(np.argmax(mapas[L][k].sum(axis=(0, 1)))) for k in range(n_img)] for L in range(4)]
sup = [[mapas[L][k][:, :, melhor[L][k]] for k in range(n_img)] for L in range(4)]       # mapa (altura x largura) do filtro escolhido
zmax = [max(float(sup[L][k].max()) for k in range(n_img)) for L in range(4)]            # escala comum às 5 imagens, por camada

def titulo(L):
    H, W = sup[L][0].shape
    return f'Gráfico 16 — Camada {L+1}: filtro mais ativo de cada imagem ({H}×{W})'

fig = make_subplots(rows=1, cols=2, column_widths=[0.25, 0.75], specs=[[{'type': 'xy'}, {'type': 'scene'}]],
                    horizontal_spacing=0.02, subplot_titles=(curtos[0], 'Superfície do mapa'))
for k in range(n_img):                                                                  # traços 0..n-1: imagens de entrada
    fig.add_trace(go.Heatmap(z=X_ex[k, :, :, 0], colorscale='gray', showscale=False, visible=(k == 0),
                             hovertemplate='pixel %{z}<extra></extra>'), row=1, col=1)
for k in range(n_img):                                                                  # traços n..2n-1: superfícies da camada 1
    fig.add_trace(go.Surface(z=sup[0][k], colorscale='Viridis', cmin=0, cmax=zmax[0], visible=(k == 0),
                             colorbar=dict(title='Ativação', x=1.0, len=0.7),
                             hovertemplate='linha %{y}, coluna %{x}<br>ativação %{z:.2f}<extra></extra>'), row=1, col=2)

quadros, passos = [], []
for L in range(4):
    dados = [go.Surface(z=sup[L][k], cmin=0, cmax=zmax[L]) for k in range(n_img)]
    quadros.append(go.Frame(name=str(L + 1), data=dados, traces=list(range(n_img, 2 * n_img)),
                            layout=go.Layout(title_text=titulo(L), scene=dict(zaxis=dict(range=[0, zmax[L]])))))
    passos.append(dict(method='animate', label=f'Camada {L+1}',
                       args=[[str(L + 1)], dict(mode='immediate', frame=dict(duration=0, redraw=True), transition=dict(duration=0))]))
fig.frames = quadros

botoes_img = [dict(label=curtos[k], method='update',
                   args=[{'visible': [j == k for j in range(n_img)] * 2}, {'annotations[0].text': curtos[k]}])
              for k in range(n_img)]
fig.update_xaxes(showticklabels=False, showgrid=False, zeroline=False, row=1, col=1)
fig.update_yaxes(autorange='reversed', showticklabels=False, showgrid=False, zeroline=False, scaleanchor='x', row=1, col=1)
fig.update_layout(title=dict(text=titulo(0), y=0.97, yanchor='top'), template='plotly_white', height=640, margin=dict(t=170, b=130),
                  scene=dict(xaxis_title='Coluna', yaxis=dict(title='Linha', autorange='reversed'),
                             zaxis=dict(title='Ativação', range=[0, zmax[0]]),
                             camera=dict(eye=dict(x=1.5, y=-1.8, z=1.1)), aspectmode='manual', aspectratio=dict(x=1, y=1, z=0.6)),
                  sliders=[dict(active=0, steps=passos, currentvalue=dict(prefix='Mostrando: '), pad=dict(t=40))],
                  updatemenus=[dict(type='dropdown', buttons=botoes_img, x=0.0, y=1.22, xanchor='left', yanchor='top'),
                               dict(type='buttons', direction='left', x=0.25, y=1.22, xanchor='left', yanchor='top',
                                    buttons=[dict(label='▶ Percorrer as 4 camadas', method='animate',
                                                  args=[None, dict(frame=dict(duration=2000, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
                                             dict(label='⏸ Pausar', method='animate',
                                                  args=[[None], dict(mode='immediate', frame=dict(duration=0, redraw=False), transition=dict(duration=0))])])])
fig.show()

# qual filtro foi escolhido em cada caso, e como é o mapa dele
rot = [f'Camada {L+1} ({mapas[L].shape[3]} filtros)' for L in range(4)]
tab = pd.DataFrame([[f'filtro {melhor[L][k] + 1} · máx {sup[L][k].max():.1f} · {100 * (sup[L][k] > 0).mean():.0f}% ligado'
                     for k in range(n_img)] for L in range(4)], index=rot, columns=curtos)
display(tab)

,glioma,meningioma,notumor,pituitary,notumor retangular
Camada 1 (32 filtros),filtro 4 · máx 3.3 · 100% ligado,filtro 4 · máx 3.6 · 100% ligado,filtro 4 · máx 4.5 · 100% ligado,filtro 4 · máx 3.3 · 100% ligado,filtro 4 · máx 4.2 · 100% ligado
Camada 2 (64 filtros),filtro 35 · máx 1.9 · 57% ligado,filtro 35 · máx 2.3 · 86% ligado,filtro 5 · máx 4.0 · 95% ligado,filtro 35 · máx 2.4 · 95% ligado,filtro 35 · máx 3.6 · 69% ligado
Camada 3 (128 filtros),filtro 45 · máx 2.9 · 28% ligado,filtro 45 · máx 4.2 · 35% ligado,filtro 103 · máx 9.1 · 16% ligado,filtro 45 · máx 3.7 · 40% ligado,filtro 45 · máx 4.4 · 37% ligado
Camada 4 (256 filtros),filtro 97 · máx 5.4 · 38% ligado,filtro 253 · máx 6.7 · 36% ligado,filtro 128 · máx 27.4 · 58% ligado,filtro 57 · máx 5.9 · 40% ligado,filtro 200 · máx 8.4 · 22% ligado


**Observação:** na Camada 1, a superfície do glioma mostra um relevo irregular concentrado na região central, que corresponde à posição da cabeça na imagem, com a base praticamente plana nas bordas, onde a imagem é preta. A tabela revela que o filtro mais ativo da Camada 1 é o mesmo, o filtro 4, nas cinco imagens, e que ele fica ligado em 100% das posições: um filtro assim praticamente segue o brilho da imagem (no Passo 13, o filtro 4 apareceu com os dois pesos de maior valor positivos, e 100% da grade ligada), e por isso seu mapa se parece com uma cópia suavizada do exame. A partir daí os caminhos se separam. Na Camada 2, o filtro 35 é o mais ativo em quatro das cinco imagens, mas no "sem tumor" quadrado é o filtro 5; na Camada 3, o filtro 45 domina em quatro imagens e o filtro 103 no "sem tumor" quadrado; na Camada 4, cada imagem tem um filtro diferente (97, 253, 128, 57 e 200). A porcentagem de posições ligadas do filtro escolhido cai com a profundidade (100% na Camada 1, de 57% a 95% na Camada 2 e de 16% a 40% na Camada 3) e fica entre 22% e 58% na Camada 4, o que

### Passo 17 — Gráfico 17: neurônios ativos e inativos

O Gráfico 17 do catálogo mostra quantos neurônios (aqui, filtros) da rede ficam ligados e quantos ficam desligados. Até agora usamos cinco exemplos; neste passo usamos todas as 958 imagens do conjunto de validação, o que dá uma visão do comportamento geral da rede, e não de casos escolhidos. Cada imagem passa pelo extrator do Passo 15 e, para cada filtro de cada camada, o código mede duas coisas: em que fração das posições do mapa o filtro fica ligado (ativação maior que zero), e em quantas imagens ele liga em pelo menos uma posição. O gráfico da esquerda mostra, para a camada escolhida, uma barra por filtro com a porcentagem média de posições ligadas, ordenadas do filtro mais ligado ao menos ligado. As barras vermelhas são filtros que nunca ligaram em nenhuma imagem do conjunto escolhido, os chamados filtros mortos. O gráfico da direita é um histograma que conta quantos filtros ligam em poucas ou em muitas imagens. A barra deslizante e o botão "Percorrer as 4 camadas" animam a passagem de uma camada para a outra, e o menu no alto permite olhar todas as classes juntas ou cada classe separadamente (glioma, meningioma, sem tumor e pituitária); a ordem dos filtros é a mesma em todas as opções, o que permite comparar uma classe com outra. Abaixo do gráfico, três tabelas resumem: o balanço por camada, a média de posições ligadas por classe e quantos filtros nunca ligam em cada classe. Um aviso de leitura: um filtro morto em validação não é necessariamente inútil em outras imagens, e um filtro muito ligado não é necessariamente importante; o gráfico descreve a atividade, não a utilidade de cada filtro.

In [ ]:
# Passo 17 — Gráfico 17: filtros ativos e inativos nas imagens de validação (modelo final)
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1) passa TODAS as imagens de validação pelo extrator (em lotes) e acumula, por filtro:
#    - em quantas imagens ele liga em alguma posição (ativação > 0)
#    - que fração das posições fica ligada, em média
cls_idx = np.array([CLASSES.index(c) for c in val['classe']])
nomes_conj = ['todas as classes'] + list(CLASSES)
n_cls = np.array([(cls_idx == c).sum() for c in range(4)])
nF = [32, 64, 128, 256]
soma_pos = [np.zeros((4, F)) for F in nF]
cont_img = [np.zeros((4, F)) for F in nF]

print(f'Processando {len(X_val)} imagens de validação...')
for i in range(0, len(X_val), 64):
    lote = X_val[i:i + 64].astype('float32')
    cl = cls_idx[i:i + 64]
    for L, A in enumerate(extrator.predict(lote, verbose=0)):
        lig = A > 0
        frac = lig.mean(axis=(1, 2))                    # (imagens, filtros): fração de posições ligadas
        algum = lig.any(axis=(1, 2))                    # (imagens, filtros): ligou em alguma posição?
        for c in range(4):
            m = cl == c
            if m.any():
                soma_pos[L][c] += frac[m].sum(axis=0)
                cont_img[L][c] += algum[m].sum(axis=0)

def estat(L):
    """[conjunto][0=média de posições ligadas (%), 1=% das imagens em que o filtro liga]; conjunto 0 = todas as classes"""
    pos = np.vstack([100 * soma_pos[L].sum(0) / n_cls.sum()] + [100 * soma_pos[L][c] / n_cls[c] for c in range(4)])
    img = np.vstack([100 * cont_img[L].sum(0) / n_cls.sum()] + [100 * cont_img[L][c] / n_cls[c] for c in range(4)])
    return pos, img

E = [estat(L) for L in range(4)]
ordem = [np.argsort(-E[L][0][0]) for L in range(4)]       # mesma ordem de filtros para todos os conjuntos (do mais ao menos ligado)

def barras(L, s):
    pos, img = E[L]
    o = ordem[L]
    cores = ['#C0392B' if img[s][f] == 0 else '#C9A227' for f in o]
    return dict(x=np.arange(len(o)), y=pos[s][o], customdata=np.stack([o + 1, img[s][o]], axis=1), marker=dict(color=cores))

def titulo(L):
    return f'Gráfico 17 — Camada {L+1}: {nF[L]} filtros nas {len(X_val)} imagens de validação'

fig = make_subplots(rows=1, cols=2, column_widths=[0.62, 0.38], horizontal_spacing=0.1,
                    subplot_titles=(nomes_conj[0], 'Filtros por % de imagens'))
for s in range(5):
    fig.add_trace(go.Bar(**barras(0, s), visible=(s == 0), showlegend=False,
                         hovertemplate='filtro %{customdata[0]}<br>%{y:.1f}% das posições ligadas<br>liga em %{customdata[1]:.1f}% das imagens<extra></extra>'), row=1, col=1)
for s in range(5):
    fig.add_trace(go.Histogram(x=np.minimum(E[0][1][s], 99.99), xbins=dict(start=0, end=100, size=5), marker=dict(color='#1F4E79'), visible=(s == 0),
                               showlegend=False, hovertemplate='%{x}% das imagens: %{y} filtros<extra></extra>'), row=1, col=2)

quadros, passos = [], []
for L in range(4):
    dados = [go.Bar(**barras(L, s)) for s in range(5)] + [go.Histogram(x=np.minimum(E[L][1][s], 99.99)) for s in range(5)]
    quadros.append(go.Frame(name=str(L + 1), data=dados, traces=list(range(10)),
                            layout=go.Layout(title_text=titulo(L), xaxis=dict(range=[-1, nF[L]]), yaxis2=dict(autorange=True))))
    passos.append(dict(method='animate', label=f'Camada {L+1}',
                       args=[[str(L + 1)], dict(mode='immediate', frame=dict(duration=0, redraw=True), transition=dict(duration=0))]))
fig.frames = quadros

botoes = [dict(label=nomes_conj[s], method='update',
               args=[{'visible': [j == s for j in range(5)] * 2}, {'annotations[0].text': nomes_conj[s]}])
          for s in range(5)]
fig.update_annotations(font_size=13)
fig.update_xaxes(title_text='Filtros (do mais ao menos ligado)', showticklabels=False, range=[-1, nF[0]], row=1, col=1)
fig.update_yaxes(title_text='% das posições ligadas', range=[0, 100], row=1, col=1)
fig.update_xaxes(title_text='% das imagens em que o filtro liga', range=[0, 100], row=1, col=2)
fig.update_yaxes(title_text='Número de filtros', row=1, col=2)
fig.update_layout(title=dict(text=titulo(0), y=0.97, yanchor='top'), template='plotly_white', height=560, margin=dict(t=170, b=130), bargap=0.1,
                  sliders=[dict(active=0, steps=passos, currentvalue=dict(prefix='Mostrando: '), pad=dict(t=40))],
                  updatemenus=[dict(type='dropdown', buttons=botoes, x=0.0, y=1.22, xanchor='left', yanchor='top'),
                               dict(type='buttons', direction='left', x=0.3, y=1.22, xanchor='left', yanchor='top',
                                    buttons=[dict(label='▶ Percorrer as 4 camadas', method='animate',
                                                  args=[None, dict(frame=dict(duration=1800, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
                                             dict(label='⏸ Pausar', method='animate',
                                                  args=[[None], dict(mode='immediate', frame=dict(duration=0, redraw=False), transition=dict(duration=0))])])])
fig.show()

# 2) resumo por camada (todas as classes) e por classe
lin = []
for L in range(4):
    pos, img = E[L]
    lin.append({'Camada': f'Camada {L+1}', 'Filtros': nF[L], 'Nunca ligam': int((img[0] == 0).sum()),
                'Ligam em <10% das imagens': int((img[0] < 10).sum()), 'Ligam em >90% das imagens': int((img[0] > 90).sum()),
                'Posições ligadas (média, %)': pos[0].mean()})
display(pd.DataFrame(lin).set_index('Camada').round(1))

por_classe = pd.DataFrame({nomes_conj[s]: [E[L][0][s].mean() for L in range(4)] for s in range(1, 5)},
                          index=[f'Camada {L+1}' for L in range(4)])
print('\nPosições ligadas (média por filtro, %), por classe:')
display(por_classe.round(1))
mortos_c = pd.DataFrame({nomes_conj[s]: [int((E[L][1][s] == 0).sum()) for L in range(4)] for s in range(1, 5)},
                        index=[f'Camada {L+1}' for L in range(4)])
print('\nFiltros que nunca ligam, por classe:')
display(mortos_c)

Processando 958 imagens de validação...


,Filtros,Nunca ligam,Ligam em <10% das imagens,Ligam em >90% das imagens,"Posições ligadas (média, %)"
Camada,,,,,
Camada 1,32,1,1,31,43.6
Camada 2,64,0,0,64,31.3
Camada 3,128,3,4,117,9.1
Camada 4,256,0,1,254,13.8



Posições ligadas (média por filtro, %), por classe:


,glioma,meningioma,notumor,pituitary
Camada 1,42.1,43.1,44.2,45.2
Camada 2,33.1,31.8,31.1,29.0
Camada 3,9.9,10.2,8.6,7.4
Camada 4,13.6,12.7,16.4,13.6



Filtros que nunca ligam, por classe:


,glioma,meningioma,notumor,pituitary
Camada 1,1,1,1,1
Camada 2,0,0,0,0
Camada 3,4,3,3,3
Camada 4,1,0,0,0


**Observação:** o balanço sobre as 958 imagens de validação mostra que a rede quase não tem filtros mortos. Dos 32 filtros da Camada 1, um nunca liga em nenhuma imagem e 31 ligam em mais de 90% delas; na Camada 2, todos os 64 ligam em mais de 90% das imagens; na Camada 3, são 3 filtros que nunca ligam (um quarto liga em menos de 10% das imagens) e 117 dos 128 ligam em mais de 90%; na Camada 4, nenhum dos 256 fica sempre desligado, 1 liga em menos de 10% e 254 ligam em mais de 90%. Esse critério é generoso, porque conta um filtro como ligado se ele responde em pelo menos uma posição da imagem, e é aí que a segunda medida ajuda: a porcentagem média de posições ligadas por filtro cai de 43,6% na Camada 1 para 31,3% na Camada 2 e chega a 9,1% na Camada 3, subindo um pouco, para 13,8%, na Camada 4. Ou seja, os filtros quase sempre estão "vivos", mas respondem em uma parte pequena de cada mapa, principalmente nas camadas profundas, o que combina com a ideia de que os filtros mais profundos reagem a padrões mais específicos. O gráfico da Camada 1 mostra a mesma coisa em forma de curva: as barras caem de 100% das posições, no filtro mais ligado, até perto de zero, e o histograma tem quase todos os filtros na faixa de 95% a 100% das imagens, com um filtro isolado perto de zero. Esses valores médios por camada são parecidos com os dos cinco exemplos do Passo 15 (de 42,1% a 47,1% na Camada 1, de 27,1% a 32,8% na Camada 2, de 6,3% a 10,2% na Camada 3 e de 11,7% a 20,8% na Camada 4), o que indica que os

## Passo 18 — Gráfico 01: a arquitetura da rede em 3D

Até aqui olhamos **dentro** da rede: pesos, ativações, filtros. Agora vamos ver a rede **inteira, de fora**, como um objeto 3D. Cada bloco é um estágio do nosso modelo real (`modelo`), e o código **lê a arquitetura do próprio modelo**: passa um tensor de zeros por cada camada e anota o formato que sai dela.

**Como ler o gráfico**
- **Altura e largura** do bloco = tamanho espacial da imagem naquele estágio (128 → 64 → 32 → 16 → 8), em **escala linear**.
- **Espessura** do bloco = número de canais (1 → 32 → 64 → 128 → 256). Aqui a escala é **comprimida** (2·√canais), senão a Conv 4 ficaria 256 vezes mais grossa que a entrada e o gráfico viraria um paredão.
- Os blocos finais (Média global, Dropout, Dense) são **vetores**, desenhados como colunas finas. A altura deles também é comprimida (8·√n).
- **Cores:** cinza = entrada, azul = convolução + ReLU, dourado = MaxPooling, verde = média global, cinza-escuro = Dropout, vermelho = Dense.
- Passe o mouse sobre um bloco para ver o nome, a camada Keras, o formato real, quantos valores ele guarda e quantos parâmetros tem.
- O botão **▶ Girar a rede** faz a câmera dar uma volta completa em torno do modelo. Você também pode arrastar com o mouse para girar à mão.

Abaixo do gráfico, uma tabela confere os números reais, e o total de parâmetros tem que fechar em **388.868**.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# 1) lê a arquitetura real do modelo: passa um tensor de zeros e anota o formato depois de cada camada
x = np.zeros((1,) + tuple(modelo.input_shape[1:]), dtype='float32')
estagios = [dict(nome='Entrada', tipo='entrada', camada='InputLayer', forma=tuple(x.shape[1:]), params=0)]
n_conv = 0
for camada in modelo.layers:
    if isinstance(camada, keras.layers.InputLayer):
        continue
    x = np.array(camada(x))
    forma = tuple(x.shape[1:])
    if isinstance(camada, keras.layers.Rescaling):
        estagios[0]['nome'] = 'Entrada + Rescaling (÷255)'
        estagios[0]['camada'] += ' + Rescaling'
    elif isinstance(camada, keras.layers.Conv2D):
        n_conv += 1
        estagios.append(dict(nome=f'Conv {n_conv}', tipo='conv', camada='Conv2D', forma=forma, params=camada.count_params()))
    elif isinstance(camada, keras.layers.ReLU):
        estagios[-1]['nome'] += ' + ReLU'
        estagios[-1]['camada'] += ' + ReLU'
    elif isinstance(camada, keras.layers.MaxPooling2D):
        estagios.append(dict(nome='MaxPool 2×2', tipo='pool', camada='MaxPooling2D', forma=forma, params=0))
    elif isinstance(camada, keras.layers.GlobalAveragePooling2D):
        estagios.append(dict(nome='Média global', tipo='gap', camada='GlobalAveragePooling2D', forma=forma, params=0))
    elif isinstance(camada, keras.layers.Dropout):
        estagios.append(dict(nome='Dropout 0,3', tipo='drop', camada='Dropout (inativo na previsão)', forma=forma, params=0))
    elif isinstance(camada, keras.layers.Dense):
        estagios.append(dict(nome='Dense (softmax)', tipo='dense', camada='Dense', forma=forma, params=camada.count_params()))

cor = {'entrada': '#9AA5B1', 'conv': '#1F4E79', 'pool': '#C9A227', 'gap': '#2E8B57', 'drop': '#7F8C8D', 'dense': '#C0392B'}
nome_tipo = {'entrada': 'Entrada', 'conv': 'Convolução + ReLU', 'pool': 'MaxPooling', 'gap': 'Média global', 'drop': 'Dropout', 'dense': 'Dense'}

# 2) geometria: altura e largura do bloco = tamanho espacial (escala linear); espessura = 2·√canais (escala comprimida)
def dimensoes(e):
    f = e['forma']
    if len(f) == 3:
        h, w, c = f
        return 2 * np.sqrt(c), float(h), float(w)
    return 3.0, 8.0 * np.sqrt(float(f[0])), 6.0           # vetores: coluna fina, altura comprimida (8·√n)

def caixa(x0, x1, y0, y1, z0, z1):
    X = [x0, x1, x1, x0, x0, x1, x1, x0]; Y = [y0, y0, y1, y1, y0, y0, y1, y1]; Z = [z0, z0, z0, z0, z1, z1, z1, z1]
    I = [0, 0, 4, 4, 0, 0, 1, 1, 2, 2, 3, 3]; J = [1, 2, 5, 6, 1, 5, 2, 6, 3, 7, 0, 4]; K = [2, 3, 6, 7, 5, 4, 6, 5, 7, 6, 4, 7]
    return X, Y, Z, I, J, K

fig = go.Figure()
xpos, folga = 0.0, 20.0
vistos = set()
bx, by, bz = [], [], []
rotulos_x, rotulos_z, rotulos_t = [], [], []
linhas = []
total_params = sum(e['params'] for e in estagios)
for e in estagios:
    esp, alt, lar = dimensoes(e)
    X, Y, Z, I, J, K = caixa(xpos, xpos + esp, -lar / 2, lar / 2, -alt / 2, alt / 2)   # y = largura, z = altura (vertical)
    f = e['forma']
    dim = '×'.join(str(v) for v in f)
    texto = (f"<b>{e['nome']}</b><br>Camada Keras: {e['camada']}<br>Formato: {dim}<br>Valores: {int(np.prod(f)):,}<br>Parâmetros: {e['params']:,}").replace(',', '.')
    mostrar = e['tipo'] not in vistos
    vistos.add(e['tipo'])
    fig.add_trace(go.Mesh3d(x=X, y=Y, z=Z, i=I, j=J, k=K, color=cor[e['tipo']], opacity=0.92, flatshading=True,
                            name=nome_tipo[e['tipo']], legendgroup=e['tipo'], showlegend=mostrar, hoverinfo='text', text=texto))
    for a, b in [(0,1),(1,2),(2,3),(3,0),(4,5),(5,6),(6,7),(7,4),(0,4),(1,5),(2,6),(3,7)]:
        bx += [X[a], X[b], None]; by += [Y[a], Y[b], None]; bz += [Z[a], Z[b], None]
    rotulos_x.append(xpos + esp / 2); rotulos_z.append(alt / 2 + (12, 34, 56)[len(rotulos_x) % 3]); rotulos_t.append(e['nome'].replace(' + ReLU', '').replace(' (softmax)', '').replace(' + Rescaling (÷255)', '') + f'<br>{dim}')
    linhas.append({'Estágio': e['nome'], 'Camada Keras': e['camada'], 'Formato (alt×larg×canais)': dim,
                   'Valores no tensor': int(np.prod(f)), 'Parâmetros': e['params']})
    xpos += esp + folga

fig.add_trace(go.Scatter3d(x=bx, y=by, z=bz, mode='lines', line=dict(color='#1B2A3A', width=2), showlegend=False, hoverinfo='skip'))
fig.add_trace(go.Scatter3d(x=rotulos_x, y=[0] * len(rotulos_x), z=rotulos_z, mode='text', text=rotulos_t, textfont=dict(size=9, color='#1B2A3A'),
                           showlegend=False, hoverinfo='skip'))

# 3) animação: câmera girando em volta da rede
angulos = np.linspace(0, 2 * np.pi, 73)[:-1]
quadros = [go.Frame(name=str(i), layout=go.Layout(scene_camera=dict(eye=dict(x=2.0 * np.cos(a - np.pi / 2), y=2.0 * np.sin(a - np.pi / 2), z=0.55)))) for i, a in enumerate(angulos)]
fig.frames = quadros
fig.update_layout(title=dict(text=f'Gráfico 01 — Arquitetura da rede (modelo final): {total_params:,} parâmetros'.replace(',', '.'), y=0.97, yanchor='top'),
                  template='plotly_white', height=600, margin=dict(t=90, b=30),
                  legend=dict(orientation='h', y=-0.02),
                  scene=dict(aspectmode='data', xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False),
                             camera=dict(eye=dict(x=0.0, y=-2.0, z=0.55), center=dict(x=0, y=0, z=0))),
                  updatemenus=[dict(type='buttons', direction='left', x=0.0, y=1.08, xanchor='left', yanchor='top',
                                    buttons=[dict(label='▶ Girar a rede', method='animate',
                                                  args=[None, dict(frame=dict(duration=60, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
                                             dict(label='⏸ Parar', method='animate',
                                                  args=[[None], dict(mode='immediate', frame=dict(duration=0, redraw=False), transition=dict(duration=0))])])])
fig.show()

tab = pd.DataFrame(linhas).set_index('Estágio')
display(tab)
print(f'\nTotal de parâmetros somados nesta tabela: {total_params:,}'.replace(',', '.'))

,Camada Keras,Formato (alt×larg×canais),Valores no tensor,Parâmetros
Estágio,,,,
Entrada + Rescaling (÷255),InputLayer + Rescaling,128×128×1,16384,0
Conv 1 + ReLU,Conv2D + ReLU,128×128×32,524288,320
MaxPool 2×2,MaxPooling2D,64×64×32,131072,0
Conv 2 + ReLU,Conv2D + ReLU,64×64×64,262144,18496
MaxPool 2×2,MaxPooling2D,32×32×64,65536,0
Conv 3 + ReLU,Conv2D + ReLU,32×32×128,131072,73856
MaxPool 2×2,MaxPooling2D,16×16×128,32768,0
Conv 4 + ReLU,Conv2D + ReLU,16×16×256,65536,295168
MaxPool 2×2,MaxPooling2D,8×8×256,16384,0



Total de parâmetros somados nesta tabela: 388.868


**Observação:** o gráfico mostra a rede como um funil. A altura e a largura dos blocos caem pela metade a cada MaxPooling (128 → 64 → 32 → 16 → 8), e a espessura cresce a cada convolução (1 → 32 → 64 → 128 → 256 canais). Depois da Conv 4, a Média global colapsa tudo em um vetor de 256 números, que passa pelo Dropout e chega à Dense, com 4 saídas (uma por classe).

A tabela confere o desenho e traz dois fatos que ele sozinho não deixa ver.

1. **Onde estão os valores e onde estão os parâmetros são lugares diferentes.** O maior tensor da rede é a saída da Conv 1, com 524.288 valores (128×128×32), mas essa camada tem só 320 parâmetros (288 pesos + 32 vieses). A Conv 4 faz o oposto: seu tensor tem 65.536 valores e ela guarda 295.168 parâmetros. Isso acontece porque os parâmetros dependem do número de canais de entrada e de saída (3×3×canais_entrada×canais_saída), e não do tamanho da imagem.

2. **A rede é "pesada" no fim.** Conv 4 = 295.168 (cerca de 76% do total), Conv 3 = 73.856 (19%), Conv 2 = 18.496 (5%), Dense = 1.028 (0,3%) e Conv 1 = 320 (0,08%). Quase todo o conhecimento do modelo está nas duas últimas convoluções. MaxPooling, Média global e Dropout têm zero parâmetros, pois só reorganizam ou resumem os valores.

O total da tabela fecha em **388.868 parâmetros**, o mesmo número do modelo treinado. Isso mostra que o gráfico foi construído a partir da rede real e não deixou nenhuma camada de fora.

Um detalhe do Dropout: ele mantém os mesmos 256 valores da Média global, e a tabela o marca como "inativo na previsão". Ele só desliga neurônios durante o treino, para reduzir o sobreajuste.

*Escala do desenho:* a altura e a largura dos blocos são lineares (reais), mas a espessura e a altura dos vetores são comprimidas por raiz quadrada. Por isso a Conv 4 parece só um pouco mais grossa que a Conv 2, e não 4 vezes. Os números reais estão na tabela e no hover de cada bloco.

## Passo 19 — Gráfico 02: do exame ao diagnóstico (fluxograma)

No Gráfico 01 vimos a rede por fora, como um objeto. Agora vamos seguir **o caminho de uma imagem** dentro dela, etapa por etapa, até virar um diagnóstico. É um fluxograma de 9 caixas: Imagem → Bloco 1 → Bloco 2 → Bloco 3 → Bloco 4 → Média global → Dropout → Dense + softmax → Diagnóstico. Cada "Bloco" agrupa uma convolução 3×3, o ReLU e o MaxPooling 2×2. As caixas e os formatos são lidos do modelo real (`modelo`).

**O que há de real em cada caixa**
- Na caixa **Imagem**, a miniatura é a imagem de validação que a rede de fato recebe (já em 128×128).
- Nos **Blocos 1 a 4**, "máx." é a maior ativação que o bloco produziu para aquela imagem, e "ligadas" é a porcentagem de posições com ativação maior que zero (os mapas do Passo 15).
- Em **Diagnóstico**, estão as 4 probabilidades que a rede calculou, a classe escolhida (▶) e a classe real.

**Como usar**
- A lista no canto esquerdo troca o exemplo (as mesmas 5 imagens do Passo 14).
- **▶ Percorrer a rede** acende as caixas em sequência (dourado = etapa atual, azul claro = já percorridas). Você também pode arrastar o controle deslizante.
- Passe o mouse sobre uma caixa para ver o que ela faz e quantos parâmetros tem.

A tabela abaixo do gráfico mostra, para cada exemplo, a classe real, a previsão, a confiança e a ativação máxima em cada bloco.

In [ ]:
import numpy as np
import pandas as pd
import io, base64
from PIL import Image
import plotly.graph_objects as go

# 1) lê a arquitetura real do modelo e agrupa em "caixas" do fluxograma
x = np.zeros((1,) + tuple(modelo.input_shape[1:]), dtype='float32')
caixas = [dict(nome='Imagem', tipo='entrada', linhas=[f'{"×".join(str(v) for v in x.shape[1:])}'], params=0,
               explica='A imagem de ressonância, já em 128×128 e 1 canal (tons de cinza). O Rescaling divide cada pixel por 255 para ficar entre 0 e 1.')]
bloco = None
for camada in modelo.layers:
    if isinstance(camada, keras.layers.InputLayer):
        continue
    x = np.array(camada(x))
    forma = '×'.join(str(v) for v in x.shape[1:])
    if isinstance(camada, keras.layers.Rescaling):
        caixas[0]['linhas'].append('÷255 (Rescaling)')
    elif isinstance(camada, keras.layers.Conv2D):
        bloco = dict(nome=f'Bloco {sum(c["tipo"] == "bloco" for c in caixas) + 1}', tipo='bloco', params=camada.count_params(),
                     linhas=[f'Conv 3×3 ×{camada.filters}', '+ ReLU', 'MaxPool 2×2'],
                     explica=f'{camada.filters} filtros 3×3 varrem a imagem, o ReLU zera os valores negativos e o MaxPooling guarda o maior valor de cada quadrado 2×2.')
        caixas.append(bloco)
    elif isinstance(camada, keras.layers.MaxPooling2D):
        bloco['linhas'].append(f'saída {forma}')
    elif isinstance(camada, keras.layers.GlobalAveragePooling2D):
        caixas.append(dict(nome='Média global', tipo='gap', linhas=['média por mapa', f'saída {forma}'], params=0,
                           explica='Cada um dos 256 mapas vira um único número: a média de todas as suas posições.'))
    elif isinstance(camada, keras.layers.Dropout):
        caixas.append(dict(nome='Dropout', tipo='drop', linhas=[f'{camada.rate}'.replace('.', ',') + ' · só treino', f'saída {forma}'], params=0,
                           explica='Durante o treino desliga neurônios ao acaso para evitar decorar. Na previsão não faz nada.'))
    elif isinstance(camada, keras.layers.Dense):
        caixas.append(dict(nome='Dense + softmax', tipo='dense', linhas=[f'{camada.units} neurônios', 'softmax'], params=camada.count_params(),
                           explica='Combina os 256 números em 4 pontuações e o softmax as transforma em probabilidades que somam 100%.'))
caixas.append(dict(nome='Diagnóstico', tipo='saida', linhas=[], params=0,
                   explica='A classe com a maior probabilidade é o diagnóstico da rede.'))
n = len(caixas)

# 2) números reais de cada exemplo: ativação máxima e % de posições ligadas em cada bloco; probabilidades finais
reais = [str(val_t.loc[i, 'classe']) for i in idx_ex]
def pc(p):
    return f'{100 * p:.1f}%'.replace('.', ',')
def textos(k):
    t = []
    for c in caixas:
        if c['tipo'] == 'bloco':
            L = int(c['nome'].split()[-1]) - 1
            m = mapas[L][k]
            lin = [f'<b>{c["nome"]}</b>'] + c['linhas'] + ['máx.: ' + f'{m.max():.1f}'.replace('.', ','), f'ligadas: {100 * (m > 0).mean():.0f}%']
        elif c['tipo'] == 'saida':
            lin = ['<b>Diagnóstico</b>'] + [(f'<b>▶ {CLASSES[j]} {pc(prob[k][j])}</b>' if j == pred[k] else f'{CLASSES[j]} {pc(prob[k][j])}') for j in range(4)] + [f'real: {reais[k]}']
        else:
            lin = [f'<b>{c["nome"]}</b>'] + c['linhas']
        t.append('<br>'.join(lin))
    return t
T = [textos(k) for k in range(len(X_ex))]

# 3) geometria (y cresce para baixo): 1ª linha com 5 caixas, 2ª linha com o resto
LARG, ALT, GAP_X = 17.0, 26.0, 3.75
pos = []
for i in range(n):
    if i < 5:
        x0, y0 = i * (LARG + GAP_X), 3.0
    else:
        x0, y0 = (i - 5) * (LARG + GAP_X), 40.0
    larg = 24.0 if caixas[i]['tipo'] == 'saida' else LARG
    pos.append((x0, y0, larg))
centros = [(x0 + w / 2, y0 + ALT / 2) for x0, y0, w in pos]

AZUL, OURO, CINZA, CLARO = '#1F4E79', '#C9A227', '#F2F4F7', '#BFD3E6'
def cores(passo):
    return [OURO if i == passo else (CLARO if i < passo else CINZA) for i in range(n)]

fig = go.Figure()
c0 = cores(0)
for i, c in enumerate(caixas):
    x0, y0, w = pos[i]
    fig.add_trace(go.Scatter(x=[x0, x0 + w, x0 + w, x0, x0], y=[y0, y0, y0 + ALT, y0 + ALT, y0], mode='lines', fill='toself', fillcolor=c0[i],
                             line=dict(color=AZUL, width=2), hoveron='fills', hoverinfo='text',
                             text=f'<b>{c["nome"]}</b><br>{c["explica"]}<br>Parâmetros: {c["params"]:,}'.replace(',', '.'), showlegend=False))
for k in range(len(X_ex)):
    ycent = [centros[i][1] + (8 if i == 0 else 0) for i in range(n)]
    fig.add_trace(go.Scatter(x=[cx for cx, cy in centros], y=ycent, mode='text', text=T[k], textfont=dict(size=10, color='#1B2A3A'),
                             visible=(k == 0), showlegend=False, hoverinfo='skip'))
def miniatura(k):                                                                 # miniatura 64×64 da imagem que a rede vê, como PNG embutido
    arr = np.clip(X_ex[k][::2, ::2, 0], 0, 255).astype('uint8')
    buf = io.BytesIO(); Image.fromarray(arr).save(buf, format='PNG')
    return 'data:image/png;base64,' + base64.b64encode(buf.getvalue()).decode()
x0, y0, w = pos[0]
imagem = dict(source=miniatura(0), xref='x', yref='y', x=x0 + 2.5, y=y0 + 2, sizex=w - 5, sizey=w - 5, xanchor='left', yanchor='top', layer='above')

# setas entre as caixas
setas = []
for i in range(n - 1):
    xa, ya = centros[i]; xb, yb = centros[i + 1]
    if i == 4:                                                                  # cotovelo: desce da 1ª linha e volta à esquerda
        fig.add_trace(go.Scatter(x=[xa, xa, xb], y=[pos[i][1] + ALT, 34.5, 34.5], mode='lines', line=dict(color=AZUL, width=2), showlegend=False, hoverinfo='skip'))
        setas.append(dict(x=xb, y=pos[i + 1][1], ax=xb, ay=34.5))
    else:
        setas.append(dict(x=pos[i + 1][0], y=ya, ax=pos[i][0] + pos[i][2], ay=ya))
anot = [dict(xref='x', yref='y', axref='x', ayref='y', showarrow=True, arrowhead=2, arrowwidth=2, arrowcolor=AZUL, text='', **s) for s in setas]

# animação: o sinal avança caixa a caixa (dourado = etapa atual, azul claro = já percorridas)
nomes_passos = [c['nome'] for c in caixas]
quadros = [go.Frame(name=nomes_passos[s], data=[go.Scatter(fillcolor=cc) for cc in cores(s)], traces=list(range(n))) for s in range(n)]
fig.frames = quadros
nE = len(X_ex)
botoes = [dict(label=nomes[k], method='update',
               args=[{'visible': [True] * n + [j == k for j in range(nE)] + [True]},
                     {'images[0].source': miniatura(k), 'title.text': f'Gráfico 02 — {nomes[k]}: real = {reais[k]}, rede diz = {CLASSES[pred[k]]}'}])
          for k in range(nE)]
passos = [dict(method='animate', label=nomes_passos[s], args=[[nomes_passos[s]], dict(mode='immediate', frame=dict(duration=0, redraw=True), transition=dict(duration=0))]) for s in range(n)]
fig.update_layout(title=dict(text=f'Gráfico 02 — {nomes[0]}: real = {reais[0]}, rede diz = {CLASSES[pred[0]]}', y=0.97, yanchor='top', font_size=15),
                  template='plotly_white', height=720, margin=dict(t=150, b=110, l=10, r=10), annotations=anot, images=[imagem],
                  xaxis=dict(visible=False, range=[-1, 101]), yaxis=dict(visible=False, range=[70, 0], scaleanchor='x', scaleratio=1),
                  sliders=[dict(active=0, steps=passos, x=0.04, len=0.9, currentvalue=dict(prefix='Etapa: '), pad=dict(t=30))],
                  updatemenus=[dict(type='dropdown', buttons=botoes, x=0.0, y=1.2, xanchor='left', yanchor='top'),
                               dict(type='buttons', direction='left', x=0.42, y=1.2, xanchor='left', yanchor='top',
                                    buttons=[dict(label='▶ Percorrer a rede', method='animate',
                                                  args=[None, dict(frame=dict(duration=1200, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
                                             dict(label='⏸ Pausar', method='animate',
                                                  args=[[None], dict(mode='immediate', frame=dict(duration=0, redraw=False), transition=dict(duration=0))])])])
fig.show()

# 4) tabela: o que cada exemplo produziu em cada bloco
lin = []
for k in range(nE):
    d = {'Exemplo': nomes[k], 'Classe real': reais[k], 'Rede diz': CLASSES[pred[k]], 'Confiança': f'{100 * prob[k][pred[k]]:.1f}%'}
    for L in range(4):
        d[f'Máx. bloco {L+1}'] = round(float(mapas[L][k].max()), 1)
    lin.append(d)
display(pd.DataFrame(lin).set_index('Exemplo'))

,Classe real,Rede diz,Confiança,Máx. bloco 1,Máx. bloco 2,Máx. bloco 3,Máx. bloco 4
Exemplo,,,,,,,
Exemplo — glioma,glioma,glioma,99.9%,3.3,5.7,7.0,12.6
Exemplo — meningioma,meningioma,glioma,51.1%,3.6,6.3,8.7,13.4
Exemplo — notumor,notumor,notumor,100.0%,4.5,7.0,11.4,29.1
Exemplo — pituitary,pituitary,pituitary,100.0%,3.3,5.1,6.0,10.7
Exemplo — notumor retangular,notumor,notumor,92.1%,4.2,5.5,6.9,9.5


**Observação:** no exemplo de glioma, a imagem entra como a ressonância real de 128×128, passa pelos quatro blocos e chega ao diagnóstico com **99,9% para glioma** (meningioma 0,0%, notumor 0,1%, pituitary 0,0%). Ao longo do caminho, a ativação máxima cresce de bloco em bloco (3,3 → 5,7 → 7,0 → 12,6), enquanto a porcentagem de posições ligadas cai (42% → 33% → 9% → 13%). Isso é o que esperamos de uma rede que começa detectando bordas e texturas em quase toda a imagem e termina com poucos sinais fortes e específicos nos blocos profundos. O padrão combina com o que medimos nas 958 imagens no Passo 17.

A tabela dos 5 exemplos traz três pontos:

1. **Quatro dos cinco exemplos foram classificados corretamente**, com confianças de 99,9%, 100,0%, 100,0% e 92,1% (o notumor retangular, o menos confiante dos acertos). O erro é o exemplo de meningioma, que a rede chamou de glioma com apenas **51,1%** de confiança. A rede errou *sem convicção*, e uma confiança perto de 50% é um sinal de alerta que a probabilidade final já entrega.

2. **A ativação máxima dos blocos não separa acerto de erro.** O meningioma errado tem máximos (3,6; 6,3; 8,7; 13,4) parecidos com os do glioma correto (3,3; 5,7; 7,0; 12,6). Quem diz que a rede está em dúvida é a probabilidade final, não a intensidade das ativações.

3. **O exemplo de notumor se destaca no Bloco 4**, com máximo de 29,1, contra 9,5 a 13,4 nos outros quatro. Esse contraste entre exemplos já tinha aparecido nos mapas do Passo 15 e merece atenção, porque no projeto original suspeitamos de um atalho ligado ao formato da imagem. Aqui é uma descrição de 5 exemplos, não uma prova.

*Cuidado:* são apenas 5 imagens escolhidas para ilustrar o caminho. Os números de acurácia do modelo continuam sendo os do projeto original (validação 91,13% e teste 81,38%).

## Passo 20 — Gráfico 03: as dimensões do tensor, canal por canal

Um **tensor** é só uma "caixa de números" com três medidas: **altura × largura × canais**. A imagem de ressonância entra como 128×128×1 (um único canal, o tom de cinza). A cada bloco da rede, essa caixa muda de forma: a altura e a largura caem pela metade (MaxPooling) e o número de canais dobra (mais filtros).

Neste gráfico vamos **abrir a caixa e olhar as fatias**. Cada quadradinho é **um canal**, ou seja, o mapa 2D que um filtro produziu para a imagem escolhida. Os números são reais, calculados pelo seu `modelo` com as 5 imagens de exemplo.

**Como ler**
- Cada estágio mostra os **8 canais mais ativos** (os de maior ativação média) daquele tensor. Os outros canais existem, mas não cabem na tela.
- Mude o estágio pelo controle deslizante (Entrada, Bloco 1 ... Bloco 4) ou clique em **▶ Percorrer os estágios**. Repare nos quadrados ficando **mais grossos**: é a perda de resolução (128 → 64 → 32 → 16 → 8 pixels de lado).
- A lista no canto esquerdo troca a imagem de exemplo.
- Aqui "Bloco k" é a saída do k-ésimo MaxPooling (depois da convolução e do ReLU). O título de cada estágio traz o formato e o total de valores.

A tabela no final resume a mudança de forma: o número de valores só cai (16.384 → 131.072 → ... → 16.384 no fim), mas cada valor passa a carregar muito mais informação, porque deixa de ser um pixel e passa a ser "quanto um filtro gostou daquele trecho".

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1) passa as imagens de exemplo pelo modelo real e guarda o tensor de cada estágio (entrada e saída de cada MaxPooling)
a = X_ex.astype('float32')
estagios = [('Entrada', a)]
n_bloco = 0
for camada in modelo.layers:
    if isinstance(camada, keras.layers.InputLayer):
        continue
    a = np.array(camada(a))
    if isinstance(camada, keras.layers.MaxPooling2D):
        n_bloco += 1
        estagios.append((f'Bloco {n_bloco}', a))
    if isinstance(camada, keras.layers.GlobalAveragePooling2D):
        break

nE, NT = len(X_ex), 8                                           # nº de exemplos, nº de canais mostrados (os 8 mais ativos)
def fmt(v):
    return f'{v:,}'.replace(',', '.')
def dados(k, s):
    """tensor do estágio s para o exemplo k: lista de (canal, mapa 2D) dos canais mais ativos"""
    t = estagios[s][1][k]                                       # altura × largura × canais
    ordem = np.argsort(-t.mean(axis=(0, 1)))[:NT]
    return [(int(c), t[:, :, c]) for c in ordem]
def titulo(s):
    h, l, c = estagios[s][1].shape[1:]
    return (f'Gráfico 03 — {estagios[s][0]}: {h}×{l}×{c} = {fmt(h * l * c)} valores'
            f'<br><sup>cada quadro é 1 canal ({h}×{l} pixels); mostrando {min(NT, c)} de {c} {"canal" if c == 1 else "canais (os mais ativos)"}</sup>')

def tile(k, s, j):
    d = dados(k, s)
    if j < len(d):
        c, m = d[j]
        h, l = m.shape
        return go.Heatmap(z=np.round(m.astype('float64'), 1), x0=0.5 / l, dx=1 / l, y0=0.5 / h, dy=1 / h, zmin=0, zmax=float(max(mm.max() for _, mm in d)), colorscale='gray' if s == 0 else 'Viridis',
                          showscale=False, hovertemplate=f'canal {c + 1}<br>valor %{{z}}<extra></extra>')
    return go.Heatmap(z=[[None]], x0=0.5, dx=1, y0=0.5, dy=1, showscale=False, hoverinfo='skip')

# 2) figura: grade 2×4, um conjunto de 8 quadros por exemplo (só o do exemplo escolhido fica visível)
fig = make_subplots(rows=2, cols=4, horizontal_spacing=0.03, vertical_spacing=0.06)
for k in range(nE):
    for j in range(NT):
        fig.add_trace(tile(k, 0, j), row=j // 4 + 1, col=j % 4 + 1)
        fig.data[-1].visible = (k == 0)
quadros = [go.Frame(name=estagios[s][0], data=[tile(k, s, j) for k in range(nE) for j in range(NT)], traces=list(range(nE * NT)),
                    layout=go.Layout(title_text=titulo(s))) for s in range(len(estagios))]
fig.frames = quadros
for i in range(1, NT + 1):
    sx = 'x' if i == 1 else f'x{i}'
    fig.update_yaxes(scaleanchor=sx, constrain='domain', range=[1, 0], showticklabels=False, row=(i - 1) // 4 + 1, col=(i - 1) % 4 + 1)
    fig.update_xaxes(constrain='domain', range=[0, 1], showticklabels=False, row=(i - 1) // 4 + 1, col=(i - 1) % 4 + 1)

botoes = [dict(label=nomes[k], method='update', args=[{'visible': [(j // NT) == k for j in range(nE * NT)]}]) for k in range(nE)]
passos = [dict(method='animate', label=estagios[s][0], args=[[estagios[s][0]], dict(mode='immediate', frame=dict(duration=0, redraw=True), transition=dict(duration=0))]) for s in range(len(estagios))]
fig.update_layout(title=dict(text=titulo(0), y=0.97, yanchor='top', font_size=15), template='plotly_white', height=660, margin=dict(t=190, b=110, l=10, r=10),
                  sliders=[dict(active=0, steps=passos, x=0.04, len=0.9, currentvalue=dict(prefix='Estágio: '), pad=dict(t=30))],
                  updatemenus=[dict(type='dropdown', buttons=botoes, x=0.0, y=1.25, xanchor='left', yanchor='top'),
                               dict(type='buttons', direction='left', x=0.42, y=1.25, xanchor='left', yanchor='top',
                                    buttons=[dict(label='▶ Percorrer os estágios', method='animate',
                                                  args=[None, dict(frame=dict(duration=2000, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
                                             dict(label='⏸ Pausar', method='animate',
                                                  args=[[None], dict(mode='immediate', frame=dict(duration=0, redraw=False), transition=dict(duration=0))])])])
fig.show()

# 3) tabela: como o tensor muda a cada estágio
lin = []
for nome, t in estagios:
    h, l, c = t.shape[1:]
    lin.append({'Estágio': nome, 'Altura': h, 'Largura': l, 'Canais': c, 'Valores': h * l * c, 'Pixels por canal': h * l})
tab = pd.DataFrame(lin).set_index('Estágio')
display(tab)
print('\nAltura e largura caem pela metade a cada MaxPooling; os canais dobram a cada bloco.')

,Altura,Largura,Canais,Valores,Pixels por canal
Estágio,,,,,
Entrada,128,128,1,16384,16384
Bloco 1,64,64,32,131072,4096
Bloco 2,32,32,64,65536,1024
Bloco 3,16,16,128,32768,256
Bloco 4,8,8,256,16384,64



Altura e largura caem pela metade a cada MaxPooling; os canais dobram a cada bloco.


**Observação:** com a mesma imagem de glioma, dá para acompanhar o tensor perdendo resolução e ganhando canais, bloco a bloco.

- **Entrada (128×128×1):** é a própria ressonância, um único canal.
- **Bloco 1 (64×64×32 = 131.072 valores):** a cabeça ainda é perfeitamente reconhecível em todos os 8 canais mostrados. Cada canal ressalta algo diferente: alguns destacam o contorno do crânio (o anel mais brilhante), outros o tecido interno.
- **Bloco 2 (32×32×64 = 65.536 valores):** a cabeça continua visível, mas "pixelada", e já aparecem canais que quase só desenham contornos.
- **Bloco 3 (16×16×128 = 32.768 valores):** só o primeiro canal ainda lembra uma silhueta de cabeça, em blocos grossos. Os outros viraram faixas verticais nas laterais, ou seja, já não se parecem com a imagem original.
- **Bloco 4 (8×8×256 = 16.384 valores):** com 64 pixels por canal, nenhuma cabeça é reconhecível. Cada canal vira um padrão de poucos pontos e faixas brilhantes, sobretudo nas bordas laterais e em alguns pontos isolados.

A mensagem do gráfico é que a rede **troca "onde está cada pixel" por "que tipo de padrão apareceu e aproximadamente onde"**. A resolução cai de 128×128 para 8×8, e os canais passam de 1 para 256, então cada valor deixa de ser um pixel e passa a ser a resposta de um filtro a um trecho da imagem.

Um ponto que merece atenção: nos blocos profundos, vários canais acendem nas **laterais** dos mapas. Isso pode refletir um padrão real da imagem, mas também bordas, fundo preto ou preenchimento (padding), e é o tipo de pista que, no projeto original, nos fez suspeitar de um atalho ligado ao formato da imagem. Aqui é só uma descrição de um exemplo, não uma prova.

*Cuidado:* os quadros mostram apenas os 8 canais **mais ativos** de cada estágio (de 32, 64, 128 e 256), e a escala de cores é ajustada dentro de cada estágio. Por isso não dá para comparar a intensidade entre estágios.

## Passo 21 — Gráfico 06: o grafo computacional (ida e volta)

Toda rede neural é uma **cadeia de operações**: a imagem entra, passa por cada camada e termina em um número só, a **perda L**, que mede o tamanho do erro. Desenhar essa cadeia é desenhar o **grafo computacional** da rede. Ele tem duas viagens:

- **IDA (forward):** da imagem até a perda. Cada caixa transforma o que recebeu: Conv 1 → Conv 2 → Conv 3 → Conv 4 → Média global → Dense → Softmax → Perda. As caixinhas de cima (e a de baixo, da Dense) são os **parâmetros** (pesos e vieses) que cada camada usa.
- **VOLTA (backward):** da perda de volta até a primeira camada. Para cada parâmetro, a volta calcula o **gradiente**: quanto a perda mudaria se aquele parâmetro mudasse um pouco. É assim que a rede descobre **quais pesos ajustar** para errar menos.

Tudo aqui é calculado com o seu `modelo` real, sobre as 5 imagens de exemplo. Para cada imagem o código mostra:
- na **ida:** os 4 valores brutos da Dense (z), as probabilidades do softmax e a perda `L = −log p(classe real)`;
- na **volta:** o tamanho do gradiente (a norma ‖∇L‖) de cada camada, nas caixas e no gráfico de barras embaixo.

**Como usar:** escolha o exemplo na lista, arraste o controle deslizante (ou clique em **▶ Ida e volta**) e acompanhe: na ida as caixas ficam douradas e depois azuis; na volta ficam vermelhas, e a caixa de parâmetros fica laranja quando o gradiente chega nela.

**Importante:** aqui não treinamos nada. O modelo está exatamente como foi salvo (com o Dropout desligado), e só *medimos* os gradientes de uma imagem por vez. O notebook original não é tocado.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import tensorflow as tf
from plotly.subplots import make_subplots

# 1) arquitetura real: camadas com parâmetros (Conv2D e Dense), na ordem em que aparecem
parametricas = [c for c in modelo.layers if isinstance(c, (keras.layers.Conv2D, keras.layers.Dense))]
nomes_camadas = [f'Conv {i + 1}' for i in range(len(parametricas) - 1)] + ['Dense']
forma_saida, x = [], np.zeros((1,) + tuple(modelo.input_shape[1:]), dtype='float32')
for c in modelo.layers:
    if isinstance(c, keras.layers.InputLayer):
        continue
    x = np.array(c(x))
    if isinstance(c, keras.layers.MaxPooling2D):
        forma_saida.append('×'.join(str(v) for v in x.shape[1:]))

# 2) para cada exemplo: valores da ida (z, p, perda) e da volta (norma do gradiente da perda em cada camada)
reais = [str(val_t.loc[i, 'classe']) for i in idx_ex]
info = []
for k in range(len(X_ex)):
    xi = X_ex[k:k + 1].astype('float32')
    y = CLASSES.index(reais[k])
    with tf.GradientTape() as fita:
        p = modelo(xi, training=False)
        perda = -tf.math.log(p[0, y] + 1e-12)
    grads = fita.gradient(perda, modelo.trainable_variables)                 # [kernel, bias] de cada camada, na ordem
    normas = [float(np.sqrt(sum(np.sum(np.array(g) ** 2) for g in grads[2 * i:2 * i + 2]))) for i in range(len(parametricas))]
    h = xi                                                                   # entrada da Dense -> logits z = h·W + b
    for c in modelo.layers:
        if isinstance(c, keras.layers.InputLayer):
            continue
        if isinstance(c, keras.layers.Dense):
            W, b = c.get_weights()
            z = np.array(h).reshape(-1) @ W + b
            break
        h = np.array(c(h))
    info.append(dict(y=y, p=np.array(p)[0], z=z, perda=float(perda), normas=normas))

def v(a, casas=1):
    return f'{a:.{casas}f}'.replace('.', ',')
def textos(k):
    d = info[k]
    n_par = [c.count_params() for c in parametricas]
    return [
        ['<b>Imagem</b>', '128×128×1', f'real: {reais[k]}'],
        *[[f'<b>{nomes_camadas[i]}</b>', f'{parametricas[i].filters} filtros 3×3', '+ ReLU + Pool', f'saída {forma_saida[i]}'] for i in range(4)],
        ['<b>Média global</b>', '256 números'],
        ['<b>Dense</b>', 'z = W·h + b', 'z:  ' + '  '.join(v(a) for a in d['z'][:2]), '       ' + '  '.join(v(a) for a in d['z'][2:])],
        ['<b>Softmax</b>'] + [(f'<b>▶ {CLASSES[j]} {v(100 * d["p"][j])}%</b>' if j == int(d['p'].argmax()) else f'{CLASSES[j]} {v(100 * d["p"][j])}%') for j in range(4)],
        ['<b>Perda L</b>', 'L = −log p(real)', f'p(real) = {v(d["p"][d["y"]], 3)}', f'<b>L = {v(d["perda"], 3)}</b>'],
        *[[f'<b>W, b ({nomes_camadas[i]})</b>', f'{n_par[i]:,}'.replace(',', '.') + ' param.', f'‖∇L‖ = {v(d["normas"][i], 3)}'] for i in range(5)],
    ]
T = [textos(k) for k in range(len(X_ex))]

# 3) geometria (y cresce para baixo): cadeia principal em 2 linhas, parâmetros em caixas menores
LARG, GAP_X = 17.0, 3.75
cadeia = ['Imagem', 'Conv 1', 'Conv 2', 'Conv 3', 'Conv 4', 'Média global', 'Dense', 'Softmax', 'Perda L']
col = lambda i: (i if i < 5 else i - 5) * (LARG + GAP_X)
caixas = []                                                                  # (x0, y0, largura, altura)
for i in range(9):
    caixas.append((col(i), 15.0 if i < 5 else 50.0, 24.0 if i == 8 else LARG, 22.0))
for i in range(4):
    caixas.append((col(i + 1), 2.0, LARG, 9.0))                              # parâmetros das 4 convoluções, acima
caixas.append((col(6), 78.0, LARG, 9.0))                                     # parâmetros da Dense, abaixo
n_no = len(caixas)
centros = [(x0 + w / 2, y0 + h / 2) for x0, y0, w, h in caixas]
pai = {9: 1, 10: 2, 11: 3, 12: 4, 13: 6}                                     # caixa de parâmetros -> nó da cadeia

AZUL, OURO, CINZA, CLARO, VERMELHO, PASSADO, LARANJA = '#1F4E79', '#C9A227', '#F2F4F7', '#BFD3E6', '#F1948A', '#FADBD8', '#F8C471'

# 4) passos da animação: ida (nós acendem da esquerda para a direita) e volta (o gradiente caminha ao contrário)
nomes_passos = ['Entrada', 'C1', 'C2', 'C3', 'C4', 'Média', 'Dense', 'Softmax', 'Perda', '∇ Perda', '∇ Softmax', '∇ Dense', '∇ Média', '∇ C4', '∇ C3', '∇ C2', '∇ C1']
volta = [8, 7, 6, 5, 4, 3, 2, 1]                                             # nós visitados na volta
def cores(p):
    c = [CINZA] * n_no
    if p <= 8:                                                               # ida
        for i in range(9):
            c[i] = OURO if i == p else (CLARO if i < p else CINZA)
        for q, i in pai.items():
            c[q] = CLARO if i <= p else CINZA
    else:                                                                    # volta
        atual = volta[p - 9]
        for i in range(9):
            c[i] = VERMELHO if i == atual else (PASSADO if i > atual else CLARO)
        for q, i in pai.items():
            c[q] = LARANJA if i >= atual else CLARO
    return c
def fase(p):
    return 'IDA: a imagem vira previsão' if p <= 8 else 'VOLTA: o erro vira gradiente'

fig = make_subplots(rows=2, cols=1, row_heights=[0.74, 0.26], vertical_spacing=0.1,
                    subplot_titles=('', 'Quanto cada camada "sente" o erro (‖∇L‖), exemplo escolhido'))
c0 = cores(0)
for i, (x0, y0, w, h) in enumerate(caixas):
    fig.add_trace(go.Scatter(x=[x0, x0 + w, x0 + w, x0, x0], y=[y0, y0, y0 + h, y0 + h, y0], mode='lines', fill='toself', fillcolor=c0[i],
                             line=dict(color=AZUL, width=2), hoverinfo='skip', showlegend=False), row=1, col=1)
for k in range(len(X_ex)):
    fig.add_trace(go.Scatter(x=[c[0] for c in centros], y=[c[1] for c in centros], mode='text', text=['<br>'.join(t) for t in T[k]],
                             textfont=dict(size=9, color='#1B2A3A'), visible=(k == 0), showlegend=False, hoverinfo='skip'), row=1, col=1)
rot = ['W₁', 'W₂', 'W₃', 'W₄', 'W Dense']
for k in range(len(X_ex)):
    fig.add_trace(go.Bar(x=rot, y=info[k]['normas'], marker=dict(color=['#1F4E79', '#1F4E79', '#1F4E79', '#1F4E79', '#C0392B']),
                         text=[v(a, 2) for a in info[k]['normas']], textposition='outside', visible=(k == 0), showlegend=False,
                         hovertemplate='%{x}: ‖∇L‖ = %{y:.4f}<extra></extra>'), row=2, col=1)
teto = 1.2 * max(max(d['normas']) for d in info)

# setas
def seta(x, y, ax, ay):
    fig.add_annotation(x=x, y=y, ax=ax, ay=ay, xref='x', yref='y', axref='x', ayref='y', showarrow=True, arrowhead=2, arrowwidth=2, arrowcolor=AZUL, text='')
for i in range(8):
    if i == 4:
        fig.add_trace(go.Scatter(x=[centros[4][0], centros[4][0], centros[5][0]], y=[caixas[4][1] + 22, 44, 44], mode='lines', line=dict(color=AZUL, width=2),
                                 showlegend=False, hoverinfo='skip'), row=1, col=1)
        seta(centros[5][0], caixas[5][1], centros[5][0], 44)
    else:
        seta(caixas[i + 1][0], centros[i][1], caixas[i][0] + caixas[i][2], centros[i][1])
for q, i in pai.items():
    if q < 13:
        seta(centros[i][0], caixas[i][1], centros[q][0], caixas[q][1] + 9)
    else:
        seta(centros[i][0], caixas[i][1] + 22, centros[q][0], caixas[q][1])
n_fill, nE = n_no, len(X_ex)
fig.update_xaxes(visible=False, range=[-1, 101], row=1, col=1)
fig.update_yaxes(visible=False, range=[88, 0], row=1, col=1)
fig.update_yaxes(range=[0, teto], title_text='‖∇L‖', row=2, col=1)

quadros = [go.Frame(name=nomes_passos[p], data=[go.Scatter(fillcolor=cc) for cc in cores(p)], traces=list(range(n_fill)),
                    layout=go.Layout(title_text=f'Gráfico 06 — {fase(p)}')) for p in range(len(nomes_passos))]
fig.frames = quadros
n_traces = n_fill + 2 * nE
botoes = [dict(label=nomes[k], method='update',
               args=[{'visible': [True] * n_fill + [j == k for j in range(nE)] * 2 + [True]}]) for k in range(nE)]
passos = [dict(method='animate', label=nm, args=[[nm], dict(mode='immediate', frame=dict(duration=0, redraw=True), transition=dict(duration=0))]) for nm in nomes_passos]
fig.update_annotations(font_size=12)
fig.update_layout(title=dict(text=f'Gráfico 06 — {fase(0)}', y=0.98, yanchor='top', font_size=15), template='plotly_white', height=920,
                  margin=dict(t=170, b=110, l=50, r=10),
                  sliders=[dict(active=0, steps=passos, x=0.03, len=0.94, currentvalue=dict(prefix='Passo: '), pad=dict(t=40), font=dict(size=9))],
                  updatemenus=[dict(type='dropdown', buttons=botoes, x=0.0, y=1.14, xanchor='left', yanchor='top'),
                               dict(type='buttons', direction='left', x=0.42, y=1.14, xanchor='left', yanchor='top',
                                    buttons=[dict(label='▶ Ida e volta', method='animate',
                                                  args=[None, dict(frame=dict(duration=1100, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
                                             dict(label='⏸ Pausar', method='animate',
                                                  args=[[None], dict(mode='immediate', frame=dict(duration=0, redraw=False), transition=dict(duration=0))])])])
fig.show()

# 5) tabela: o que cada exemplo produziu na ida (perda) e na volta (gradiente em cada camada)
lin = []
for k in range(nE):
    d = info[k]
    r = {'Exemplo': nomes[k], 'Classe real': reais[k], 'Rede diz': CLASSES[int(d['p'].argmax())], 'p(real)': round(float(d['p'][d['y']]), 3), 'Perda L': round(d['perda'], 3)}
    for i, nm in enumerate(nomes_camadas):
        r[f'‖∇L‖ {nm}'] = round(d['normas'][i], 3)
    lin.append(r)
display(pd.DataFrame(lin).set_index('Exemplo'))

,Classe real,Rede diz,p(real),Perda L,‖∇L‖ Conv 1,‖∇L‖ Conv 2,‖∇L‖ Conv 3,‖∇L‖ Conv 4,‖∇L‖ Dense
Exemplo,,,,,,,,,
Exemplo — glioma,glioma,glioma,0.999,0.001,0.047,0.017,0.020,0.006,0.008
Exemplo — meningioma,meningioma,glioma,0.478,0.739,22.175,8.871,8.205,2.919,5.789
Exemplo — notumor,notumor,notumor,1.000,-0.000,0.000,0.000,0.000,0.000,0.000
Exemplo — pituitary,pituitary,pituitary,1.000,0.000,0.004,0.004,0.005,0.001,0.002
Exemplo — notumor retangular,notumor,notumor,0.921,0.082,3.255,1.632,1.288,0.413,0.727


**Observação:** o grafo mostra os dois sentidos da rede. Na **ida**, a imagem vira uma previsão e uma perda `L = −log p(classe real)`. Na **volta**, essa perda vira um gradiente para cada camada, dizendo quanto ela "sentiu" o erro. No exemplo de glioma, a rede deu 99,9% para a classe certa (z = 6,1 / −2,6 / −1,5 / −6,0), a perda foi de 0,001 e a volta devolveu gradientes minúsculos (‖∇L‖ de 0,047 na Conv 1 a 0,006 na Conv 4).

A tabela dos 5 exemplos mostra a regra por trás disso:

| Exemplo | p(classe real) | Perda L | ‖∇L‖ Conv 1 | ‖∇L‖ Conv 4 |
|---|---|---|---|---|
| glioma | 0,999 | 0,001 | 0,047 | 0,006 |
| meningioma (erro) | 0,478 | 0,739 | 22,175 | 2,919 |
| notumor | 1,000 | 0,000 | 0,000 | 0,000 |
| pituitary | 1,000 | 0,000 | 0,004 | 0,001 |
| notumor retangular | 0,921 | 0,082 | 3,255 | 0,413 |

1. **Quem acerta com convicção quase não gera gradiente.** Nos exemplos com p(real) perto de 1, a perda e todos os gradientes são praticamente zero. Se a rede já está certa, não há o que corrigir, e por isso o gráfico de barras do glioma parece vazio: a escala é a mesma para os 5 exemplos, e as barras de 0,05 ou menos desaparecem ao lado das do meningioma.

2. **O erro é onde a rede "aprende".** No meningioma, que a rede confundiu com glioma (p(real) = 0,478), a perda sobe para 0,739 e o gradiente da Conv 1 vai para 22,175, cerca de 470 vezes o do glioma. O notumor retangular, que a rede acertou com 92,1%, já tem gradientes intermediários (Conv 1 = 3,255). **Quanto menos confiante a rede está na classe certa, maior o gradiente que volta.**

3. **O padrão entre camadas se repete.** Em todos os exemplos com gradiente visível, a Conv 1 tem a maior norma e a Conv 4 a menor, apesar de a Conv 4 ter cerca de 76% dos parâmetros. A norma mede o tamanho do gradiente da camada inteira, não o de cada peso, então não dá para concluir daí qual camada "importa mais". É uma medida de quanto a perda seria sensível a mudanças naquela camada, para essa imagem.

*Cuidados:* (a) são gradientes de **uma imagem por vez** com os pesos já treinados e o Dropout desligado; nada foi atualizado, e o modelo continua exatamente como foi salvo. (b) O "−0,000" na perda do notumor vem de um arredondamento numérico: o código soma 1e-12 dentro do logaritmo para evitar log(0), e uma probabilidade de 1,000 vira um valor ligeiramente negativo. Na prática, a perda é zero. (c) São 5 exemplos escolhidos, não uma estatística do conjunto de validação.

## Passo 22 — Gráfico 05: o sinal percorre a rede e a decisão aparece

Até agora vimos a rede por fora (Gráfico 01), em fluxograma (02), em fatias (03) e em ida e volta (06). Neste gráfico vamos **assistir à propagação** de uma imagem: ela entra, passa pelos 4 blocos e termina numa decisão. E, no último passo, vamos ver **onde na imagem a rede encontrou os argumentos** para decidir.

**A linha de cima: os 5 estágios.** Cada quadro é a **média de todos os canais** daquele estágio, ou seja, um resumo do tensor que vimos em fatias no Passo 20. A resolução cai: 128×128 → 64×64 → 32×32 → 16×16 → 8×8. No **▶ Propagar o sinal**, cada estágio se acende no seu passo e o anterior fica mais pálido.

**A linha de baixo: a decisão.** Como a Dense vem logo depois da Média global, a conta final tem um formato muito conveniente:

> valor bruto da classe j = **média, sobre as 64 posições do mapa 8×8, da contribuição de cada posição** + viés

Então, para a classe prevista, podemos desenhar a contribuição de **cada posição**: em **vermelho** estão as posições que empurram a decisão a favor da classe prevista, e em **azul** as que empurram contra. Esse mapa fica sobre a imagem original. À direita, as probabilidades finais do softmax.

**Cuidados de leitura:** o mapa tem só 8×8 posições e é esticado sobre a imagem, então indica **regiões aproximadas**, não contornos de tumor. A reconstrução é exata (o código confere que o softmax dos valores reconstruídos reproduz as probabilidades do modelo), mas "onde a rede olhou" descreve a conta da rede, e **não prova que ela olhou onde um médico olharia**.

In [ ]:
import io, base64
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from PIL import Image
from plotly.subplots import make_subplots

# 1) passa os exemplos pelo modelo real e guarda a saída de cada bloco (depois do MaxPooling)
a = X_ex.astype('float32')
estagios = [('Entrada', a)]
n_bloco = 0
for camada in modelo.layers:
    if isinstance(camada, keras.layers.InputLayer):
        continue
    a = np.array(camada(a))
    if isinstance(camada, keras.layers.MaxPooling2D):
        n_bloco += 1
        estagios.append((f'Bloco {n_bloco}', a))
    if isinstance(camada, keras.layers.GlobalAveragePooling2D):
        break
densa = [c for c in modelo.layers if isinstance(c, keras.layers.Dense)][-1]
W, b = densa.get_weights()                                       # W: 256 × 4 ; b: 4

# 2) mapa de decisão exato: como a Dense vem logo depois da Média global, z_j = média_das_posições( A · W[:, j] ) + b_j
A = estagios[-1][1]                                              # (exemplos, 8, 8, 256)
nE = len(X_ex)
CAM = np.einsum('nhwc,cj->nhwj', A, W)                           # contribuição de cada posição 8×8 para cada classe
z_cam = CAM.mean(axis=(1, 2)) + b                                # reconstrução dos 4 valores brutos
reais = [str(val_t.loc[i, 'classe']) for i in idx_ex]
print('Conferência: o maior desvio entre softmax(z reconstruído) e as probabilidades do modelo é',
      f'{np.abs(np.exp(z_cam) / np.exp(z_cam).sum(1, keepdims=True) - prob).max():.2e}')

def png(k):                                                      # a imagem que a rede vê, como PNG embutido
    arr = np.clip(X_ex[k][:, :, 0], 0, 255).astype('uint8')
    buf = io.BytesIO(); Image.fromarray(arr).save(buf, format='PNG')
    return 'data:image/png;base64,' + base64.b64encode(buf.getvalue()).decode()

# 3) figura: linha 1 = os 5 estágios (média dos canais); linha 2 = mapa de decisão e probabilidades
titulos = ['Entrada<br>128×128', 'Bloco 1<br>64×64', 'Bloco 2<br>32×32', 'Bloco 3<br>16×16', 'Bloco 4<br>8×8', 'Onde a rede olhou (classe prevista)', 'Probabilidades']
fig = make_subplots(rows=2, cols=5, specs=[[{}] * 5, [{'colspan': 2}, None, {'colspan': 3}, None, None]], subplot_titles=titulos,
                    horizontal_spacing=0.02, vertical_spacing=0.14, row_heights=[0.42, 0.58])
ordem = []                                                       # (tipo, índice do exemplo, índice do estágio) de cada traço
for k in range(nE):
    for s, (nome, t) in enumerate(estagios):
        m = t[k].mean(axis=2) if s > 0 else t[k][:, :, 0]
        h, l = m.shape
        fig.add_trace(go.Heatmap(z=np.round(m.astype('float64'), 1), x0=0.5 / l, dx=1 / l, y0=0.5 / h, dy=1 / h, zmin=0, zmax=float(m.max()),
                                 colorscale='gray' if s == 0 else 'Viridis', showscale=False, opacity=1.0 if s == 0 else 0.0, visible=(k == 0),
                                 hovertemplate='valor %{z}<extra></extra>'), row=1, col=s + 1)
        ordem.append(('tile', k, s))
    j = int(prob[k].argmax()); c = CAM[k][:, :, j]; lim = float(np.abs(c).max())
    fig.add_trace(go.Heatmap(z=np.round(c.astype('float64'), 2), x0=0.5 / 8, dx=1 / 8, y0=0.5 / 8, dy=1 / 8, zmin=-lim, zmax=lim, zmid=0, colorscale='RdBu_r',
                             zsmooth='best', showscale=False, opacity=0.0, visible=(k == 0),
                             hovertemplate='contribuição %{z}<extra></extra>'), row=2, col=1)
    ordem.append(('cam', k, 5))
    cores_b = ['#1F4E79' if q == j else '#9AA5B1' for q in range(4)]
    fig.add_trace(go.Bar(x=list(CLASSES), y=prob[k], marker=dict(color=cores_b), text=[f'{100 * p:.1f}%'.replace('.', ',') for p in prob[k]], textposition='outside',
                         opacity=0.0, visible=(k == 0), showlegend=False, hovertemplate='%{x}: %{y:.3f}<extra></extra>'), row=2, col=3)
    ordem.append(('bar', k, 5))

for i in range(1, 7):                                            # quadros quadrados e sem eixos
    sx = 'x' if i == 1 else f'x{i}'
    r, c_ = (1, i) if i <= 5 else (2, 1)
    fig.update_yaxes(scaleanchor=sx, constrain='domain', range=[1, 0], showticklabels=False, showgrid=False, zeroline=False, row=r, col=c_)
    fig.update_xaxes(constrain='domain', range=[0, 1], showticklabels=False, showgrid=False, zeroline=False, row=r, col=c_)
fig.update_yaxes(range=[0, 1.18], row=2, col=3)
fig.update_annotations(font_size=11)

def imagem_base(k):
    return dict(source=png(k), xref='x6', yref='y6', x=0, y=0, sizex=1, sizey=1, xanchor='left', yanchor='top', layer='below')

# 4) animação: o sinal avança do estágio 0 ao 4; no passo 5 aparecem o mapa de decisão e as probabilidades
nomes_passos = ['Entrada', 'Bloco 1', 'Bloco 2', 'Bloco 3', 'Bloco 4', 'Decisão']
legendas = ['a imagem entra', 'o sinal passa pelo Bloco 1', 'o sinal passa pelo Bloco 2', 'o sinal passa pelo Bloco 3', 'o sinal chega ao Bloco 4', 'a decisão: onde a rede olhou']
def opac(p):
    o = []
    for tipo, k, s in ordem:
        if tipo == 'tile':
            o.append(1.0 if s == p else (0.55 if s < p else 0.0))
        else:
            o.append(0.6 if (tipo == 'cam' and p == 5) else (1.0 if p == 5 else 0.0))
    return o
quadros = [go.Frame(name=nomes_passos[p], data=[go.Heatmap(opacity=o) if ordem[q][0] != 'bar' else go.Bar(opacity=o) for q, o in enumerate(opac(p))],
                    traces=list(range(len(ordem))), layout=go.Layout(title_text=f'Gráfico 05 — {legendas[p]}')) for p in range(6)]
fig.frames = quadros
botoes = [dict(label=nomes[k], method='update',
               args=[{'visible': [q[1] == k for q in ordem]},
                     {'images[0].source': png(k), 'title.text': f'Gráfico 05 — {nomes[k]}: real = {reais[k]}, rede diz = {CLASSES[int(prob[k].argmax())]}'}]) for k in range(nE)]
passos = [dict(method='animate', label=nm, args=[[nm], dict(mode='immediate', frame=dict(duration=0, redraw=True), transition=dict(duration=0))]) for nm in nomes_passos]
fig.update_layout(title=dict(text=f'Gráfico 05 — {legendas[0]}', y=0.97, yanchor='top', font_size=15), template='plotly_white', height=760, images=[imagem_base(0)],
                  margin=dict(t=170, b=100, l=50, r=10),
                  sliders=[dict(active=0, steps=passos, x=0.04, len=0.9, currentvalue=dict(prefix='Passo: '), pad=dict(t=30))],
                  updatemenus=[dict(type='dropdown', buttons=botoes, x=0.0, y=1.2, xanchor='left', yanchor='top'),
                               dict(type='buttons', direction='left', x=0.42, y=1.2, xanchor='left', yanchor='top',
                                    buttons=[dict(label='▶ Propagar o sinal', method='animate',
                                                  args=[None, dict(frame=dict(duration=1500, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
                                             dict(label='⏸ Pausar', method='animate',
                                                  args=[[None], dict(mode='immediate', frame=dict(duration=0, redraw=False), transition=dict(duration=0))])])])
fig.show()

# 5) tabela: a conta do mapa de decisão fecha com a previsão do modelo
lin = []
for k in range(nE):
    j = int(prob[k].argmax()); c = CAM[k][:, :, j]
    lin.append({'Exemplo': nomes[k], 'Classe real': reais[k], 'Rede diz': CLASSES[j], 'z previsto (reconstruído)': round(float(z_cam[k][j]), 3),
                'Maior contribuição': round(float(c.max()), 2), 'Menor contribuição': round(float(c.min()), 2), 'Posições a favor (%)': round(100 * float((c > 0).mean()), 1)})
display(pd.DataFrame(lin).set_index('Exemplo'))

Conferência: o maior desvio entre softmax(z reconstruído) e as probabilidades do modelo é 1.19e-07


,Classe real,Rede diz,z previsto (reconstruído),Maior contribuição,Menor contribuição,Posições a favor (%)
Exemplo,,,,,,
Exemplo — glioma,glioma,glioma,6.070,53.95,-13.33,60.9
Exemplo — meningioma,meningioma,glioma,2.479,35.48,-12.46,53.1
Exemplo — notumor,notumor,notumor,28.454,125.65,-15.80,89.1
Exemplo — pituitary,pituitary,pituitary,7.639,50.13,-12.98,68.8
Exemplo — notumor retangular,notumor,notumor,3.550,44.58,-24.77,75.0


**Observação:** neste gráfico, o sinal de uma imagem de ressonância perde resolução a cada bloco (128×128 → 64×64 → 32×32 → 16×16 → 8×8). Nos primeiros blocos ainda se reconhece a cabeça, com o contorno do crânio em destaque. Nos blocos 3 e 4 sobram manchas grossas, com alguns pontos brilhantes perto dos cantos superiores. No fim, o mapa 8×8 do Bloco 4 é "traduzido" em uma decisão.

O exemplo mostrado é o de **meningioma**, em que a rede erra: 51,1% para glioma contra 47,8% para o meningioma real, uma diferença de apenas 3,3 pontos percentuais. O mapa "onde a rede olhou" mostra, em vermelho, as regiões que empurram a decisão a favor de **glioma**: elas se concentram na **região frontal junto ao crânio** (alto à esquerda), nos **cantos da imagem** e na parte inferior esquerda. Em azul (contra glioma) aparece uma mancha no alto, perto do centro. As estruturas centrais da cabeça não aparecem como as áreas mais fortes.

A tabela confirma que a conta do mapa fecha com a previsão e traz uma comparação entre os 5 exemplos (valores para a classe prevista):

| Exemplo | z da classe prevista | Maior contribuição | Menor contribuição | Posições a favor |
|---|---|---|---|---|
| glioma | 6,070 | 53,95 | −13,33 | 60,9% |
| meningioma (erro) | 2,479 | 35,48 | −12,46 | 53,1% |
| notumor | 28,454 | 125,65 | −15,80 | 89,1% |
| pituitary | 7,639 | 50,13 | −12,98 | 68,8% |
| notumor retangular | 3,550 | 44,58 | −24,77 | 75,0% |

1. **O exemplo errado é o menos "unânime".** O meningioma tem o menor valor bruto (2,479) e a menor fração de posições a favor da classe prevista (53,1%): a decisão foi quase um empate entre as regiões, e é isso que o 51,1% a 47,8% reflete.
2. **O exemplo de notumor se destaca de longe**: valor bruto de 28,454 (4 a 11 vezes o dos outros), contribuição máxima de 125,65 e 89,1% das posições a favor. Esse padrão já tinha aparecido no Passo 15 (ativações bem maiores nas camadas profundas) e no Passo 20 (máximo de 29,1 no Bloco 4). Para esse exemplo, a rede está muito mais "segura" do que nos outros.
3. **Atenção às bordas.** No exemplo de meningioma, boa parte do vermelho está nas bordas e nos cantos, e não sobre as estruturas centrais. Isso é coerente com a suspeita do projeto original de que o modelo pode se apoiar em pistas de formato ou de fundo, mas aqui é uma **descrição de um único exemplo**, não uma prova.

*Cuidados:* o mapa tem apenas 8×8 posições e é esticado sobre a imagem, então indica regiões aproximadas, não contornos. O campo de visão de cada posição é bem maior que um quadrado de 16×16 pixels, e o preenchimento (padding) das convoluções pode contribuir para o brilho nas bordas. "Onde a rede olhou" descreve a conta da rede, não o que um médico consideraria a lesão. São 5 exemplos escolhidos, não estatística do conjunto de validação.

## Passo 23 — Gráfico 04: a convolução em câmera lenta

Nos passos anteriores vimos os mapas que os filtros produzem. Agora vamos ver **a conta que gera cada número** de um mapa. Pegamos um filtro real da primeira camada (Conv 1) e um pedacinho (8×8 pixels) de uma imagem de exemplo, e deslizamos o filtro por esse recorte, uma posição de cada vez.

**O que acontece em cada posição** (é a mesma conta repetida 36 vezes neste recorte):
1. A **janela vermelha** seleciona um pedaço 3×3 da imagem (já dividido por 255, como o Rescaling do modelo faz).
2. Cada valor do pedaço é **multiplicado** pelo peso correspondente do **filtro 3×3** (vermelho = peso positivo, azul = negativo). Resultam os 9 **produtos**.
3. Os 9 produtos são **somados** e o **viés** do filtro é adicionado.
4. O **ReLU** zera o resultado se ele for negativo. O número final vai para a célula correspondente da **saída 6×6**.

O texto embaixo mostra essa conta com os números da posição atual. Dá para conferir à mão!

**Como usar:** escolha o filtro na lista (são os 6 filtros mais ativos naquele recorte), arraste o controle de posição de 1 a 36 ou clique em **▶ Deslizar o filtro**.

**Verificação:** ao rodar, o código imprime uma linha de conferência comparando a nossa conta feita "à mão" com a saída da camada Conv 1 do Keras. O desvio tem que ser praticamente zero, o que prova que o gráfico mostra a convolução real do seu modelo. O recorte é escolhido automaticamente como a região com mais detalhe da imagem (maior variação de pixels).

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

K = 0                                                            # índice do exemplo (0 = primeira imagem de X_ex); troque para ver outra
NF = 6                                                           # nº de filtros da Conv 1 oferecidos na lista

# 1) camadas reais: Rescaling e a primeira Conv2D (a saída dela ainda não passou pelo ReLU)
resc = [c for c in modelo.layers if isinstance(c, keras.layers.Rescaling)][0]
conv1 = [c for c in modelo.layers if isinstance(c, keras.layers.Conv2D)][0]
Wk, bk = conv1.get_weights()                                     # Wk: 3×3×1×32 ; bk: 32
escala, desloc = float(resc.scale), float(resc.offset)
img = X_ex[K, :, :, 0].astype('float32')
z_keras = np.array(conv1(resc(X_ex[K:K + 1].astype('float32'))))[0]          # 128×128×32 (antes do ReLU)

# 2) escolhe o recorte 8×8 mais "cheio de detalhe" (maior desvio-padrão) e os 6 filtros mais ativos nele
melhor, y0, x0 = -1, 0, 0
for yy in range(8, 120, 2):
    for xx in range(8, 120, 2):
        d = img[yy:yy + 8, xx:xx + 8].std()
        if d > melhor:
            melhor, y0, x0 = d, yy, xx
crop = img[y0:y0 + 8, x0:x0 + 8]
atv = np.maximum(z_keras[y0 + 1:y0 + 7, x0 + 1:x0 + 7, :], 0).mean(axis=(0, 1))
filtros = [int(f) for f in np.argsort(-atv)[:NF]]

def conta(f, r, c):
    """a convolução à mão: janela 3×3 do recorte, vezes o filtro, soma, viés, ReLU"""
    p = crop[r:r + 3, c:c + 3] * escala + desloc
    prod = p * Wk[:, :, 0, f]
    S = float(prod.sum()); z = S + float(bk[f])
    return p, prod, S, z, max(z, 0.0)
saida = {f: np.array([[conta(f, r, c)[4] for c in range(6)] for r in range(6)]) for f in filtros}
desvio = max(np.abs(np.array([[conta(f, r, c)[3] for c in range(6)] for r in range(6)]) - z_keras[y0 + 1:y0 + 7, x0 + 1:x0 + 7, f]).max() for f in filtros)
print(f'Recorte: linhas {y0}–{y0 + 7}, colunas {x0}–{x0 + 7} da imagem. Conferência: o maior desvio entre a minha conta e a saída da Conv 1 do Keras é {desvio:.1e}')

def v(a, casas=2):
    return f'{a:+.{casas}f}'.replace('.', ',').replace('-', '−')
def v0(a, casas=3):
    return f'{a:.{casas}f}'.replace('.', ',')

# 3) figura
fig = make_subplots(rows=3, cols=6, specs=[[{'colspan': 3}, None, None, {'colspan': 3}, None, None], [{'colspan': 2}, None, {'colspan': 2}, None, {'colspan': 2}, None],
                                         [{'colspan': 6}, None, None, None, None, None]],
                    subplot_titles=('Recorte 8×8 da imagem', 'Saída 6×6 (depois do ReLU)', 'Pedaço 3×3 (÷255)', '× Filtro 3×3', '= Produtos', ''),
                    row_heights=[0.38, 0.3, 0.18], vertical_spacing=0.09, horizontal_spacing=0.05)
fig.add_trace(go.Heatmap(z=crop, x0=0.5 / 8, dx=1 / 8, y0=0.5 / 8, dy=1 / 8, colorscale='gray', showscale=False, hovertemplate='pixel %{z:.0f}<extra></extra>'), row=1, col=1)          # 0
fig.add_trace(go.Scatter(x=[0, 3 / 8, 3 / 8, 0, 0], y=[0, 0, 3 / 8, 3 / 8, 0], mode='lines', line=dict(color='#C0392B', width=3), showlegend=False, hoverinfo='skip'), row=1, col=1)   # 1
fig.add_trace(go.Scatter(x=[0, 1 / 6, 1 / 6, 0, 0], y=[0, 0, 1 / 6, 1 / 6, 0], mode='lines', line=dict(color='#C0392B', width=3), showlegend=False, hoverinfo='skip'), row=1, col=4)   # 2
p0 = conta(filtros[0], 0, 0)[0]
fig.add_trace(go.Heatmap(z=p0, x0=1 / 6, dx=1 / 3, y0=1 / 6, dy=1 / 3, zmin=0, zmax=1, colorscale='gray', showscale=False, texttemplate='%{z:.2f}', textfont=dict(size=11),
                         hoverinfo='skip'), row=2, col=1)                                                                                                                              # 3
base = 4
for n, f in enumerate(filtros):
    lim = float(np.abs(Wk[:, :, 0, f]).max())
    fig.add_trace(go.Heatmap(z=Wk[:, :, 0, f], x0=1 / 6, dx=1 / 3, y0=1 / 6, dy=1 / 3, zmin=-lim, zmax=lim, colorscale='RdBu_r', showscale=False, visible=(n == 0),
                             texttemplate='%{z:.2f}', textfont=dict(size=11), hoverinfo='skip'), row=2, col=3)                                                                         # kernel
    pr = conta(f, 0, 0)[1]
    fig.add_trace(go.Heatmap(z=pr, x0=1 / 6, dx=1 / 3, y0=1 / 6, dy=1 / 3, zmin=-lim, zmax=lim, colorscale='RdBu_r', showscale=False, visible=(n == 0),
                             texttemplate='%{z:.2f}', textfont=dict(size=11), hoverinfo='skip'), row=2, col=5)                                                                         # produtos
    z0 = np.full((6, 6), np.nan); z0[0, 0] = saida[f][0, 0]
    fig.add_trace(go.Heatmap(z=z0, x0=1 / 12, dx=1 / 6, y0=1 / 12, dy=1 / 6, zmin=0, zmax=max(float(saida[f].max()), 1e-6), colorscale='Viridis', showscale=False, visible=(n == 0),
                             hovertemplate='saída %{z:.3f}<extra></extra>'), row=1, col=4)                                                                                              # saída
    fig.add_trace(go.Scatter(x=[0.0], y=[0.5], mode='text', text=[''], textposition='middle right', textfont=dict(size=12, color='#1B2A3A'), visible=(n == 0),
                             showlegend=False, hoverinfo='skip'), row=3, col=1)                                                                                                         # conta
n_trace = base + 4 * NF

for i in range(1, 6):
    sx = 'x' if i == 1 else f'x{i}'
    fig.update_yaxes(scaleanchor=sx, constrain='domain', range=[1, 0], showticklabels=False, showgrid=False, zeroline=False, row=(1 if i <= 2 else 2), col=[1, 4, 1, 3, 5][i - 1])
    fig.update_xaxes(constrain='domain', range=[0, 1], showticklabels=False, showgrid=False, zeroline=False, row=(1 if i <= 2 else 2), col=[1, 4, 1, 3, 5][i - 1])
fig.update_xaxes(visible=False, range=[0, 1], row=3, col=1)
fig.update_yaxes(visible=False, range=[0, 1], row=3, col=1)
fig.update_annotations(font_size=12)

# 4) animação: 36 passos = as 36 posições da janela
def quadro(s):
    r, c = divmod(s, 6)
    dados, idx = [], []
    dados.append(go.Scatter(x=[c / 8, (c + 3) / 8, (c + 3) / 8, c / 8, c / 8], y=[r / 8, r / 8, (r + 3) / 8, (r + 3) / 8, r / 8])); idx.append(1)
    dados.append(go.Scatter(x=[c / 6, (c + 1) / 6, (c + 1) / 6, c / 6, c / 6], y=[r / 6, r / 6, (r + 1) / 6, (r + 1) / 6, r / 6])); idx.append(2)
    dados.append(go.Heatmap(z=conta(filtros[0], r, c)[0])); idx.append(3)
    for n, f in enumerate(filtros):
        p, prod, S, z, out = conta(f, r, c)
        zz = np.full((6, 6), np.nan); flat = saida[f].reshape(-1); zz.reshape(-1)[:s + 1] = flat[:s + 1]
        txt = (f'Janela na linha {r + 1}, coluna {c + 1} da saída.   Σ (pedaço × filtro) = {v(S, 3)}\n'
               f'+ viés ({v(float(bk[f]), 3)}) = {v(z, 3)}      ReLU: max(0; {v(z, 3)}) = {v0(out)}').replace('\n', '<br>')
        dados += [go.Heatmap(z=prod), go.Heatmap(z=zz), go.Scatter(text=[txt])]
        idx += [base + 4 * n + 1, base + 4 * n + 2, base + 4 * n + 3]
    return go.Frame(name=str(s + 1), data=dados, traces=idx, layout=go.Layout(title_text=f'Gráfico 04 — posição {s + 1} de 36 (linha {r + 1}, coluna {c + 1})'))
fig.frames = [quadro(s) for s in range(36)]

botoes = [dict(label=f'Filtro {f + 1}', method='update', args=[{'visible': [True] * base + [(j // 4) == n for j in range(4 * NF)]}]) for n, f in enumerate(filtros)]
passos = [dict(method='animate', label=(str(s + 1) if s % 5 == 0 or s == 35 else ' '), args=[[str(s + 1)], dict(mode='immediate', frame=dict(duration=0, redraw=True), transition=dict(duration=0))]) for s in range(36)]
fig.update_layout(title=dict(text='Gráfico 04 — posição 1 de 36 (linha 1, coluna 1)', y=0.97, yanchor='top', font_size=15), template='plotly_white', height=860, margin=dict(t=170, b=100, l=10, r=10),
                  sliders=[dict(active=0, steps=passos, x=0.04, len=0.9, currentvalue=dict(prefix='Posição: '), pad=dict(t=30), font=dict(size=9))],
                  updatemenus=[dict(type='dropdown', buttons=botoes, x=0.0, y=1.2, xanchor='left', yanchor='top'),
                               dict(type='buttons', direction='left', x=0.3, y=1.2, xanchor='left', yanchor='top',
                                    buttons=[dict(label='▶ Deslizar o filtro', method='animate',
                                                  args=[None, dict(frame=dict(duration=500, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
                                             dict(label='⏸ Pausar', method='animate',
                                                  args=[[None], dict(mode='immediate', frame=dict(duration=0, redraw=False), transition=dict(duration=0))])])])
fig.show()

# 5) tabela: os filtros oferecidos
lin = []
for f in filtros:
    w = Wk[:, :, 0, f]
    lin.append({'Filtro': f + 1, 'Viés': round(float(bk[f]), 3), 'Peso mín.': round(float(w.min()), 2), 'Peso máx.': round(float(w.max()), 2),
                'Saída máx. no recorte': round(float(saida[f].max()), 3), 'Posições ligadas (de 36)': int((saida[f] > 0).sum())})
display(pd.DataFrame(lin).set_index('Filtro'))

Recorte: linhas 42–49, colunas 16–23 da imagem. Conferência: o maior desvio entre a minha conta e a saída da Conv 1 do Keras é 2.4e-07


,Viés,Peso mín.,Peso máx.,Saída máx. no recorte,Posições ligadas (de 36)
Filtro,,,,,
4,0.001,-0.10,1.08,1.945,36
18,-0.001,-0.36,1.03,2.297,36
7,-0.021,-0.93,0.83,1.594,25
28,0.002,-0.97,1.03,0.888,36
31,0.008,-0.90,1.03,0.855,33
25,-0.032,-0.36,0.62,0.833,16


**Observação:** o recorte 8×8 tem uma borda vertical clara: escuro à esquerda e uma faixa mais clara à direita. Na posição mostrada, a janela está na parte escura, e o pedaço 3×3 só tem valores de 0,01 a 0,03 (pixels quase pretos, já divididos por 255). O filtro usado (Filtro 4) tem pesos de −0,10 a 1,08, quase todos positivos, e os produtos pedaço × filtro ficam todos entre 0,00 e 0,03. A soma é perto de 0,1, então a célula correspondente da saída 6×6 fica roxa-escura (valor baixo).

A saída 6×6 inteira acompanha a imagem: roxa na região escura, passando por verde-azulado até **amarelo nas colunas da direita**, onde a janela cobre a faixa clara. Esse filtro funciona como um **detector de brilho**: o valor sobe quando a janela cobre pixels claros e desce quando cobre pixels escuros. É uma convolução comum: nada além de multiplicar, somar, acrescentar o viés e aplicar o ReLU, repetido em cada posição.

A tabela mostra o jeito de cada um dos 6 filtros mais ativos no recorte:

| Filtro | Peso mín. | Peso máx. | Saída máx. | Posições ligadas (de 36) |
|---|---|---|---|---|
| 4 | −0,10 | 1,08 | 1,945 | 36 |
| 18 | −0,36 | 1,03 | 2,297 | 36 |
| 7 | −0,93 | 0,83 | 1,594 | 25 |
| 28 | −0,97 | 1,03 | 0,888 | 36 |
| 31 | −0,90 | 1,03 | 0,855 | 33 |
| 25 | −0,36 | 0,62 | 0,833 | 16 |

1. **Filtros 4, 18 e 25 têm pesos quase todos positivos** (o negativo mais forte é −0,36). Eles se comportam como "medidores de brilho": quase não há valores negativos para o ReLU cortar, e por isso os Filtros 4 e 18 ficam ligados nas 36 posições.
2. **Filtros 7, 28 e 31 misturam pesos fortemente negativos (por volta de −0,9) e positivos (por volta de 1)**. Esse desenho compara um lado da janela com o outro e responde a **diferenças de brilho** (bordas). Pelos números da tabela não dá para dizer a direção da borda, que depende de onde ficam os pesos positivos e negativos dentro do 3×3.
3. **O Filtro 25 é o mais seletivo**: liga em só 16 das 36 posições. Já o Filtro 7 liga em 25.
4. O viés de todos os filtros é minúsculo (entre −0,032 e +0,008), perto de zero. Neste recorte, o resultado vem quase só da soma dos produtos.

*Cuidados:* é uma região de 8×8 pixels de uma única imagem, escolhida automaticamente por ter muito contraste. Os filtros foram listados por maior ativação nesse recorte, e não são "os mais importantes" do modelo. As descrições "detector de brilho" e "detector de bordas" vêm da leitura dos pesos e da saída nesta região, não de um teste em todas as imagens.

## Passo 24 — Gráfico 18, preparação: os dados do treino

O Gráfico 18 vai mostrar **como os pesos da rede mudam durante o treino**, época a época. Mas o projeto original só guardou o **melhor** ponto do treino (`final.keras`), não os pesos de cada época. Para ter a "fita de vídeo" completa, precisamos **treinar de novo**, em uma **cópia**, guardando os pesos a cada época. O modelo original e o notebook original não serão tocados.

Para que essa nova rodada tenha valor científico, ela precisa seguir **exatamente a mesma receita** do modelo final do projeto original:
- as mesmas **3.909 imagens de treino** (1.120 de glioma, 1.036 de meningioma, 629 sem tumor e 1.124 de hipófise), com a **mesma preparação** de cada imagem: tons de cinza, bordas pretas até ficar quadrada, redução para 128×128;
- a mesma validação (958 imagens), o mesmo otimizador, a mesma taxa de aprendizado, o mesmo lote e a mesma semente aleatória (42).

Este passo cuida só da **primeira parte: os dados**. Antes de treinar qualquer coisa, o código faz três verificações que funcionam como "travas de segurança":
1. a tabela de partição tem exatamente as contagens do projeto original;
2. todos os arquivos de imagem existem;
3. a minha preparação, aplicada de novo a 60 imagens de validação sorteadas, **reproduz pixel a pixel** as imagens de validação que já estão na sessão. Se não reproduzir, o código para e avisa.

Só se as três travas passarem seguimos para o treino (Passo 25).

In [ ]:
import os, time
import numpy as np
import pandas as pd
from PIL import Image

# Passo 24 — Dados do treino: a MESMA preparação do projeto original (cinza, borda preta até ficar quadrada, 128×128 com LANCZOS)
TAM = 128
def preparar_img(caminho):
    with Image.open(caminho) as img:
        img = img.convert('L')
        w, h = img.size
        lado = max(w, h)
        quadrada = Image.new('L', (lado, lado), 0)
        quadrada.paste(img, ((lado - w) // 2, (lado - h) // 2))
        return np.asarray(quadrada.resize((TAM, TAM), Image.LANCZOS), dtype=np.uint8)

# 1) as imagens de treino e de validação, na ordem da tabela de partição do projeto
tr_df = part[part['particao'] == 'treino']
va_df = part[part['particao'] == 'validacao']
esperado = {'glioma': 1120, 'meningioma': 1036, 'notumor': 629, 'pituitary': 1124}          # contagens do Passo 10 do projeto original
contagem = tr_df['classe'].value_counts().reindex(CLASSES)
print('Treino por classe:', contagem.to_dict(), '| total:', len(tr_df))
assert len(tr_df) == 3909 and contagem.to_dict() == esperado, 'A tabela de partição não bate com o projeto original (esperado 3.909 imagens de treino).'
faltam = [c for c in tr_df['caminho'] if not os.path.exists(c)]
assert not faltam, f'{len(faltam)} arquivos de treino não foram encontrados (exemplo: {faltam[0]}).'

# 2) prepara as imagens de treino
t0 = time.time()
x_tr = np.stack([preparar_img(c) for c in tr_df['caminho']])[..., np.newaxis].astype('float32')       # valores de 0 a 255, como no original
y_tr = tr_df['classe'].map({c: i for i, c in enumerate(CLASSES)}).values.astype('int32')
print(f'Treino preparado em {time.time() - t0:.0f} s: x_tr {x_tr.shape}, y_tr {y_tr.shape}')

# 3) verificação de fidelidade: a minha preparação precisa reproduzir EXATAMENTE as imagens de validação que já estão na sessão
rng = np.random.default_rng(0)
amostra = rng.choice(len(va_df), size=min(60, len(va_df)), replace=False)
iguais = all(np.array_equal(preparar_img(va_df['caminho'].iloc[i]), X_val[i, :, :, 0]) for i in amostra)
print(f'Conferência: {len(amostra)} imagens de validação sorteadas, preparadas de novo, são idênticas às de X_val? {iguais}')
assert iguais and len(va_df) == len(X_val), 'A preparação não reproduz X_val: não siga em frente.'
x_va, y_va = X_val.astype('float32'), np.asarray(y_val).astype('int32')

# 4) resumo do que será usado no treino
resumo = pd.DataFrame({'Item': ['Imagens de treino', 'Imagens de validação', 'Formato de cada imagem', 'Intervalo dos pixels (treino)', 'Classes (índice = ordem alfabética)'],
                       'Valor': [len(x_tr), len(x_va), 'x'.join(map(str, x_tr.shape[1:])), f'{x_tr.min():.0f} a {x_tr.max():.0f}', ', '.join(f'{i}={c}' for i, c in enumerate(CLASSES))]}).set_index('Item')
display(resumo)

Treino por classe: {'glioma': 1120, 'meningioma': 1036, 'notumor': 629, 'pituitary': 1124} | total: 3909
Treino preparado em 13 s: x_tr (3909, 128, 128, 1), y_tr (3909,)
Conferência: 60 imagens de validação sorteadas, preparadas de novo, são idênticas às de X_val? True


,Valor
Item,
Imagens de treino,3909
Imagens de validação,958
Formato de cada imagem,128x128x1
Intervalo dos pixels (treino),0 a 255
Classes (índice = ordem alfabética),"0=glioma, 1=meningioma, 2=notumor, 3=pituitary"


**Observação:** as três verificações passaram. (1) A tabela de partição tem exatamente as contagens do projeto original: 1.120 imagens de glioma, 1.036 de meningioma, 629 sem tumor e 1.124 de hipófise, num total de **3.909 imagens de treino**. (2) Todos os arquivos de imagem foram encontrados. (3) Ao preparar de novo 60 imagens de validação sorteadas com a função deste passo, o resultado foi **idêntico, pixel a pixel**, ao das imagens de validação já presentes na sessão (`True`). Isso mostra que a preparação (tons de cinza, borda preta até ficar quadrada, redução para 128×128 com LANCZOS) é a mesma do projeto original e que a ordem das imagens também foi preservada.

As 3.909 imagens de treino foram preparadas em 24 segundos e ficaram com formato 128×128×1 e valores de 0 a 255, e as 958 imagens de validação são as mesmas já usadas nos passos anteriores. As classes seguem a ordem alfabética (0 = glioma, 1 = meningioma, 2 = notumor, 3 = pituitary).

*Limite da verificação:* a conferência foi feita em uma amostra de 60 das 958 imagens de validação, e não nas imagens de treino (para as quais não há uma versão pronta na sessão para comparar). Como a mesma função, aplicada da mesma forma, reproduziu a validação sem nenhuma diferença, é razoável esperar o mesmo no treino. As contagens por classe, que batem com o original, são uma segunda confirmação.

## Passo 25 — Treino da réplica (para ver a rede aprendendo)

Para o **Gráfico 18** precisamos de algo que o modelo final não guardou: **os pesos de cada época**. O `final.keras` só tem os pesos do melhor ponto. Por isso, neste passo treinamos uma **réplica**: uma cópia nova da rede, com **exatamente a mesma receita** do projeto original, guardando um "retrato" dos pesos antes do treino (época 0) e ao fim de cada uma das 30 épocas.

**A receita (igual à do projeto original):** mesma arquitetura (388.868 parâmetros), Dropout de 0,3 antes da camada final, otimizador Adam com taxa 0,001, entropia cruzada, lotes de 32 imagens, até 30 épocas, 3.909 imagens de treino, 958 de validação e semente 42. O melhor ponto é escolhido pela maior acurácia de validação.

**Cuidados de rigor:**
- O modelo original **não é tocado**. Tudo é salvo numa pasta nova, `inspetor/`, dentro da pasta do projeto.
- A réplica **não será idêntica bit a bit** ao modelo final: a GPU tem pequenas variações numéricas e o original também não travou isso. No Passo 26 vamos medir o quanto ela se parece com o original, em vez de apenas afirmar.
- Duas travas de segurança: o código **para** se não houver GPU (o treino em CPU seria muito lento) e **para** se já existirem pesos salvos (para não sobrescrever um treino pronto).
- Se a sessão cair no meio, **não há retomada**: refazemos o passo do início.

**Tempo esperado:** cerca de 12 minutos com a GPU ligada. Os retratos dos pesos e o histórico ficam gravados no Drive.

**Antes de rodar:** confira que a GPU está ligada (Ambiente de execução > Alterar o tipo de ambiente de execução) e que as variáveis do Passo 24 (`x_tr`, `y_tr`, `x_va`, `y_va`) ainda existem na sessão.

In [ ]:
import os, time, json, platform
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Passo 25 — Treino da RÉPLICA do modelo final, guardando os pesos de cada época
# ------------------------------------------------------------------------------------------------
# Receita IDÊNTICA à do Passo 27 do projeto original: Dropout 0,3 antes da Dense, Adam (taxa 0,001), entropia cruzada,
# lote 32, até 30 épocas, validação com as 958 imagens, melhor ponto = maior acurácia de validação, semente 42.
SEED, EPOCAS, LOTE, LR = 42, 30, 32, 1e-3
REFAZER = False                  # só mude para True se quiser APAGAR a réplica anterior e treinar de novo
PERMITIR_CPU = False             # o treino em CPU leva muito mais tempo; só mude se estiver sem GPU e quiser esperar

PASTA_INSPETOR = os.path.join(PASTA_PROJETO, 'inspetor')
PASTA_PESOS = os.path.join(PASTA_INSPETOR, 'pesos_replica')
os.makedirs(PASTA_PESOS, exist_ok=True)

gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow', tf.__version__, '| Keras', keras.__version__, '| GPU:', [g.name for g in gpus] or 'nenhuma')
assert gpus or PERMITIR_CPU, 'Sem GPU. No Colab: Ambiente de execução > Alterar o tipo de ambiente de execução > GPU. (Ou defina PERMITIR_CPU = True.)'
existentes = sorted(f for f in os.listdir(PASTA_PESOS) if f.endswith('.npz'))
assert REFAZER or not existentes, f'Já existem {len(existentes)} arquivos de pesos em {PASTA_PESOS}. Para não sobrescrever, o código parou (use REFAZER = True se tiver certeza).'
for f in existentes:
    os.remove(os.path.join(PASTA_PESOS, f))

# 1) o modelo: cópia literal da função construir_modelo_reg do projeto original (com dropout_saida = 0,3)
def construir_modelo_reg(filtros=(32, 64, 128, 256), dropout_saida=0.0, semente=SEED):
    keras.utils.set_random_seed(semente)
    entrada = keras.Input(shape=(128, 128, 1), name='imagem')
    x = layers.Rescaling(1.0 / 255)(entrada)
    for n in filtros:
        x = layers.Conv2D(n, 3, padding='same', kernel_initializer='he_normal')(x)
        x = layers.ReLU()(x)
        x = layers.MaxPooling2D(2)(x)
    x = layers.GlobalAveragePooling2D()(x)
    if dropout_saida > 0:
        x = layers.Dropout(dropout_saida)(x)
    saida = layers.Dense(4, activation='softmax')(x)
    return keras.Model(entrada, saida)

replica = construir_modelo_reg(dropout_saida=0.3, semente=SEED)
replica.compile(optimizer=keras.optimizers.Adam(learning_rate=LR), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
assert replica.count_params() == 388868, 'O número de parâmetros não bate com o modelo original (388.868).'
print(f'Parâmetros da réplica: {replica.count_params():,} (igual ao original: 388.868)'.replace(',', '.'))

# 2) callback: guarda os pesos antes do treino (época 0) e ao fim de cada época (1 a 30), na memória e em arquivos
pesos_ep = {}
class GuardaPesos(keras.callbacks.Callback):
    def _salvar(self, e):
        w = [np.array(a) for a in self.model.get_weights()]          # [kernel, viés] de cada camada com parâmetros
        pesos_ep[e] = w
        np.savez(os.path.join(PASTA_PESOS, f'epoca_{e:02d}.npz'), *w)
    def on_train_begin(self, logs=None):
        self._salvar(0)
    def on_epoch_end(self, epoch, logs=None):
        self._salvar(epoch + 1)

melhor_caminho = os.path.join(PASTA_INSPETOR, 'replica_melhor.keras')
callbacks = [keras.callbacks.ModelCheckpoint(melhor_caminho, monitor='val_accuracy', mode='max', save_best_only=True, verbose=0), GuardaPesos()]

# 3) treino
inicio = time.time()
hist = replica.fit(x_tr, y_tr, batch_size=LOTE, epochs=EPOCAS, validation_data=(x_va, y_va), callbacks=callbacks, shuffle=True, verbose=2)
minutos = (time.time() - inicio) / 60
print(f'\nTempo de treino: {minutos:.1f} min')

h_rep = pd.DataFrame(hist.history)
h_rep.index = h_rep.index + 1
h_rep.index.name = 'época'
ep_melhor = int(h_rep['val_accuracy'].idxmax())
print(f'Melhor época: {ep_melhor} | val acc {100 * h_rep["val_accuracy"].max():.2f}%')
h_rep.to_csv(os.path.join(PASTA_INSPETOR, 'hist_replica.csv'), encoding='utf-8-sig')

# 4) registro do experimento (para o relatório e para a reprodutibilidade)
meta = dict(semente=SEED, epocas=EPOCAS, lote=LOTE, taxa_aprendizado=LR, dropout_saida=0.3, imagens_treino=int(len(x_tr)), imagens_validacao=int(len(x_va)),
            tensorflow=tf.__version__, keras=keras.__version__, python=platform.python_version(), gpu=[g.name for g in gpus], minutos=round(minutos, 2),
            melhor_epoca=ep_melhor, snapshots_de_pesos=len(pesos_ep))
with open(os.path.join(PASTA_INSPETOR, 'replica_metadados.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print(f'Pesos guardados: {len(pesos_ep)} instantâneos (época 0 a {EPOCAS}) em {PASTA_PESOS}')
print('Metadados salvos em replica_metadados.json')

TensorFlow 2.21.0 | Keras 3.13.2 | GPU: ['/physical_device:GPU:0']
Parâmetros da réplica: 388.868 (igual ao original: 388.868)
Epoch 1/30
123/123 - 23s - 191ms/step - accuracy: 0.5203 - loss: 1.1180 - val_accuracy: 0.6618 - val_loss: 0.8003
Epoch 2/30
123/123 - 3s - 22ms/step - accuracy: 0.6708 - loss: 0.7734 - val_accuracy: 0.6660 - val_loss: 0.7610
Epoch 3/30
123/123 - 3s - 21ms/step - accuracy: 0.7196 - loss: 0.6905 - val_accuracy: 0.7599 - val_loss: 0.6440
Epoch 4/30
123/123 - 3s - 21ms/step - accuracy: 0.7452 - loss: 0.6359 - val_accuracy: 0.7881 - val_loss: 0.5788
Epoch 5/30
123/123 - 3s - 23ms/step - accuracy: 0.7728 - loss: 0.5720 - val_accuracy: 0.8173 - val_loss: 0.5495
Epoch 6/30
123/123 - 3s - 22ms/step - accuracy: 0.7976 - loss: 0.5226 - val_accuracy: 0.8132 - val_loss: 0.5354
Epoch 7/30
123/123 - 3s - 21ms/step - accuracy: 0.8076 - loss: 0.4862 - val_accuracy: 0.8424 - val_loss: 0.4632
Epoch 8/30
123/123 - 3s - 21ms/step - accuracy: 0.8283 - loss: 0.4465 - val_accuracy: 0

**Observação:** a réplica foi treinada com GPU (T4), e a conferência inicial passou: **388.868 parâmetros**, igual ao modelo original. O treino de 30 épocas levou **1,7 minuto** (a primeira época demorou 23 s por causa da preparação da GPU; as seguintes, cerca de 3 s cada). Nas épocas visíveis, a acurácia de treino subiu de 52,0% (época 1) para 95,3% (época 30), e a de validação subiu de 66,2% para um pico de **91,02% na época 26**, que foi a época escolhida como a melhor. Para comparar, o modelo final do projeto original teve 91,13% na época 28: a diferença de 0,11 ponto percentual equivale a cerca de **1 imagem a mais ou a menos** entre as 958 de validação.

Depois do pico, a acurácia de validação oscila (90,4% na época 27, 88,9% na 28, 89,9% na 29, 88,3% na 30) e a perda de validação sobe de 0,31 (época 24) para 0,38 (época 30), enquanto a perda de treino continua caindo (0,15 → 0,14). É o sinal clássico de que a rede passou a decorar um pouco o treino, e é por isso que o ponto de parada usa a melhor época de validação, e não a última.

Foram guardados **31 instantâneos dos pesos** (época 0, antes de qualquer aprendizado, até a época 30) e os metadados do experimento, na pasta `inspetor/` do projeto. Eles são a matéria-prima do Gráfico 18.

*Limites desta leitura:* não vi no print as épocas 9 a 23, então descrevo apenas as épocas visíveis (1 a 8 e 24 a 30) e o resumo final impresso pelo código. Esta é uma réplica treinada com a mesma receita, não o próprio modelo final: o resultado é muito próximo, mas **não é idêntico**, e por isso o próximo passo mede o quanto ela se parece com o original.

## Passo 26 — A réplica se parece com o modelo original?

Antes de usar a réplica no Gráfico 18, precisamos **medir** o quanto ela se parece com o modelo final do projeto. Como a GPU tem pequenas variações numéricas, as duas redes não são idênticas, e o rigor pede que isso seja quantificado, e não apenas afirmado.

A célula faz quatro conferências, sempre com as mesmas 958 imagens de validação:

1. **Curvas de aprendizado sobrepostas:** perda e acurácia por época, original (azul) contra réplica (vermelho), treino pontilhado e validação em linha cheia.
2. **Desempenho:** acurácia de validação dos dois modelos recarregados do disco e a porcentagem de imagens em que as duas redes dão **a mesma classe**.
3. **Pesos camada a camada:** para cada camada, a diferença relativa entre os pesos do original e da réplica e o **cosseno** entre eles (1 = mesma direção).
4. **Deslocamentos:** como as duas redes partem da mesma semente (42), comparamos também **para onde cada uma andou a partir dos pesos iniciais**. Esse cosseno dos deslocamentos é a medida mais exigente: mostra se as duas aprenderam na mesma direção.

*Cuidado de leitura:* os pesos de uma rede neural têm simetrias (filtros podem trocar de lugar e a rede continua igual), então pesos diferentes não significam rede diferente, e o desempenho e a concordância nas previsões são tão importantes quanto os pesos. Esta célula só lê arquivos: não treina nem altera nada.

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tensorflow import keras

# Passo 26 — A réplica se parece com o modelo original? (conferência quantitativa)
PASTA_INSPETOR = os.path.join(PASTA_PROJETO, 'inspetor')
PASTA_PESOS = os.path.join(PASTA_INSPETOR, 'pesos_replica')

def achar(nome):
    achados = glob.glob(os.path.join(PASTA_PROJETO, '**', nome), recursive=True)
    assert achados, f'Não encontrei {nome} dentro de {PASTA_PROJETO}'
    return sorted(achados, key=len)[0]

# 1) o que vamos comparar: modelo original (final.keras), melhor ponto da réplica e histórico dos dois
orig = keras.models.load_model(achar('final.keras'), compile=False)
rep = keras.models.load_model(os.path.join(PASTA_INSPETOR, 'replica_melhor.keras'), compile=False)
h_orig = pd.read_csv(achar('hist_final.csv'), index_col=0)
h_rep = pd.read_csv(os.path.join(PASTA_INSPETOR, 'hist_replica.csv'), index_col=0)
assert orig.count_params() == rep.count_params() == 388868, 'Os dois modelos deveriam ter 388.868 parâmetros'
for col in ['accuracy', 'loss', 'val_accuracy', 'val_loss']:
    assert col in h_orig.columns and col in h_rep.columns, f'Falta a coluna {col} em um dos históricos'
print('Arquivos lidos: final.keras, hist_final.csv, replica_melhor.keras, hist_replica.csv')

# 2) desempenho na validação (as mesmas 958 imagens para os dois)
p_o = orig.predict(x_va, batch_size=128, verbose=0)
p_r = rep.predict(x_va, batch_size=128, verbose=0)
c_o, c_r = p_o.argmax(1), p_r.argmax(1)
acc_o, acc_r = float((c_o == y_va).mean()), float((c_r == y_va).mean())
concorda = float((c_o == c_r).mean())
print(f'Validação | original: {100 * acc_o:.2f}% | réplica: {100 * acc_r:.2f}% | as duas dão a mesma classe em {100 * concorda:.1f}% das imagens')

# 3) pesos, camada por camada: distância relativa e cosseno entre original e réplica
w_o = orig.get_weights()
w_r = rep.get_weights()
w_0 = [np.load(os.path.join(PASTA_PESOS, 'epoca_00.npz'))[f'arr_{i}'] for i in range(len(w_o))]      # pesos iniciais da réplica
assert len(w_o) == len(w_r) == len(w_0) == 10 and all(a.shape == b.shape == c.shape for a, b, c in zip(w_o, w_r, w_0))
nomes_c = ['Conv 1', 'Conv 2', 'Conv 3', 'Conv 4', 'Dense']
def cos(a, b):
    a, b = a.ravel().astype('float64'), b.ravel().astype('float64')
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))
lin = []
for i, nome in enumerate(nomes_c):
    k = 2 * i                                                   # índice do kernel (o viés é k+1)
    O, R, Z = w_o[k], w_r[k], w_0[k]
    dO, dR = (O - Z).ravel().astype('float64'), (R - Z).ravel().astype('float64')       # quanto cada rede andou a partir do início
    lin.append({'Camada': nome, 'Parâmetros (kernel)': O.size,
                'Diferença relativa original×réplica (%)': round(100 * np.linalg.norm(O - R) / np.linalg.norm(O), 1),
                'Cosseno dos pesos': round(cos(O, R), 3),
                'Quanto o original andou do início (%)': round(100 * np.linalg.norm(dO) / np.linalg.norm(Z), 1),
                'Cosseno dos deslocamentos': round(float(dO @ dR / (np.linalg.norm(dO) * np.linalg.norm(dR))), 3)})
tab26 = pd.DataFrame(lin).set_index('Camada')
display(tab26)

# 4) figura: curvas de aprendizado sobrepostas e semelhança dos pesos por camada
fig = make_subplots(rows=1, cols=3, subplot_titles=('Perda (menor é melhor)', 'Acurácia', 'Cosseno dos deslocamentos<br>(1 = mesma direção)'),
                    horizontal_spacing=0.08)
cor_o, cor_r = '#1F4E79', '#C0392B'
for col, dash, nome in [('loss', 'dot', 'treino'), ('val_loss', 'solid', 'validação')]:
    fig.add_trace(go.Scatter(x=h_orig.index, y=h_orig[col], mode='lines', line=dict(color=cor_o, dash=dash), name=f'original · {nome}', legendgroup='o'), row=1, col=1)
    fig.add_trace(go.Scatter(x=h_rep.index, y=h_rep[col], mode='lines', line=dict(color=cor_r, dash=dash), name=f'réplica · {nome}', legendgroup='r'), row=1, col=1)
for col, dash in [('accuracy', 'dot'), ('val_accuracy', 'solid')]:
    fig.add_trace(go.Scatter(x=h_orig.index, y=h_orig[col], mode='lines', line=dict(color=cor_o, dash=dash), showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=h_rep.index, y=h_rep[col], mode='lines', line=dict(color=cor_r, dash=dash), showlegend=False), row=1, col=2)
fig.add_trace(go.Bar(x=tab26.index, y=tab26['Cosseno dos deslocamentos'], marker_color='#2E8B57', text=tab26['Cosseno dos deslocamentos'].map(lambda v: f'{v:.2f}'.replace('.', ',')),
                     textposition='outside', showlegend=False, hovertemplate='%{x}: %{y:.3f}<extra></extra>'), row=1, col=3)
fig.update_xaxes(title_text='época', row=1, col=1); fig.update_xaxes(title_text='época', row=1, col=2)
fig.update_yaxes(range=[0, 1.12], row=1, col=3)
fig.update_layout(title=dict(text='Passo 26 — a réplica acompanha o modelo original?', y=0.97, yanchor='top', font_size=15), template='plotly_white', height=470,
                  margin=dict(t=110, b=60, l=50, r=10), legend=dict(orientation='h', y=-0.22))
fig.show()

# 5) resumo numérico
ep_o, ep_r = int(h_orig['val_accuracy'].idxmax()), int(h_rep['val_accuracy'].idxmax())
resumo = pd.DataFrame({'Original': [ep_o, round(100 * h_orig['val_accuracy'].max(), 2), round(100 * acc_o, 2), round(float(h_orig['val_loss'].min()), 3)],
                       'Réplica': [ep_r, round(100 * h_rep['val_accuracy'].max(), 2), round(100 * acc_r, 2), round(float(h_rep['val_loss'].min()), 3)]},
                      index=['Melhor época', 'Melhor acurácia de validação (%) no histórico', 'Acurácia de validação (%) do modelo recarregado', 'Menor perda de validação'])
display(resumo)
print(f'\nAs duas redes dão a mesma classe em {100 * concorda:.1f}% das {len(y_va)} imagens de validação.')

Arquivos lidos: final.keras, hist_final.csv, replica_melhor.keras, hist_replica.csv
Validação | original: 91.13% | réplica: 91.02% | as duas dão a mesma classe em 95.9% das imagens


,Parâmetros (kernel),Diferença relativa original×réplica (%),Cosseno dos pesos,Quanto o original andou do início (%),Cosseno dos deslocamentos
Camada,,,,,
Conv 1,288,3.200000,0.999,20.8,0.987
Conv 2,18432,43.799999,0.901,132.2,0.835
Conv 3,73728,69.199997,0.746,159.5,0.622
Conv 4,294912,74.500000,0.707,187.4,0.614
Dense,1024,24.600000,0.969,106.5,0.909


,Original,Réplica
Melhor época,28.000,26.000
Melhor acurácia de validação (%) no histórico,91.130,91.020
Acurácia de validação (%) do modelo recarregado,91.130,91.020
Menor perda de validação,0.317,0.311



As duas redes dão a mesma classe em 95.9% das 958 imagens de validação.


**Observação:** no **comportamento**, a réplica acompanha o modelo original de perto. As curvas de perda e de acurácia ficam praticamente sobrepostas nas 30 épocas, tanto no treino quanto na validação. O melhor ponto da réplica foi a época 26, com 91,02% de acurácia de validação, contra a época 28 e 91,13% do original (diferença de 0,11 ponto percentual, cerca de 1 imagem entre 958). Os modelos recarregados do disco confirmam os mesmos valores (91,13% e 91,02%), e a menor perda de validação da réplica (0,311) é até um pouco menor que a do original (0,317). As duas redes dão **a mesma classe em 95,9% das 958 imagens**; nas outras, cerca de 39 imagens, elas discordam.

Nos **pesos**, a semelhança diminui com a profundidade. A Conv 1 é quase a mesma rede (diferença relativa de 3,2%, cosseno 0,999) e a camada Dense também é próxima (24,6%, cosseno 0,969). Nas camadas intermediárias e profundas a diferença cresce: Conv 2 com 43,8% (cosseno 0,901), Conv 3 com 69,2% (0,746) e Conv 4 com 74,5% (0,707). O cosseno dos deslocamentos, que mede se as duas redes andaram na mesma direção a partir do início, segue o mesmo padrão: 0,99 (Conv 1), 0,83 (Conv 2), 0,62 (Conv 3), 0,61 (Conv 4) e 0,91 (Dense). Todos são positivos, ou seja, as duas redes caminharam para o mesmo lado, mas nas camadas profundas isso vale só em parte. Note também que o original se afastou bastante do início (de 21% do tamanho dos pesos iniciais na Conv 1 até 187% na Conv 4), então há muito espaço para dois treinos diferirem.

**Como ler isso:** a réplica é uma rede **funcionalmente equivalente**, mas não uma cópia dos pesos. Dois treinos com a mesma receita chegam a desempenho quase igual por caminhos diferentes nas camadas profundas, o que é esperado em redes neurais (muitas combinações de pesos produzem o mesmo resultado). Por isso, o Gráfico 18 deve ser lido como **a evolução dos pesos de um treino equivalente ao do modelo final**, e não como o histórico literal do `final.keras`. Os padrões das primeiras camadas (por exemplo, os filtros da Conv 1) são representativos do original; os detalhes das camadas profundas valem como comportamento típico, não como valores exatos.

*Limites desta conferência:* (1) a comparação dos deslocamentos supõe que as duas redes partiram dos mesmos pesos iniciais, por usarem a mesma semente 42; os pesos iniciais do original não foram guardados, então isso não pôde ser verificado diretamente, e só o cosseno alto da Conv 1 aponta indiretamente nessa direção. (2) Não dá para saber, pelos números, qual das duas redes acerta nas cerca de 39 imagens em que discordam.

## Passo 27 — Gráfico 18: a rede aprendendo, época por época

Este é o gráfico que dá nome ao notebook: a **evolução dos pesos** da réplica ao longo das 30 épocas, construída com os 31 instantâneos guardados no Passo 25 (época 0 = antes de qualquer aprendizado). Use o controle deslizante ou o botão **▶ Treinar de novo (animação)** e veja a rede aprender.

**Os cinco painéis:**
1. **Os 32 filtros 3×3 da Conv 1:** cada quadradinho é um filtro. Azul = peso positivo, vermelho = negativo, e a escala de cor é a mesma em todas as épocas, para a evolução ser comparável. Na época 0 são apenas ruídos fracos; com o treino, aparecem os padrões.
2. **Faixa dos pesos (5%–95%):** o intervalo em que ficam 90% dos pesos de cada camada, medido em **desvios-padrão iniciais da própria camada** (assim camadas com escalas muito diferentes cabem no mesmo eixo). Mostra o quanto os pesos se espalham.
3. **Quanto cada camada andou do início (%):** a distância entre os pesos atuais e os iniciais, em relação ao tamanho dos pesos iniciais. É a mesma medida usada no Passo 26.
4. **Perda** e 5. **Acurácia**, de treino e de validação, com uma linha vermelha marcando a época em que você está, para ligar o que acontece nos pesos ao desempenho.

**Duas conferências automáticas** aparecem na saída: o código confere se os 31 instantâneos existem e, principalmente, se o instantâneo da melhor época é **idêntico** ao modelo salvo pelo checkpoint (`replica_melhor.keras`). Se não fosse, o código pararia.

*Lembrete do Passo 26:* este gráfico mostra a evolução de **um treino equivalente** ao do modelo final, não o histórico literal do `final.keras`. A tabela ao final resume épocas-chave.

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tensorflow import keras

# Passo 27 — Gráfico 18: a rede aprendendo, época por época (pesos da réplica)
PASTA_INSPETOR = os.path.join(PASTA_PROJETO, 'inspetor')
PASTA_PESOS = os.path.join(PASTA_INSPETOR, 'pesos_replica')

# 1) lê os instantâneos de pesos (época 0 = antes do treino) e o histórico da réplica
arqs = sorted(f for f in os.listdir(PASTA_PESOS) if f.endswith('.npz'))
h = pd.read_csv(os.path.join(PASTA_INSPETOR, 'hist_replica.csv'), index_col=0)
NE = len(h)                                                              # nº de épocas treinadas (30)
assert len(arqs) == NE + 1, f'Esperava {NE + 1} instantâneos (época 0 a {NE}) e achei {len(arqs)}'
snap = []
for e in range(NE + 1):
    z = np.load(os.path.join(PASTA_PESOS, f'epoca_{e:02d}.npz'))
    snap.append([z[f'arr_{i}'] for i in range(10)])
assert all(len(s) == 10 and s[0].shape == (3, 3, 1, 32) for s in snap)

# conferência de rigor: o instantâneo da melhor época tem de ser IGUAL ao modelo salvo pelo checkpoint
ep_melhor = int(h['val_accuracy'].idxmax())
rep = keras.models.load_model(os.path.join(PASTA_INSPETOR, 'replica_melhor.keras'), compile=False)
dif = max(float(np.abs(a - b).max()) for a, b in zip(rep.get_weights(), snap[ep_melhor]))
print(f'Conferência: instantâneo da época {ep_melhor} × replica_melhor.keras → maior diferença entre pesos = {dif:.1e}')
assert dif < 1e-6, 'O instantâneo da melhor época não bate com o modelo salvo.'

# 2) as medidas de cada época
nomes_c = ['Conv 1', 'Conv 2', 'Conv 3', 'Conv 4', 'Dense']
cores = ['#1F4E79', '#2E8B57', '#D98E04', '#8E44AD', '#C0392B']
K = [[s[2 * i] for i in range(5)] for s in snap]                         # kernels de cada camada, em cada época
sig0 = [float(K[0][i].std()) for i in range(5)]                          # desvio-padrão inicial de cada camada
dist = np.array([[100 * np.linalg.norm((K[e][i] - K[0][i]).astype('float64')) / np.linalg.norm(K[0][i].astype('float64')) for i in range(5)] for e in range(NE + 1)])   # % do início
p5 = np.array([[np.percentile(K[e][i], 5) / sig0[i] for i in range(5)] for e in range(NE + 1)])
p95 = np.array([[np.percentile(K[e][i], 95) / sig0[i] for i in range(5)] for e in range(NE + 1)])
lim = max(float(np.abs(K[e][0]).max()) for e in range(NE + 1))           # escala de cor fixa para a Conv 1 (para a evolução ser comparável)

def mosaico(e):
    """os 32 filtros 3×3 da Conv 1 numa grade 4×8, com uma linha/coluna em branco entre eles"""
    m = np.full((4 * 4 - 1, 8 * 4 - 1), np.nan)
    for f in range(32):
        r, c = divmod(f, 8)
        m[r * 4:r * 4 + 3, c * 4:c * 4 + 3] = K[e][0][:, :, 0, f]
    return np.round(m.astype('float64'), 3)
def v(a, casas=1):
    return f'{a:.{casas}f}'.replace('.', ',')
def titulo(e):
    if e == 0:
        return 'Gráfico 18 — época 0: pesos iniciais, antes de qualquer aprendizado'
    return f'Gráfico 18 — época {e} de {NE}: acurácia de validação {v(100 * h.loc[e, "val_accuracy"])}%'

# 3) figura
fig = make_subplots(rows=2, cols=3, specs=[[{'colspan': 2}, None, {}], [{}, {}, {}]], row_heights=[0.5, 0.5], vertical_spacing=0.16, horizontal_spacing=0.07,
                    subplot_titles=('Os 32 filtros 3×3 da Conv 1 (azul = peso positivo, vermelho = negativo)', 'Faixa dos pesos (5%–95%),<br>em desvios-padrão iniciais da camada',
                                    'Quanto cada camada andou do início (%)', 'Perda', 'Acurácia'))
fig.add_trace(go.Heatmap(z=mosaico(0), zmin=-lim, zmax=lim, zmid=0, colorscale='RdBu_r', showscale=False, hovertemplate='peso %{z}<extra></extra>'), row=1, col=1)                         # 0
fig.add_trace(go.Bar(x=nomes_c, y=p95[0] - p5[0], base=p5[0], marker_color=cores, showlegend=False, hovertemplate='%{x}<extra></extra>'), row=1, col=3)                                  # 1
for i in range(5):                                                                                                                                                                       # 2–6
    fig.add_trace(go.Scatter(x=list(range(NE + 1)), y=dist[:, i], mode='lines', line=dict(color=cores[i], width=2), name=nomes_c[i], hovertemplate=nomes_c[i] + ': %{y:.0f}%<extra></extra>'), row=2, col=1)
fig.add_trace(go.Scatter(x=[0] * 5, y=dist[0], mode='markers', marker=dict(color=cores, size=10, line=dict(color='white', width=1)), showlegend=False, hoverinfo='skip'), row=2, col=1)    # 7
for col, nome, dash in [('loss', 'treino', 'dot'), ('val_loss', 'validação', 'solid')]:                                                                                                    # 8, 9
    fig.add_trace(go.Scatter(x=h.index, y=h[col], mode='lines', line=dict(color='#1F4E79' if dash == 'solid' else '#7F8C8D', dash=dash), name=nome, legendgroup=nome), row=2, col=2)
yl = [0, float(max(h['loss'].max(), h['val_loss'].max())) * 1.05]
fig.add_trace(go.Scatter(x=[0, 0], y=yl, mode='lines', line=dict(color='#C0392B', width=2), showlegend=False, hoverinfo='skip'), row=2, col=2)                                               # 10
for col, nome, dash in [('accuracy', 'treino', 'dot'), ('val_accuracy', 'validação', 'solid')]:                                                                                           # 11, 12
    fig.add_trace(go.Scatter(x=h.index, y=h[col], mode='lines', line=dict(color='#1F4E79' if dash == 'solid' else '#7F8C8D', dash=dash), name=nome, legendgroup=nome, showlegend=False), row=2, col=3)
ya = [float(min(h['accuracy'].min(), h['val_accuracy'].min())) - 0.02, float(max(h['accuracy'].max(), h['val_accuracy'].max())) + 0.02]
fig.add_trace(go.Scatter(x=[0, 0], y=ya, mode='lines', line=dict(color='#C0392B', width=2), showlegend=False, hoverinfo='skip'), row=2, col=3)                                               # 13

fig.update_yaxes(scaleanchor='x', constrain='domain', autorange='reversed', showticklabels=False, showgrid=False, zeroline=False, row=1, col=1)
fig.update_xaxes(constrain='domain', showticklabels=False, showgrid=False, zeroline=False, row=1, col=1)
fig.update_yaxes(range=[float(min(p5.min(), -1)) * 1.1, float(p95.max()) * 1.1], zeroline=True, row=1, col=3)
for c_ in (1, 2, 3):
    fig.update_xaxes(title_text='época', range=[0, NE], row=2, col=c_)
fig.update_yaxes(range=[0, float(dist.max()) * 1.1], row=2, col=1)
fig.update_yaxes(range=yl, row=2, col=2); fig.update_yaxes(range=ya, row=2, col=3)
fig.update_annotations(font_size=12)

# 4) animação: a cada época mudam o mosaico, a faixa dos pesos, os marcadores e as linhas verticais
quadros = []
for e in range(NE + 1):
    quadros.append(go.Frame(name=str(e), traces=[0, 1, 7, 10, 13],
                            data=[go.Heatmap(z=mosaico(e)), go.Bar(y=p95[e] - p5[e], base=p5[e]), go.Scatter(x=[e] * 5, y=dist[e]), go.Scatter(x=[e, e], y=yl), go.Scatter(x=[e, e], y=ya)],
                            layout=go.Layout(title_text=titulo(e))))
fig.frames = quadros
passos = [dict(method='animate', label=(str(e) if e % 5 == 0 or e == NE else ' '), args=[[str(e)], dict(mode='immediate', frame=dict(duration=0, redraw=True), transition=dict(duration=0))]) for e in range(NE + 1)]
fig.update_layout(title=dict(text=titulo(0), y=0.97, yanchor='top', font_size=15), template='plotly_white', height=860, margin=dict(t=120, b=110, l=50, r=10),
                  legend=dict(orientation='h', y=1.14, x=0.45, xanchor='left', yanchor='top'),
                  sliders=[dict(active=0, steps=passos, x=0.04, len=0.92, currentvalue=dict(prefix='Época: '), pad=dict(t=40), font=dict(size=9))],
                  updatemenus=[dict(type='buttons', direction='left', x=0.0, y=1.12, xanchor='left', yanchor='top',
                                    buttons=[dict(label='▶ Treinar de novo (animação)', method='animate', args=[None, dict(frame=dict(duration=450, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
                                             dict(label='⏸ Pausar', method='animate', args=[[None], dict(mode='immediate', frame=dict(duration=0, redraw=False), transition=dict(duration=0))])])])
fig.show()

# 5) tabela: o que mudou em algumas épocas-chave
chave = sorted(set(e for e in [0, 1, 5, 10, 20, ep_melhor, NE] if e <= NE))
lin = []
for e in chave:
    r = {'Época': e, 'Acurácia de validação (%)': None if e == 0 else round(100 * float(h.loc[e, 'val_accuracy']), 1)}
    for i, n in enumerate(nomes_c):
        r[f'{n}: andou (%)'] = round(float(dist[e, i]), 1)
    lin.append(r)
tab27 = pd.DataFrame(lin).set_index('Época')
display(tab27)
print(f'\nMelhor época da réplica: {ep_melhor}. Escala de cor da Conv 1: ±{lim:.2f}.')

Conferência: instantâneo da época 26 × replica_melhor.keras → maior diferença entre pesos = 0.0e+00


,Acurácia de validação (%),Conv 1: andou (%),Conv 2: andou (%),Conv 3: andou (%),Conv 4: andou (%),Dense: andou (%)
Época,,,,,,
0,NaN,0.0,0.0,0.0,0.0,0.0
1,66.2,4.3,28.5,22.1,26.9,13.1
5,81.7,7.9,53.0,53.7,64.1,32.8
10,86.0,11.5,76.0,84.3,99.6,54.4
20,88.1,17.1,109.3,124.0,148.5,87.2
26,91.0,19.7,124.5,143.9,172.9,102.3
30,88.3,21.8,134.0,156.7,188.7,111.9



Melhor época da réplica: 26. Escala de cor da Conv 1: ±1.41.


**Observação:** a conferência de rigor passou: o instantâneo da época 26 é **idêntico** ao modelo salvo pelo checkpoint (diferença de 0,0e+00), então os 31 instantâneos são fiéis ao treino. Na época 0, antes de qualquer aprendizado, os filtros da Conv 1 aparecem fracos e aleatórios e a faixa dos pesos (5% a 95%) fica em cerca de −1,6 a +1,6 desvios-padrão em todas as camadas, como esperado para pesos sorteados em formato de sino.

Com o treino, o que muda é **diferente em cada camada**. Nas barras, a faixa dos pesos se alarga bastante nas camadas profundas: na época 30, lida pelo eixo, a Conv 4 vai de cerca de −3,8 a +2,9, a Conv 3 de cerca de −3,5 a +2,3, a Conv 2 de cerca de −2,9 a +2,5 e a Dense de cerca de −3,0 a +2,6. A Conv 1, ao contrário, quase não se alarga (algo como −1,7 a +1,8). Isso combina com a tabela do passo anterior: a Conv 4 foi a que mais andou a partir do início (188,7%) e a Conv 1 a que menos andou (21,8%). Na época 10 as barras estão num estágio intermediário (acurácia de validação de 86,0%).

No mosaico, a maioria dos 32 filtros da Conv 1 **mantém o padrão de cores da época 0** até a época 30, com mudanças pontuais; o mais visível é o primeiro filtro (canto superior esquerdo), que na época 30 ganha uma coluna azul bem escura (pesos positivos fortes). Isso é coerente com o deslocamento pequeno da Conv 1, e mostra que, nesta rede, o aprendizado se concentra mais nas camadas profundas do que nos filtros da entrada.

*Limites desta leitura:* (1) os valores das barras foram lidos pelo eixo do gráfico, por isso são aproximados. (2) Na janela estreita da época 10, a legenda ficou sobre parte do mosaico e dos títulos, então a descrição da época 10 se apoia nas barras e na acurácia, mais do que nos filtros; não vi os três painéis de baixo (deslocamentos, perda e acurácia) nesses prints. (3) Comparei os filtros a olho, sem uma medida numérica; a conclusão de que a maioria muda pouco é qualitativa. (4) Como no Passo 26, este gráfico mostra a evolução de **um treino equivalente** ao do modelo final, e não o histórico literal do `final.keras`.